# Imports

In [ ]:
# --- Cell: imports ---
import os
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import sys
sys.path.append('.')  # folder containing vti_magnetic_import.py
from vti_magnetic_import import load_vti_sample, inspect_vti

from scipy.ndimage import gaussian_filter, map_coordinates
from skimage.measure import marching_cubes
import pyvista as pv

import matplotlib.ticker as mticker
import pandas as pd

import json

import Functions_3D as vis

from scipy import ndimage as ndi
from skimage.filters import apply_hysteresis_threshold
from matplotlib.colors import ListedColormap

from scipy.optimize import curve_fit
from matplotlib.gridspec import GridSpec


from skimage.measure import find_contours
from scipy.interpolate import splprep, splev

from matplotlib.collections import LineCollection
from matplotlib.colors import TwoSlopeNorm


from scipy.spatial import cKDTree
from matplotlib.colors import Normalize


from scipy.stats import spearmanr
# from curvature_segmentation import plot_domain_curvature, curvature_field_dict, curvature_volume_dict


import json, hashlib, datetime


import curvature_segmentation as curv
# from curvature_segmentation import curvature_field_dict

In [ ]:
def plot_energy_maps_fancy(data_dict, key, axis='z', index=None, cmap='viridis', vmin=None, vmax=None, voxel_nm = 8.0, title = ''):
    """
    """
    nz, ny, nx = data_dict['shape']
    default_index = {'x': nx // 2, 'y': ny // 2, 'z': nz // 2}
    print(index)
    if index is None:
        index = default_index[axis]

    data = data_dict[key]
    slice_x = data[:, :, default_index['x']]   # (nz, ny) -- y-z plane at fixed x
    slice_y = data[:, default_index['y'], :]   # (nz, nx) -- x-z plane at fixed y
    slice_z = data[index, :, :]   # (ny, nx) -- x-y plane at fixed z


    LABEL_FONT = {'family': 'Times New Roman', 'weight': 'bold', 'size': 16}
    TITLE_FONT = {'family': 'Times New Roman', 'weight': 'bold', 'size': 20}
    TICK_FONT = {'family': 'Times New Roman', 'size': 13}
    CBAR_LABEL_FONT = {'family': 'Times New Roman', 'weight': 'bold', 'size': 20}

    def style_axis(ax):
        ax.set_xlabel(ax.get_xlabel(), fontdict=LABEL_FONT)
        ax.set_ylabel(ax.get_ylabel(), fontdict=LABEL_FONT)
        for label in ax.get_xticklabels() + ax.get_yticklabels():
            label.set_fontname('Times New Roman')
            label.set_fontsize(TICK_FONT['size'])

    def style_colorbar(cbar, label_text):
        cbar.set_label(label_text, fontdict=CBAR_LABEL_FONT)
        cbar.formatter = mticker.ScalarFormatter(useMathText=True)
        cbar.formatter.set_powerlimits((0, 0))
        cbar.update_ticks()
        for label in cbar.ax.get_yticklabels():
            label.set_fontname('Times New Roman')
            label.set_fontsize(TICK_FONT['size'])
        cbar.ax.yaxis.get_offset_text().set_fontname('Times New Roman')
        cbar.ax.yaxis.get_offset_text().set_fontsize(TICK_FONT['size'])

    # Conversion factor: 1 pix = 8 nm
    PIX_TO_NM = voxel_nm #2 #8


    # Calculate extents in nm
    extent_x = [0, ny * PIX_TO_NM / 1000, 0, nz * PIX_TO_NM]  # y, z
    extent_y = [0, nx * PIX_TO_NM / 1000, 0, nz * PIX_TO_NM]  # x, z
    extent_z = [0, nx * PIX_TO_NM / 1000, 0, ny * PIX_TO_NM / 1000]  # x, y


    # Create figure with GridSpec for custom layout
    fig = plt.figure(figsize=(25,8)) ##################################################################################
    gs = fig.add_gridspec(2, 3, width_ratios=[1, 2, 0.15], height_ratios=[1, 1], 
                        hspace=0.37, wspace=0.15)

    # Z slice (left, full square)
    ax_z = fig.add_subplot(gs[:, 0])
    im_z = ax_z.imshow(slice_z, origin='lower', aspect='equal', cmap=cmap,
                        vmin=vmin, vmax=vmax, extent=extent_z)
    ax_z.set_title(f"Z slice", fontdict=TITLE_FONT, color='lime')
    ax_z.set_xlabel('x (μm)', fontdict=LABEL_FONT)
    ax_z.set_ylabel('y (μm)', fontdict=LABEL_FONT)
    ax_z.axhline(default_index['y'] * PIX_TO_NM/1000, color='magenta', lw=5, linestyle='--', zorder=3)
    ax_z.axvline(default_index['x'] * PIX_TO_NM/1000, color='darkorange', lw=5, linestyle='--', zorder=3)
    style_axis(ax_z)

    # X slice (top right, thin and long)
    ax_x = fig.add_subplot(gs[0, 1])
    im_x = ax_x.imshow(slice_x, origin='lower', aspect='auto', cmap=cmap,
                        vmin=vmin, vmax=vmax, extent=extent_x)
    ax_x.axhline(index* PIX_TO_NM + PIX_TO_NM/2, color='lime', lw=5, linestyle='--', zorder=3)
    ax_x.set_title(f"X slice", fontdict=TITLE_FONT, color='darkorange')
    ax_x.set_xlabel('y (μm)', fontdict=LABEL_FONT)
    ax_x.set_ylabel('z (nm)', fontdict=LABEL_FONT)
    style_axis(ax_x)

    # Y slice (bottom right, thin and long)
    ax_y = fig.add_subplot(gs[1, 1])
    im_y = ax_y.imshow(slice_y, origin='lower', aspect='auto', cmap=cmap,
                        vmin=vmin, vmax=vmax, extent=extent_y)
    ax_y.axhline(index* PIX_TO_NM + PIX_TO_NM/2, color='lime', lw=5, linestyle='--', zorder=3)
    ax_y.set_title(f"Y slice", fontdict=TITLE_FONT, color='magenta')
    ax_y.set_xlabel('x (μm)', fontdict=LABEL_FONT)
    ax_y.set_ylabel('z (nm)', fontdict=LABEL_FONT)
    style_axis(ax_y)


    # Colorbar for Z slice
    cbar_xy = fig.colorbar(im_x, ax=[ax_x, ax_y], orientation='vertical', 
                            pad=0.05, aspect=30)
    style_colorbar(cbar_xy, key)

    if index is None:
        fig.suptitle(f"{key} Contrast at Volume Center", 
                    fontfamily=LABEL_FONT['family'],
                    fontweight=LABEL_FONT['weight'],
                    fontsize=25)

    else:
        fig.suptitle(f"{key} Contrast at thickness = {index*voxel_nm} nm", 
                    fontfamily=LABEL_FONT['family'],
                    fontweight=LABEL_FONT['weight'],
                    fontsize=25)
    #plt.show()

    # fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
    # for ax, key in zip(axes, keys):
    #     data = data_dict[key]
    #     img, xlabel, ylabel, extent = get_slice(data, axis, index, nz, ny, nx)

    #     im = ax.imshow(img, origin='lower', aspect='auto', cmap=panel_cmap[key], extent=extent)
    #     ax.set_title(key)
    #     ax.set_xlabel(xlabel)
    #     ax.set_ylabel(ylabel)
    #     fig.colorbar(im, ax=ax, label=key)

    # fig.suptitle(f"{title} for slice {axis} at index {index}")
    # plt.tight_layout()
    # plt.show()
    return #fig


def plot_energy_maps_fancy_oneslice(data_dict, key, axis='z', index=None, cmap='viridis',
                           vmin=None, vmax=None, voxel_nm=8.0, title=''):
    """
    Plot a Z slice (x-y plane) and a Y slice (x-z plane) of a 3D volume.
    """
    nz, ny, nx = data_dict['shape']
    default_index = {'x': nx // 2, 'y': ny // 2, 'z': nz // 2}
    print(index)
    if index is None:
        index = default_index[axis]

    data = data_dict[key]
    slice_y = data[:, default_index['y'], :]   # (nz, nx) -- x-z plane at fixed y
    slice_z = data[index, :, :]                # (ny, nx) -- x-y plane at fixed z

    LABEL_FONT = {'family': 'Times New Roman', 'weight': 'bold', 'size': 16}
    TITLE_FONT = {'family': 'Times New Roman', 'weight': 'bold', 'size': 20}
    TICK_FONT = {'family': 'Times New Roman', 'size': 13}
    CBAR_LABEL_FONT = {'family': 'Times New Roman', 'weight': 'bold', 'size': 20}

    def style_axis(ax):
        ax.set_xlabel(ax.get_xlabel(), fontdict=LABEL_FONT)
        ax.set_ylabel(ax.get_ylabel(), fontdict=LABEL_FONT)
        for label in ax.get_xticklabels() + ax.get_yticklabels():
            label.set_fontname('Times New Roman')
            label.set_fontsize(TICK_FONT['size'])

    def style_colorbar(cbar, label_text):
        cbar.set_label(label_text, fontdict=CBAR_LABEL_FONT)
        cbar.formatter = mticker.ScalarFormatter(useMathText=True)
        cbar.formatter.set_powerlimits((0, 0))
        cbar.update_ticks()
        for label in cbar.ax.get_yticklabels():
            label.set_fontname('Times New Roman')
            label.set_fontsize(TICK_FONT['size'])
        cbar.ax.yaxis.get_offset_text().set_fontname('Times New Roman')
        cbar.ax.yaxis.get_offset_text().set_fontsize(TICK_FONT['size'])

    PIX_TO_NM = voxel_nm

    # Extents (x, y in μm; z in nm)
    extent_y = [0, nx * PIX_TO_NM / 1000, 0, nz * PIX_TO_NM]                 # x, z
    extent_z = [0, nx * PIX_TO_NM / 1000, 0, ny * PIX_TO_NM / 1000]          # x, y

    # Figure: Z slice (left), Y slice (right)
    fig = plt.figure(figsize=(20, 6.5))
    gs = fig.add_gridspec(1, 2, width_ratios=[1, 2], wspace=0.15)

    # Z slice (left, square)
    ax_z = fig.add_subplot(gs[0, 0])
    im_z = ax_z.imshow(slice_z, origin='lower', aspect='equal', cmap=cmap,
                       vmin=vmin, vmax=vmax, extent=extent_z)
    # ax_z.set_title("Z slice", fontdict=TITLE_FONT, color='lime')
    ax_z.set_xlabel('x (μm)', fontdict=LABEL_FONT)
    ax_z.set_ylabel('y (μm)', fontdict=LABEL_FONT)
    # Line showing where the Y slice is taken
    ax_z.axhline(default_index['y'] * PIX_TO_NM / 1000, color='k', lw=5,
                 linestyle='--', zorder=3)
    style_axis(ax_z)

    # Y slice (right, wide)
    ax_y = fig.add_subplot(gs[0, 1])
    im_y = ax_y.imshow(slice_y, origin='lower', aspect='auto', cmap=cmap,
                       vmin=vmin, vmax=vmax, extent=extent_y)
    # Line showing where the Z slice is taken
    # ax_y.axhline(index * PIX_TO_NM + PIX_TO_NM / 2, color='lime', lw=5,
    #              linestyle='--', zorder=3)
    ax_y.set_title("Slice through thickness", fontdict=TITLE_FONT, color='k')
    ax_y.set_xlabel('x (μm)', fontdict=LABEL_FONT)
    ax_y.set_ylabel('z (nm)', fontdict=LABEL_FONT)
    ax_y.set_box_aspect(0.25)   
    style_axis(ax_y)

    # # Colorbar
    # cbar = fig.colorbar(im_y, ax=ax_y, orientation='vertical', pad=0.03, aspect=30)
    # style_colorbar(cbar, key)

    cbar = fig.colorbar(im_y, ax=ax_y, orientation='horizontal',
                        pad=0.2, aspect=40, shrink=0.9)
    style_colorbar(cbar, key)

    fig.suptitle(f"{key} Contrast at thickness = {index * voxel_nm} nm",
                 fontfamily=LABEL_FONT['family'],
                 fontweight=LABEL_FONT['weight'],
                 fontsize=25)

    return  # fig


# load Volume of interst and visually check

In [ ]:
def load_sample_data(save_dir, sample_name, date):
    """
    Load back the cropped volume for one sample
    """
    volume_path = os.path.join(save_dir, f'{sample_name}_volume_{date}.npz')

    npz = np.load(volume_path)
    cropped = {
        'Mx': npz['Mx'], 'My': npz['My'], 'Mz': npz['Mz'],
        'magnitude': npz['magnitude'],
        'shape': tuple(npz['shape']),
    }
    spacing = tuple(npz['spacing'])
    z_offset = 0

    return cropped, spacing, z_offset

def get_slice(data, axis, index, nz, ny, nx):
    if axis == 'x':
        return data[:, :, index], 'y (voxel)', 'z (voxel)', [0, ny, 0, nz]
    elif axis == 'y':
        return data[:, index, :], 'x (voxel)', 'z (voxel)', [0, nx, 0, nz]
    elif axis == 'z':
        return data[index, :, :], 'x (voxel)', 'y (voxel)', [0, nx, 0, ny]
    else:
        raise ValueError("axis must be 'x', 'y', or 'z'")

In [ ]:
sample_names = ['452', '270', '105', '665'] 
# dates = ['mag_const_updated_interpolation', '14AUG26_interpolated', '22AUG26_interpolated', '22AUG26_interpolated']
#dates = ['normalized_interpolated','original_interpolated','normalized_interpolated' ]

#dates = ['mag_const_TV_NOTinterpolated_t_OCT26', '14AUG26_interpolated', '22AUG26_interpolated', '22AUG26_interpolated']

#dates = ['mag_const_updated_interpolation', '14AUG26_interpolated', '22AUG26_interpolated', 'mag_const_TV_interpolated']

#dates = ['full_volume_mag_const_sqrt(2)_nointerp_OCT26', 'full_volume_mag_const_sqrt(2)_nointerp_OCT26', 'full_volume_mag_const_sqrt(2)_nointerp_OCT26', 'full_volume_mag_const_sqrt(2)_nointerp_OCT26']


# dates = ['cropped_mag_const_2interp_bsd_OCT26', '14AUG26_interpolated', 'mag_const_interpolated_t_OCT26', 'mag_const_TV_interpolated']


################ Cropped from full volume to center, with TV, no interpolation ################
dates = ['TV_5mask_cropped', 'TV_9thick_cropped', 'TV_5thick_cropped', 'TV_8nm_NOV26'] 


sample_colors = ['limegreen', 'deeppink', 'cyan', 'gold']


sample = 0
sample_name = sample_names[sample]
date = dates[sample]
sample_color = sample_colors[sample]


data_dir = f'C:/Users/emmab/Projects/Vector_Tomography/data_dir/{sample_name}/'


volume, spacing, ___ = load_sample_data(data_dir, sample_name, date)
print(spacing)
voxel_nm = spacing[0]

In [ ]:
print(volume['shape'])

In [ ]:
# vis.plot_three_slice_fancy(volume, field='Mz', cmap = 'viridis', voxel_nm = voxel_nm)
# vis.plot_three_slice_fancy(volume, field='My', voxel_nm = voxel_nm)
# vis.plot_three_slice_fancy(volume, field='Mx', voxel_nm = voxel_nm)
# vis.plot_three_slice_fancy(volume, field='Mag', voxel_nm = voxel_nm)
# # vis.plot_three_slice_fancy(volume, field='My', voxel_nm = 8.0)

# apply a gaussian smoothing

In [ ]:
def preview_gaussian_blur(sample, sigma, axis='z', index=None,
                           cmap='viridis', vmin=None, vmax=None):
    """
    Preview the effect of Gaussian smoothing on Mx, My, Mz at a given
    sigma, for a single chosen slice. Purely for visual comparison,
    nothing is saved or cached here.

    Parameters
    ----------
    sample : dict
        Sample dict with (nz, ny, nx)-shaped 'Mx', 'My', 'Mz'.
    sigma : float or tuple of 3 floats
        Smoothing applied identically to Mx, My, Mz for this preview.
    axis : {'x', 'y', 'z'}
        Which slice to look at.
    index : int or None
        Voxel index along that axis; defaults to center.
    """
    components = ['Mx', 'My', 'Mz']
    nz, ny, nx = sample['shape']
    default_index = {'x': nx // 2, 'y': ny // 2, 'z': nz // 2}
    if index is None:
        index = default_index[axis]

    if vmin is None:
        vmin = min(np.nanmin(sample[k]) for k in components)
    if vmax is None:
        vmax = max(np.nanmax(sample[k]) for k in components)


    fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
    for ax, key in zip(axes, components):
        smoothed = gaussian_filter(sample[key], sigma=sigma)
        img, xlabel, ylabel, extent = get_slice(smoothed, axis, index, nz, ny, nx)
        im = ax.imshow(img, origin='lower', aspect='auto', cmap=cmap,
                        vmin=-0.4, vmax=0.4, extent=extent)
        ax.set_title(key)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        fig.colorbar(im, ax=ax, label=key)

    fig.suptitle(f"{axis}-slice for index {index} \n (sigma={sigma})")
    plt.tight_layout()
    plt.show()
    return fig, axes

def make_smoothed_volume(sample, sigma):
    """
    Apply Gaussian smoothing at the chosen sigma to Mx, My, Mz (and
    recompute magnitude from the smoothed components), returning a new,
    flat volume dict with the same structure as `sample`, ready to use
    directly in downstream analysis without any nested/cached indexing.

    Parameters
    ----------
    sample : dict
        Original sample dict with (nz, ny, nx)-shaped 'Mx', 'My', 'Mz',
        and 'shape'.
    sigma : float or tuple of 3 floats
        The smoothing strength you've settled on after using
        preview_gaussian_blur.

    Returns
    -------
    volume_smoothed : dict
        New dict with 'Mx', 'My', 'Mz', 'magnitude', 'shape', and
        'sigma' (recording what was used, for provenance).
    """
    Mx_s = gaussian_filter(sample['Mx'], sigma=sigma)
    My_s = gaussian_filter(sample['My'], sigma=sigma)
    Mz_s = gaussian_filter(sample['Mz'], sigma=sigma)
    magnitude_s = np.sqrt(Mx_s**2 + My_s**2 + Mz_s**2)

    volume_smoothed = {
        'Mx': Mx_s,
        'My': My_s,
        'Mz': Mz_s,
        'magnitude': magnitude_s,
        'shape': sample['shape'],
        'sigma': sigma,
        'name': sample.get('name', '') + f'_smoothed_sigma{sigma}',
    }
    return volume_smoothed

In [ ]:
sigmas_to_test = [0.1, 0.5, 1.0, 2.0]

for s in sigmas_to_test:
    preview_gaussian_blur(volume, axis='z', index=None, sigma=s)


############## SELECT SIGMA TO SAVE ##############
sigma_save = 0.1
volume_smoothed = make_smoothed_volume(volume, sigma=sigma_save)
volume_smoothed.keys()
##################################################

use_smoothed = True 

if use_smoothed: 
    volume = volume_smoothed

# Compute the Jacobian

In [ ]:
def compute_jacobian(volume, spacing=(8.0, 8.0, 8.0)):
    """
    Compute the full 3x3 gradient tensor (Jacobian) of the magnetization
    field from a smoothed volume dict.

    Parameters
    ----------
    volume_smoothed : dict
        Must contain 'Mx', 'My', 'Mz', each shape (nz, ny, nx), e.g. the
        output of make_smoothed_volume.
    spacing : tuple
        (dz, dy, dx) physical voxel size in nm, used so gradients come
        out in physical units (per nm) rather than per voxel.

    Returns
    -------
    jacobian : dict
        Nine entries, keyed as 'dMi_dj' for i in {x,y,z}, j in {x,y,z},
        e.g. jacobian['dMx_dx'], jacobian['dMz_dy'], etc. Each entry is
        an array the same shape as the input volume. Also includes
        'spacing' for provenance.
    """
    dz, dy, dx = spacing
    M_mag = np.sqrt(volume['Mx']**2 + volume['My']**2 + volume['Mz']**2)
    components = {'x': volume['Mx']/M_mag,
                  'y': volume['My']/M_mag,
                  'z': volume['Mz']/M_mag}

    jacobian = {}
    for comp_name, comp_data in components.items():
        dM_dz, dM_dy, dM_dx = np.gradient(comp_data, dz, dy, dx)
        jacobian[f'dM{comp_name}_dz'] = dM_dz
        jacobian[f'dM{comp_name}_dy'] = dM_dy
        jacobian[f'dM{comp_name}_dx'] = dM_dx

    jacobian['spacing'] = spacing
    jacobian['shape'] = volume['shape']
    return jacobian

In [ ]:
jacobian = compute_jacobian(volume, spacing=spacing)

## Gradient and isolated dw

In [ ]:

def normalize_volume(volume, eps=1e-6, fill=0.0):
    """
    Return a copy of `volume` with (Mx, My, Mz) rescaled to unit vectors.
    Voxels with |m| <= eps (empty/outside the sample) are set to `fill`.
    """
    Mx = volume['Mx'].astype(np.float32)
    My = volume['My'].astype(np.float32)
    Mz = volume['Mz'].astype(np.float32)

    # Norm of the vectors as they are now (after smoothing)
    norm = np.sqrt(Mx**2 + My**2 + Mz**2)
    valid = norm > eps

    out = {}
    for name, comp in (('Mx', Mx), ('My', My), ('Mz', Mz)):
        u = np.full(norm.shape, fill, dtype=np.float32)
        np.divide(comp, norm, out=u, where=valid)
        out[name] = u

    out['magnitude'] = volume['magnitude']      # original magnitude kept for building the sample mask
    out['sample_mask'] = valid                  # boolean: where a direction exists
    out['shape'] = volume['shape']
    out['sigma'] = volume.get('sigma')
    out['name'] = volume.get('name', '') + '_normalized'
    return out

volume_unit = normalize_volume(volume)

# Sanity check: |m| should be 1 wherever the mask is True
n = np.sqrt(volume_unit['Mx']**2 + volume_unit['My']**2 + volume_unit['Mz']**2)
print(n[volume_unit['sample_mask']].min(), n[volume_unit['sample_mask']].max())  # both ~1.0
print(f"{(~volume_unit['sample_mask']).mean()*100:.1f}% of voxels empty")



In [ ]:
# jacobian_norm = compute_jacobian(volume_unit, spacing=spacing)
grad_sq = np.zeros(jacobian['shape'], dtype=np.float32)
for key, val in jacobian.items():
    if key.startswith('dM'):          
        grad_sq += val ** 2

frobenius_norm = np.sqrt(grad_sq)
print(frobenius_norm.shape)           # should be (Z, Y, X), not 9 channels

def frobenius_from_jacobian(jac, axes=("x", "y")):
    """Frobenius norm over all three components of m, summed over the chosen derivative axes."""
    keys = [f"dM{c}_d{d}" for c in "xyz" for d in axes]
    grad_sq = np.zeros(jac["shape"], dtype=np.float32)
    for k in keys:
        grad_sq += jac[k] ** 2
    return np.sqrt(grad_sq)

frobenius_2d = frobenius_from_jacobian(jacobian, axes=("x", "y"))   # in-plane only
frobenius_z  = frobenius_from_jacobian(jacobian, axes=("z",))       # through-thickness only
print(frobenius_2d.shape)    # (Z, Y, X)

gradient = {'grad_xy': frobenius_2d, 
            'grad_3D': frobenius_norm }

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4))

im = ax[0].imshow(frobenius_norm[0], cmap="magma")
ax[0].set_title(f"|grad m|, z={0}")

im = ax[1].imshow(frobenius_2d[0], cmap="magma")
ax[1].set_title(f"|grad m (x,y)|, z={0}")

im = ax[2].imshow(frobenius_z[0], cmap="magma")
ax[2].set_title(f"|grad m(z)|, z={0}")



# Calculate Curl from the Jacobian

In [ ]:
def compute_curl(jacobian, volume):
    """
    standard definition:
        curl(M)_x = dMz/dy - dMy/dz
        curl(M)_y = dMx/dz - dMz/dx
        curl(M)_z = dMy/dx - dMx/dy

    Parameters
    ----------
    jacobian : dict
        Output of compute_jacobian, containing dMx_dx, dMx_dy, dMx_dz,
        dMy_dx, dMy_dy, dMy_dz, dMz_dx, dMz_dy, dMz_dz, plus 'spacing'
        and 'shape'.

    Returns
    -------
    curl : dict
        'curl_x', 'curl_y', 'curl_z' (each shape (nz, ny, nx), units
        per nm assuming jacobian was computed with physical spacing),
        'curl_magnitude' (Euclidean norm of the three components at
        each voxel), plus 'shape' carried through for convenience.
    """

    M_mag = np.sqrt(volume['Mx']**2 + volume['My']**2 + volume['Mz']**2)
    unit_Mx = volume['Mx']/M_mag
    unit_My = volume['My']/M_mag
    unit_Mz = volume['Mz']/M_mag

    curl_x = jacobian['dMz_dy'] - jacobian['dMy_dz']
    curl_y = jacobian['dMx_dz'] - jacobian['dMz_dx']
    curl_z = jacobian['dMy_dx'] - jacobian['dMx_dy']

    curl_magnitude = np.sqrt(curl_x**2 + curl_y**2 + curl_z**2)

    helicity = curl_x*unit_Mx + curl_y*unit_My + curl_z*unit_Mz    ### Curl dot M

    cx = curl_y*unit_Mz - curl_z*unit_My                       
    cy = curl_z*unit_Mx - curl_x*unit_Mz
    cz = curl_x*unit_My - curl_y*unit_Mx
    curl_perp = np.sqrt(cx**2 + cy**2 + cz**2)             # ||Curl x M||

    return {
        'curl_x': curl_x,
        'curl_y': curl_y,
        'curl_z': curl_z,
        'curl_magnitude': curl_magnitude,
        'helicity': helicity,   #### scalar projection length
        'parallel': helicity**2, #### value of scalar projection length
        'perpendicular': curl_perp, #### length of vector 
        'par_perp': helicity**2 + curl_perp**2,
        'shape': jacobian['shape'],
    }

In [ ]:
curl = compute_curl(jacobian, volume)

In [ ]:
plot_energy_maps_fancy_oneslice(curl, 'curl_magnitude', axis='z', index=None, cmap='coolwarm', vmin=None, vmax=None, voxel_nm = voxel_nm, title = '')
plot_energy_maps_fancy_oneslice(curl, 'helicity', axis='z', index=None, cmap='coolwarm', vmin=None, vmax=None, voxel_nm = voxel_nm, title = '')
plot_energy_maps_fancy_oneslice(curl, 'parallel', axis='z', index=None, cmap='coolwarm', vmin=None, vmax=None, voxel_nm = voxel_nm, title = '')
# plot_energy_maps_fancy_oneslice(curl, 'perpendicular', axis='z', index=None, cmap='coolwarm', vmin=None, vmax=None, voxel_nm = voxel_nm, title = '')
# plot_energy_maps_fancy_oneslice(curl, 'par_perp', axis='z', index=None, cmap='coolwarm', vmin=None, vmax=None, voxel_nm = voxel_nm, title = '')

In [ ]:
total_sum = np.sum(curl['helicity'][2])
print("Total pixel sum:", total_sum)

total_mean = np.mean(curl['helicity'][2])
print("Total pixel mean:", total_mean)

total_std = np.std(curl['helicity'][2])
print("Total pixel std:", total_std)

In [ ]:
stop

## skip

In [ ]:
# set threshold

def threshold_sweep(curl_magnitude, valid_eroded, ref,
                    fracs=None, min_size=500, connectivity_structure=None):
    """
    For each threshold fraction f, mask = curl_magnitude > f * ref (inside valid_eroded),
    then label connected components and record summary statistics.
    """
    if fracs is None:
        fracs = np.linspace(0.05, 0.8, 40)
    if connectivity_structure is None:
        connectivity_structure = np.ones((3, 3, 3))   # 26-connectivity

    results = {k: [] for k in
               ["f", "n_voxels", "n_components", "n_large", "largest_frac"]}

    for f in fracs:
        mask = (curl_magnitude > f * ref) & valid_eroded
        n_vox = int(mask.sum())

        lab, n_comp = ndi.label(mask, structure=connectivity_structure)
        if n_comp > 0:
            sizes = np.bincount(lab.ravel())[1:]          # skip background (label 0)
            n_large = int((sizes >= min_size).sum())
            largest_frac = sizes.max() / n_vox
        else:
            n_large, largest_frac = 0, 0.0

        results["f"].append(f)
        results["n_voxels"].append(n_vox)
        results["n_components"].append(n_comp)
        results["n_large"].append(n_large)
        results["largest_frac"].append(largest_frac)

    return {k: np.array(v) for k, v in results.items()}


def plot_threshold_sweep(res, min_size, chosen_f=None, save_path=None):
    f = res["f"]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True)

    # Panel 1: how much of the volume survives
    ax = axes[0]
    ax.semilogy(f, np.maximum(res["n_voxels"], 1), "o-", ms=3)
    ax.set_xlabel("threshold fraction f  (of reference curl)")
    ax.set_ylabel("voxels above threshold")
    ax.set_title("Mask size")

    # Panel 2: fragmentation
    ax = axes[1]
    ax.plot(f, res["n_components"], "o-", ms=3, color="tab:gray",
            label="all components")
    ax.plot(f, res["n_large"], "o-", ms=3, color="tab:red",
            label=f"components ≥ {min_size} voxels")
    ax.set_xlabel("threshold fraction f")
    ax.set_ylabel("component count")
    ax.set_title("Fragmentation")
    ax.legend()

    # Panel 3: merging
    ax = axes[2]
    ax.plot(f, res["largest_frac"], "o-", ms=3, color="tab:green")
    ax.set_xlabel("threshold fraction f")
    ax.set_ylabel("largest component / total masked voxels")
    ax.set_ylim(0, 1.05)
    ax.set_title("Merging")

    if chosen_f is not None:
        for ax in axes:
            ax.axvline(chosen_f, color="k", ls="--", lw=1)

    for ax in axes:
        ax.grid(alpha=0.3)

    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=200)
    plt.show()


def overlay_wall_mask(Mz, wall_mask, spacing=(2.0, 2.0, 2.0), idx=None,
                      alpha=0.35, title=None, save_path=None):
    """
    Show Mz with the wall mask overlaid in the three orthogonal planes.

    Mz        : 3D array indexed [x, y, z]
    wall_mask : boolean 3D array, same shape
    spacing   : voxel size (dx, dy, dz) in nm
    idx       : (ix, iy, iz) slice indices; defaults to the mask's center of mass
    """
    nx, ny, nz = Mz.shape
    dx, dy, dz = spacing

    if idx is None:
        if wall_mask.any():
            idx = tuple(int(round(c)) for c in ndi.center_of_mass(wall_mask))
        else:
            idx = (nx // 2, ny // 2, nz // 2)
    ix, iy, iz = idx

    # pixel-center coordinates in nm (for contours)
    xc, yc, zc = (np.arange(n) * d + d / 2 for n, d in zip((nx, ny, nz), spacing))

    # (label, image, mask, horizontal coords, vertical coords, extent, axis labels)
    planes = [
        (f"xy plane, z = {iz*dz:.0f} nm", Mz[:, :, iz], wall_mask[:, :, iz],
         xc, yc, (0, nx*dx, 0, ny*dy), "x (nm)", "y (nm)"),
        (f"xz plane, y = {iy*dy:.0f} nm", Mz[:, iy, :], wall_mask[:, iy, :],
         xc, zc, (0, nx*dx, 0, nz*dz), "x (nm)", "z (nm)"),
        (f"yz plane, x = {ix*dx:.0f} nm", Mz[ix, :, :], wall_mask[ix, :, :],
         yc, zc, (0, ny*dy, 0, nz*dz), "y (nm)", "z (nm)"),
    ]

    vmax = np.nanmax(np.abs(Mz))
    red = ListedColormap(["red"])

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    for ax, (lab, img, msk, hc, vc, ext, xl, yl) in zip(axes, planes):
        # transpose so the first array axis is horizontal; origin="lower" puts 0 at bottom
        im = ax.imshow(img.T, origin="lower", extent=ext, cmap="viridis",
                       vmin=-vmax, vmax=vmax, aspect="equal")
        ax.imshow(np.ma.masked_where(~msk.T, msk.T), origin="lower", extent=ext,
                  cmap=red, alpha=alpha, aspect="equal", interpolation="nearest")
        if msk.any():
            ax.contour(hc, vc, msk.T.astype(float), levels=[0.5],
                       colors="red", linewidths=0.8)
        ax.set_title(lab)
        ax.set_xlabel(xl)
        ax.set_ylabel(yl)

    fig.colorbar(im, ax=axes, shrink=0.8, label="Mz")
    if title:
        fig.suptitle(title)
    if save_path:
        fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()


def threshold_domain_walls(volume, curl, lo = 0.2, hi=0.6, spacing = (8.0, 8.0, 8.0), wall_thickness_nm = 30): 

    M_mag = np.sqrt(volume['Mx']**2 + volume['My']**2 + volume['Mz']**2)
    unit_Mx = volume['Mx']/M_mag
    unit_My = volume['My']/M_mag
    unit_Mz = volume['Mz']/M_mag

    curl_mag = curl['curl_magnitude']
    curl_par= curl['parallel']
    curl_perp = curl['perpendicular']


    norm = np.sqrt(volume['Mx']**2 + volume['My']**2 + volume['Mz']**2)
    valid = norm > 1e-6 * norm.max()

    struct_xy = np.ones((1, 3, 3), dtype=bool)
    valid_eroded = ndi.binary_erosion(valid, structure=struct_xy, iterations=2)

    # drop boundary voxels where derivatives are unreliable
    # valid_eroded = ndi.binary_erosion(valid, iterations=2)
    vals = curl_mag[valid_eroded]

    ref = np.percentile(vals, 99.9)     # robust stand-in for the peak
    plt.hist(vals, bins=200, log=True)
    plt.axvline(ref, color="k", ls="--")
    plt.xlabel("|curl m|"); plt.ylabel("voxel count")


        
    # ---- usage ----
    dx, dy, dz = spacing                
    min_patch_area_nm2 = 400.0                  # smallest wall patch you care about


    min_size = int(min_patch_area_nm2 * wall_thickness_nm / (dx * dy * dz))

    res = threshold_sweep(curl_mag, valid_eroded, ref,
                        fracs=np.linspace(0.05, 0.8, 40), min_size=min_size)
    plot_threshold_sweep(res, min_size, chosen_f=0.40, save_path=None)


    ## hysteresis threshold
    struct = np.ones((3, 3, 3))          # 26-connectivity keeps thin walls connected

    low, high = lo * ref, hi * ref     # starting guesses; tune from the sweep
    mask = apply_hysteresis_threshold(curl_mag * valid_eroded, low, high)

    lab, n = ndi.label(mask, structure=struct)
    sizes = np.bincount(lab.ravel())[1:]
    # min_size = 100                        # voxels; choose from the size distribution
    keep_ids = 1 + np.flatnonzero(sizes >= min_size)
    wall_mask = np.isin(lab, keep_ids)
    wall_labels, n_walls = ndi.label(wall_mask, structure=struct)


    overlay_wall_mask(unit_Mz, wall_mask, spacing=(dx, dy, dz), title="f = 0.4")

    return wall_mask, ref



In [ ]:
lo = 0.2
hi=0.6

wall_mask, ref = threshold_domain_walls(volume, curl, lo = lo, hi=hi, spacing = (voxel_nm, voxel_nm, voxel_nm), wall_thickness_nm = 40)

In [ ]:
# stop

In [ ]:


def _gaussian(x, A, mu, sigma):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2))


def core_positions(curl_par, wall_mask, min_peak, dz_nm=2.0, bin_nm=8.0):
    """Arrays are (z, x, y). Returns fine-grid and native-resolution core layers."""
    wall_mask = wall_mask.astype(bool)
    nz, nx, ny = wall_mask.shape
    g = int(round(bin_nm / dz_nm))                 # fine layers per native voxel (4)
    assert nz % g == 0, "nz must be a multiple of bin_nm/dz_nm"
    n_nat = nz // g

    bloch = np.abs(curl_par) * wall_mask           # zero outside the mask
    col_ok = (bloch.max(axis=0) > min_peak) & wall_mask.any(axis=0)

    k_fine = bloch.argmax(axis=0)                  # fine-grid peak, per column
    lay_fine = k_fine // g                         # native layer containing that peak

    nat = bloch.reshape(n_nat, g, nx, ny).mean(axis=1)   # mean over each group of g layers
    lay_nat = nat.argmax(axis=0)                   # native-resolution peak

    n_ok = col_ok.sum()
    out = dict(
        col_ok=col_ok, k_fine=k_fine, lay_fine=lay_fine, lay_nat=lay_nat, g=g, n_nat=n_nat,
        frac_fine=np.bincount(lay_fine[col_ok], minlength=n_nat) / n_ok,
        frac_nat=np.bincount(lay_nat[col_ok], minlength=n_nat) / n_ok,
        agree=float((lay_fine == lay_nat)[col_ok].mean()),
        centers=(np.arange(n_nat) + 0.5) * bin_nm - n_nat * bin_nm / 2,
    )
    print("fine-grid fractions :", np.round(out['frac_fine'], 4))
    print("native-avg fractions:", np.round(out['frac_nat'], 4))
    print(f"columns where both methods pick the same layer: {out['agree']:.1%}")
    return out




def plot_Bloch_core_vs_z_m0(curl, wall_mask, min_peak,
                            voxel_nm=8.0, bin_nm=8.0,
                            color='green', title=None,
                             show_fit=True, ylim=None, normalize_bins = True, 
                             figsize=(24, 8)):
    """

    """
    if bin_nm is None:
        bin_nm = voxel_nm

    curl_par = curl['parallel']
    par_perp = curl['par_perp']
    wall_mask = wall_mask.astype(bool)

    nz = wall_mask.shape[0]
    print(nz)
    T = nz * voxel_nm                                    # film thickness (nm)
    z_fine = (np.arange(nz) + 0.5) * voxel_nm - T / 2    # layer centers relative to film center

    
    # ---- per-layer profile: mean and std over masked voxels in each z layer ----

    n_bins = max(int(round(T / bin_nm)), 1) 
    print(n_bins)

    if nz % n_bins == 0:
        g = nz // n_bins                                     # e.g. 16 // 4 = 4 layers per bucket

        avg_wall_angle_voxel = np.full(n_bins, np.nan)
        std_wall_angle_voxel = np.full(n_bins, np.nan)
        edges = np.linspace(-T / 2, T / 2, n_bins + 1)
        x_lineplot = 0.5 * (edges[:-1] + edges[1:])
        for b in range(n_bins):
            sl = slice(b * g, (b + 1) * g)                   
            v = par_perp[sl][wall_mask[sl]]                  
            # if v.size > 1:
            avg_wall_angle_voxel[b] = v.mean()
            std_wall_angle_voxel[b] = v.std()

    else: 
        print(f"nz = {nz} is not divisible by n_bins = {n_bins}")
        avg_wall_angle_voxel = np.full(nz, np.nan)
        std_wall_angle_voxel = np.full(nz, np.nan)
        edges = np.linspace(-nz, nz, nz + 1)
        x_lineplot = 0.5 * (edges[:-1] + edges[1:])
        for k in range(nz):
            v = par_perp[k][wall_mask[k]]
            # if v.size > 1:
            avg_wall_angle_voxel[k] = v.mean()
            std_wall_angle_voxel[k] = v.std()


    # ---- histogram of bloch position per layer  ----

    bloch = np.abs(curl_par) * wall_mask            # zero outside the mask

    k_peak   = np.argmax(bloch, axis=0)             # z index of the peak, per (x, y) column
    bloch_peak = bloch.max(axis=0)                  # strength of that peak
    n_layers = wall_mask.sum(axis=0)              # masked voxels in each column

    min_peak   = 0.1 * ref                        # guess, may be too strong or weak
    col_count = (bloch_peak > min_peak)

    z_peaks = (k_peak[col_count] + 0.5) * voxel_nm - T / 2  # nm relative to film center

    edges = np.linspace(-T / 2, T / 2, n_bins + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])
    y, _ = np.histogram(z_peaks, bins=edges)

    if normalize_bins: 
        y = y / y.sum() 

    # ---- statistics from the individual z_pekas values (independent of bin size) ----
    stats = {'n': len(z_peaks)}
    if len(z_peaks) > 1:
        stats.update(mean=float(np.mean(z_peaks)), median=float(np.median(z_peaks)),
                     variance=float(np.var(z_peaks, ddof=1)), std=float(np.std(z_peaks, ddof=1)), sterr = (float(np.std(z_peaks, ddof=1)) / np.sqrt(len(z_peaks))))
    else:
        stats = None


    # ---- Gaussian fit to the binned counts ----
    A = mu = sigma = fwhm = r2 = None
    if show_fit and stats is not None and y.sum() > 0:
        try:
            p0 = [y.max(), stats['mean'], max(stats['std'], bin_nm)]
            (A, mu, sigma), _ = curve_fit(_gaussian, centers, y, p0=p0, maxfev=10000)
            sigma = abs(sigma)
            fwhm = 2 * np.sqrt(2 * np.log(2)) * sigma
            resid = y - _gaussian(centers, A, mu, sigma)
            ss_tot = np.sum((y - y.mean()) ** 2)
            r2 = 1 - np.sum(resid ** 2) / ss_tot if ss_tot > 0 else np.nan
            stats.update(A=float(A), mu=float(mu), sigma=float(sigma),
                         fwhm=float(fwhm), r2=float(r2))
        except RuntimeError:
            print("Gaussian fit did not converge; plotting without it")
            show_fit = False
    else:
        if show_fit:
            print(f"Skipping fit: need >= 4 bins with counts (have {len(centers)} bins)")
        show_fit = False

    if stats is not None:
        print(f"z (nm from center): n = {stats['n']}, mean = {stats['mean']:.2f}, "
              f"median = {stats['median']:.2f}, variance = {stats['variance']:.2f} nm^2")
        if show_fit:
            print(f"Gaussian fit: mu = {mu:.2f} nm, sigma = {sigma:.2f} nm, "
                  f"FWHM = {fwhm:.2f} nm, R² = {r2:.2f}")


    # ## old plot
    # plt.hist(z_peaks, bins=centers, edgecolor="k")
    # plt.xlabel("z of Bloch peak |curl·m| (nm)")
    # plt.ylabel("number of columns")
    # plt.show()
    # print(f"{col_count.sum()} of {(n_layers > 0).sum()} wall columns kept")

    # ---- plots ----
    xlabel = 'z relative to film center (nm)'
    fig = plt.figure(figsize=figsize)
    gs = GridSpec(1, 2, width_ratios=[1, 1], figure=fig)
    ax2 = fig.add_subplot(gs[0])
    ax3 = fig.add_subplot(gs[1], sharex=ax2)

    # # Panel 1: curl-based wall profile vs z (fine grid)
    ax2.errorbar(x_lineplot, avg_wall_angle_voxel, yerr = std_wall_angle_voxel / np.sqrt(nz), fmt='o-', capsize=3, color=color, label='masked mean, err = stderr')
    # ax2.axhline(0, color='magenta', ls='--', label='Bloch (0)')
    # ax2.axhline(1, color='cyan', ls='--', label='Néel (±1)')
    # ax2.axhline(-1, color='cyan', ls='--')
    ax2.axvline(0, color='k', ls='--', label='Film center')
    if ylim is not None:
        ax2.set_ylim(*ylim)
    if stats:
        ax2.axvline(stats['mean'], color='red', lw=1.5, label=f"mean = {stats['mean']:.2f} +/- {stats['sterr']:.2f} nm")
        ax2.axvline(stats['median'], color='tab:blue', lw=1.5, ls='-.',
                    label=f"median = {stats['median']:.2f} nm")
    ax2.set_xlabel(xlabel, fontsize=25)
    ax2.set_ylabel('Wall profile (Curl)', fontsize=25)
    ax2.legend(fontsize=20)

    # res = core_positions(curl_par, wall_mask, min_peak=0.1 * ref)
    # ax2b = ax2.twinx()
    # ax2b.plot(centers, 100 * res['frac_nat'], 's--', color='darkorange', ms=12, lw=2,
    #         label='core layer, native avg (test 5)')
    # ax2b.plot(centers, 100 * res['frac_fine'], 'o', mfc='none', mec='k', mew=2, ms=18,
    #         label='core layer, fine-grid argmax')
    # ax2b.set_ylim(0, 100)
    # ax2b.set_ylabel('% of Bloch columns', fontsize=25)
    # ax2b.legend(loc='lower right', fontsize=16)


    # Panel 2: histogram of Bloch core positions
    ax3.bar(centers, y, width=0.9 * bin_nm, color=color, alpha=0.5, edgecolor='gray')
    ax3.axvline(0, color='k', ls='--', label='Film center')
    if stats:
        ax3.axvline(stats['mean'], color='red', lw=1.5, label=f"mean = {stats['mean']:.2f} +/- {stats['sterr']:.2f} nm")
        ax3.axvline(stats['median'], color='tab:blue', lw=1.5, ls='-.', label=f"median = {stats['median']:.2f} nm")
        ax3.plot([], [], ' ', label=f"variance = {stats['variance']:.2f} nm²")
    if show_fit:
        xs = np.linspace(edges[0], edges[-1], 300)
        ax3.plot(xs, _gaussian(xs, A, mu, sigma), color='k', lw=2,
                 label=f"Gaussian fit")
        ax3.plot([], [], ' ', label=f"FWHM = {fwhm:.2f} nm")
        ax3.axvline(mu, color='darkorange', lw=1, ls=':', label=f"fit μ = {mu:.2f} nm")

    ax3.legend(loc='upper left', fontsize=20)
    ax3.set_xlabel(xlabel, fontsize=25)

    if normalize_bins:
        ax3.set_ylabel(f'fraction of Bloch points per {bin_nm:g} nm', fontsize=20)
    else: 
        ax3.set_ylabel(f'# of Bloch points per {bin_nm:g} nm', fontsize=20)
    ax3.set_xticks(centers)
    ax3.set_xticklabels([f'{v:g}' for v in centers], rotation=45)

    fig.suptitle(f'Bloch Core vs. thickness position ({bin_nm:g} nm bins) \n {title}', fontsize=30)
    plt.tight_layout()
    plt.show()

    profile = dict(z_nm=centers, mean=avg_wall_angle_voxel, std=std_wall_angle_voxel)
    return stats, profile

In [ ]:
stats, profile = plot_Bloch_core_vs_z_m0(curl, wall_mask, min_peak=0.1 * ref,
                                         voxel_nm=voxel_nm, bin_nm=voxel_nm, color = sample_color, title=sample_name)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb

def darken(color, factor=0.55):
    """Return a darker version of any matplotlib color."""
    return tuple(c * factor for c in to_rgb(color))

def plot_curl_sums_vs_z(curl, wall_mask=None, voxel_nm=8.0, color='green',
                        title=None, use_abs=True, reduce='sum', normalize=False,
                        figsize=(20, 7)):
    """
    curl      : dict with 'curl_magnitude' and 'parallel', arrays (z, x, y)
    wall_mask : optional boolean array (z, x, y); if given, only masked voxels count
    use_abs   : sum |curl_parallel| (recommended; the signed sum cancels if both
                chiralities are present)
    reduce    : 'sum' (what you asked for) or 'mean' (divides by voxel count per layer,
                which removes the effect of wall area changing from layer to layer)
    normalize : if True, divide each curve by its own total so both sum to 1
                (compares shapes instead of magnitudes)
    """
    mag = np.asarray(curl['curl_magnitude'], dtype=float)
    par = np.asarray(curl['parallel'], dtype=float)
    if use_abs:
        par = np.abs(par)

    nz = mag.shape[0]
    w = np.ones(mag.shape, bool) if wall_mask is None else np.asarray(wall_mask, bool)

    def per_layer(a):
        a = np.where(w & np.isfinite(a), a, 0.0)
        s = a.sum(axis=(1, 2))
        if reduce == 'mean':
            n = w.sum(axis=(1, 2))
            s = np.where(n > 0, s / np.maximum(n, 1), np.nan)
        return s

    s_mag, s_par = per_layer(mag), per_layer(par)
    if normalize:
        s_mag = s_mag / np.nansum(s_mag)
        s_par = s_par / np.nansum(s_par)

    # layer centers relative to film center (e.g. -16, -8, 0, 8, 16 for nz = 5)
    z = (np.arange(nz) - (nz - 1) / 2) * voxel_nm

    c_par = color
    c_mag = darken(color)
    ylabel = {('sum', False): 'Sum of curl over layer',
              ('mean', False): 'Mean curl over layer',
              ('sum', True): 'Fraction of total curl',
              ('mean', True): 'Fraction of total (mean curl)'}[(reduce, normalize)]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize, sharex=True)

    # Left: line profiles
    ax1.plot(z, s_mag, 'o-', color=c_mag, lw=2, ms=8, label='|curl| (magnitude)')
    ax1.plot(z, s_par, 's-', color=c_par, lw=2, ms=8,
             label='|curl·m| (parallel / Bloch)' if use_abs else 'curl·m (parallel)')
    ax1.axvline(0, color='k', ls='--', lw=1, label='Film center')
    ax1.set_xlabel('z relative to film center (nm)', fontsize=22)
    ax1.set_ylabel(ylabel, fontsize=22)
    ax1.legend(fontsize=16)
    ax1.grid(alpha=0.3)

    # Right: grouped bars, centered on voxel centers
    bw = 0.4 * voxel_nm
    ax2.bar(z - bw / 2, s_mag, width=bw, color=c_mag, alpha=0.8,
            edgecolor='gray', label='|curl|')
    ax2.bar(z + bw / 2, s_par, width=bw, color=c_par, alpha=0.6,
            edgecolor='gray', label='|curl·m|' if use_abs else 'curl·m')
    ax2.axvline(0, color='k', ls='--', lw=1)
    ax2.set_xlabel('z relative to film center (nm)', fontsize=22)
    ax2.set_ylabel(ylabel, fontsize=22)
    ax2.legend(fontsize=16)

    for ax in (ax1, ax2):
        ax.set_xticks(z)
        ax.set_xticklabels([f'{v:g}' for v in z])
        ax.tick_params(labelsize=14)

    fig.suptitle(f'Curl vs. thickness position ({voxel_nm:g} nm layers)\n{title or ""}',
                 fontsize=26)
    plt.tight_layout()
    plt.show()

    ratio = np.where(s_mag > 0, s_par / s_mag, np.nan)
    print("layer  z(nm)   curl_mag      curl_par     par/mag")
    for k in range(nz):
        print(f"{k:5d} {z[k]:6.0f} {s_mag[k]:12.4g} {s_par[k]:12.4g} {ratio[k]:10.3f}")

    return dict(z_nm=z, sum_mag=s_mag, sum_par=s_par, ratio=ratio)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb

def darken(color, factor=0.55):
    return tuple(c * factor for c in to_rgb(color))

def weighted_stats(z, w, voxel_nm, median_method='interp'):
    """
    Mean and median of z, using per-layer values w as weights.
    median_method: 'interp'   -> interpolate the cumulative distribution within the
                                 layer where it crosses 0.5 (assumes weight is uniform
                                 across each voxel)
                   'discrete' -> center of the layer where the cumulative crosses 0.5
    """
    w = np.where(np.isfinite(w), w, 0.0)
    tot = w.sum()
    if tot <= 0:
        return np.nan, np.nan
    p = w / tot
    mean = float(np.sum(z * p))
    cum = np.cumsum(p)
    k = int(np.searchsorted(cum, 0.5))            # first layer where cum >= 0.5
    if median_method == 'discrete' or p[k] == 0:
        median = float(z[k])
    else:
        before = cum[k] - p[k]
        median = float(z[k] - voxel_nm / 2 + (0.5 - before) / p[k] * voxel_nm)
    return mean, median

def plot_curl_sums_vs_z(curl, wall_mask=None, voxel_nm=8.0, color='green',
                        title=None, use_abs=True, reduce='sum', normalize=False,
                        median_method='interp', figsize=(20, 7)):
    mag = np.asarray(curl['curl_magnitude'], dtype=float)
    par = np.asarray(curl['parallel'], dtype=float)
    if use_abs:
        par = np.abs(par)

    nz = mag.shape[0]
    w = np.ones(mag.shape, bool) if wall_mask is None else np.asarray(wall_mask, bool)

    def per_layer(a):
        a = np.where(w & np.isfinite(a), a, 0.0)
        s = a.sum(axis=(1, 2))
        if reduce == 'mean':
            n = w.sum(axis=(1, 2))
            s = np.where(n > 0, s / np.maximum(n, 1), np.nan)
        return s

    s_mag, s_par = per_layer(mag), per_layer(par)
    if normalize:
        s_mag = s_mag / np.nansum(s_mag)
        s_par = s_par / np.nansum(s_par)

    z = (np.arange(nz) - (nz - 1) / 2) * voxel_nm

    # mean / median of z for each condition
    mean_mag, med_mag = weighted_stats(z, s_mag, voxel_nm, median_method)
    mean_par, med_par = weighted_stats(z, s_par, voxel_nm, median_method)

    c_par, c_mag = color, darken(color)
    ylabel = {('sum', False): 'Sum of curl over layer',
              ('mean', False): 'Mean curl over layer',
              ('sum', True): 'Fraction of total curl',
              ('mean', True): 'Fraction of total (mean curl)'}[(reduce, normalize)]
    lab_mag = '|curl| (magnitude)'
    lab_par = '|curl·m| (parallel / Bloch)' if use_abs else 'curl·m (parallel)'

    def add_stat_lines(ax):
        ax.axvline(mean_mag, color=c_mag, lw=2, ls='-',
                   label=f'|curl| mean = {mean_mag:.2f} nm')
        ax.axvline(med_mag, color=c_mag, lw=2, ls='-.',
                   label=f'|curl| median = {med_mag:.2f} nm')
        ax.axvline(mean_par, color=c_par, lw=2, ls='-',
                   label=f'|curl·m| mean = {mean_par:.2f} nm')
        ax.axvline(med_par, color=c_par, lw=2, ls='-.',
                   label=f'|curl·m| median = {med_par:.2f} nm')

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize, sharex=True)

    # Left: line profiles
    ax1.plot(z, s_mag, 'o-', color=c_mag, lw=2, ms=8, label=lab_mag)
    ax1.plot(z, s_par, 's-', color=c_par, lw=2, ms=8, label=lab_par)
    ax1.axvline(0, color='k', ls='--', lw=1, label='Film center')
    add_stat_lines(ax1)

    # Right: grouped bars
    bw = 0.4 * voxel_nm
    ax2.bar(z - bw / 2, s_mag, width=bw, color=c_mag, alpha=0.8,
            edgecolor='gray', label=lab_mag)
    ax2.bar(z + bw / 2, s_par, width=bw, color=c_par, alpha=0.6,
            edgecolor='gray', label=lab_par)
    ax2.axvline(0, color='k', ls='--', lw=1, label='Film center')
    add_stat_lines(ax2)

    for ax in (ax1, ax2):
        ax.set_xlabel('z relative to film center (nm)', fontsize=22)
        ax.set_ylabel(ylabel, fontsize=22)
        ax.set_xticks(z)
        ax.set_xticklabels([f'{v:g}' for v in z])
        ax.tick_params(labelsize=14)
        ax.legend(fontsize=13)
    ax1.grid(alpha=0.3)

    fig.suptitle(f'Curl vs. thickness position ({voxel_nm:g} nm layers)\n{title or ""}',
                 fontsize=26)
    plt.tight_layout()
    plt.show()

    ratio = np.where(s_mag > 0, s_par / s_mag, np.nan)
    print("layer  z(nm)   curl_mag      curl_par     par/mag")
    for k in range(nz):
        print(f"{k:5d} {z[k]:6.0f} {s_mag[k]:12.4g} {s_par[k]:12.4g} {ratio[k]:10.3f}")
    print(f"\n|curl|    : mean = {mean_mag:.2f} nm, median = {med_mag:.2f} nm")
    print(f"|curl·m|  : mean = {mean_par:.2f} nm, median = {med_par:.2f} nm")

    return dict(z_nm=z, sum_mag=s_mag, sum_par=s_par, ratio=ratio,
                mean_mag=mean_mag, median_mag=med_mag,
                mean_par=mean_par, median_par=med_par)

In [ ]:
res = plot_curl_sums_vs_z(curl,  voxel_nm=voxel_nm, color=sample_color, title=sample_name, normalize = True)

In [ ]:
def _jsonable(o):
    """Convert numpy types so json can serialize them."""
    if isinstance(o, (np.integer,)):  return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    if isinstance(o, np.ndarray):     return o.tolist()
    if isinstance(o, (np.bool_,)):    return bool(o)
    return str(o)

def save_bloch_result(path, stats, profile, wall_mask, metadata):
    """
    path     : output file, e.g. 'results/sampleA_bloch.npz'
    stats    : dict (or None) returned by plot_Bloch_core_vs_z_m0
    profile  : dict of arrays returned by plot_Bloch_core_vs_z_m0
    wall_mask: boolean array, used only for bookkeeping (shape, voxel count, hash)
    metadata : dict of the inputs you want recorded
    """
    meta = dict(metadata)
    meta.update(
        saved_at=datetime.datetime.now().isoformat(timespec="seconds"),
        numpy_version=np.__version__,
        mask_shape=list(wall_mask.shape),
        mask_voxels=int(wall_mask.sum()),
        mask_sha1=hashlib.sha1(np.packbits(wall_mask.astype(bool)).tobytes()).hexdigest(),
    )

    arrays = {f"profile_{k}": np.asarray(v) for k, v in profile.items()}
    np.savez_compressed(
        path,
        stats_json=json.dumps(stats, default=_jsonable),
        meta_json=json.dumps(meta, default=_jsonable),
        **arrays,
    )
    print(f"saved {path}")

def load_bloch_result(path):
    with np.load(path, allow_pickle=False) as f:
        stats = json.loads(str(f["stats_json"]))
        meta = json.loads(str(f["meta_json"]))
        profile = {k[len("profile_"):]: f[k] for k in f.files if k.startswith("profile_")}
    return stats, profile, meta

In [ ]:
save = False


if save ==  True: 

    save_bloch_result(
        f"data_dir/{sample_name}/{sample_name}_bloch_from_curl_{date}.npz", stats, profile, wall_mask,
        metadata=dict(
            sample_name=sample_name,
            voxel_nm=voxel_nm, bin_nm=voxel_nm,
            min_peak=0.1 * ref, min_peak_fraction=0.1, ref=ref,
            ref_definition="99.9th percentile of |curl|",   # edit to match what you used
            hysteresis_low=lo, hysteresis_high=hi,       # absolute values, same units as curl
            smoothing_sigma_vox=sigma_save,                 # whatever you smoothed m with
            erosion_iterations=2,
            grid_spacing_nm=(voxel_nm, voxel_nm, voxel_nm),                   # spacing passed to np.gradient
            interpolate_z=False,
            notes="",
        ),
    )

In [ ]:
# stop

# evaluate gradient for dw thickness

In [ ]:
from functools import partial

z_idx = 0
Mz_slice = volume['Mz'][z_idx]
Mx_slice = volume['Mx'][z_idx]
My_slice = volume['My'][z_idx]
Grad_slice_2d = frobenius_2d[z_idx]
Grad_slice_3d = frobenius_norm[z_idx]

half = 5
CFG = dict(pixel_size=voxel_nm, half_width=half * voxel_nm, n_samples=81, order=3, sin_min=0.2, model="gauss")

# takes a 2D slice as its first argument, so it works on any z layer
contour_fn = partial(curv.contour_curvature_evenspacing,
                     level=0.0, smoothing=sigma_save,
                     point_spacing=voxel_nm, pixel_size=voxel_nm)

# plot first to check the overlap, then save exactly the results you looked at
fig, ax, results = curv.plot_domain_curvature_evenspace(
    Mz_slice, contour_fn=contour_fn, pixel_size=voxel_nm,
    vmax_percentile=68, unit_label='nm', show_points=True)
plt.show()

curvature_dict = curv.curvature_field_dict(
    contour_fn, Mz_slice, results=results, slice_axis='z', slice_index=z_idx)

In [ ]:
# 2. from isolated spline and saved normals, evaluate model for dw width fitting by comparing residual 
## ultimately gauss was selected because it was model agnostic 

# 452 = 3 
# 665 = 2
#105 = 8
#270 = 0


contour_idx=9
point_idx=410
half = 5


fig, axes, s_gauss, prof_, f_gauss, resid_gauss = curv.plot_normal_cross_section(
    Mz_slice, Grad_slice_2d, results,
    contour_idx=contour_idx, point_idx=point_idx,
    pixel_size=voxel_nm, half_width= half * voxel_nm,
    n_samples=half*2, model="gauss")      

fig, axes, s_sech2, prof_, f_sech2, resid_sech2 = curv.plot_normal_cross_section(
    Mz_slice, Grad_slice_2d, results,
    contour_idx=contour_idx, point_idx=point_idx,
    pixel_size=voxel_nm, half_width= half * voxel_nm,
    n_samples=half*2, model="sech2")  

fig, axes, s_sech, prof_, f_sech, resid_sech = curv.plot_normal_cross_section(
    Mz_slice, Grad_slice_2d, results,
    contour_idx=contour_idx, point_idx=point_idx,
    pixel_size=voxel_nm, half_width= half * voxel_nm,
    n_samples=half*2, model="sech")  

plt.figure(figsize=(8, 5), dpi=150)
plt.plot(s_gauss, resid_gauss, label = f'Gaussian fit \n FWHM = {f_gauss['fwhm']:.1f} nm' )
plt.plot(s_sech2, resid_sech2,  label = f'sech2 fit\n FWHM = {f_sech2['fwhm']:.1f} nm' )
plt.plot(s_sech, resid_sech,  label = f'sech fit\n FWHM = {f_sech['fwhm']:.1f} nm' )
plt.axvline(0, color = 'red', linestyle = '--', label = 'DW center')
plt.axhline(0, color = 'gray', linestyle = '--', label = '0')
plt.xlabel('distance along normal (nm)')
plt.ylabel('residual')
plt.ylim(-0.02, 0.02)
plt.legend(loc = 'upper left')
plt.show()


In [ ]:
# 3. Iterate over one continuoous wall and look at depth dependence 

fw = curv.fwhm_along_contour(Grad_slice_2d, results[contour_idx], voxel_nm, half_width=half * voxel_nm, n_samples = half)
fig, axes = curv.plot_fwhm_along_contour(Mz_slice, results[contour_idx], fw, contour_idx=contour_idx,
                                    pixel_size=voxel_nm, half_width = half * voxel_nm, mark_pt=point_idx)

rows, tracked = curv.fwhm_vs_z(volume['Mz'], frobenius_2d, contour_fn,
                          z_ref=4, ref_result=results[6],
                          pixel_size=voxel_nm, half_width=half * voxel_nm,
                          tol=3* voxel_nm, model = 'gauss')

# fig, axes = curv.plot_fwhm_vs_z_nobin(volume['Mz'], rows, tracked, z_ref=z_idx,
#                            dz_nm=spacing[0], pixel_size=voxel_nm, model = 'gauss')

fig, binned = curv.plot_fwhm_vs_z_bin_onecontour(volume['Mz'], rows, tracked, z_ref=z_idx,
                                   dz_nm=spacing[0], pixel_size=voxel_nm, bin_nm=voxel_nm, model = 'gauss', plot_spline = False)


In [ ]:
# 4. Iterate over all walls and average to get average depth dependence 
rows = curv.fwhm_vs_z_slices(volume['Mz'], frobenius_2d, contour_fn,
                        voxel_nm, half * voxel_nm, z_indices=range(0, len(volume['Mz'])), model = 'gauss')

fig, axes, binned = curv.plot_fwhm_vs_z_slices(volume['Mz'], rows, z_ref=4,
                                          dz_nm=spacing[0], pixel_size=voxel_nm,
                                          bin_nm=voxel_nm, err="stderr", model = 'gauss')


# Evaluate polar and azimuthal angle

In [ ]:
# with surface normal, compute theta and chi and fit with the arctan model for bloch walls then, plot it for all z and compare to Legrand
point_idxs = [120] #, 160, 278, 530]  

for pt in point_idxs:
    contour_slice = contour_fn(Mz_slice)
    psi_window_profile = curv.wall_profile 

    fig, axes, out = curv.plot_angle_decomposition(
        Mx_slice, My_slice, Mz_slice, Grad_slice_2d, contour_slice,
        contour_idx=contour_idx, point_idx=pt, jac=None, k=None, **CFG)

    profiles = curv.psi_vs_z(volume['Mx'], volume['My'], volume['Mz'], frobenius_2d, contour_fn,
                        z_ref=z_idx, ref_result=contour_slice[contour_idx], ref_pt=pt,
                        tol=3 * voxel_nm, window=0.5, **CFG)

    fig, axes = curv.plot_psi_vs_z(volume['Mz'], profiles, z_ref=z_idx,
                            dz_nm=spacing[0], pixel_size=voxel_nm, ylim = (-195, 195), window=0.5, smooth=0)

In [ ]:
point_idxs = [300] #, 160, 278, 530]  

for pt in point_idxs:
    contour_slice = contour_fn(Mz_slice)
    psi_window_profile = curv.wall_profile 

    fig, axes, out = curv.plot_angle_decomposition(
        Mx_slice, My_slice, Mz_slice, Grad_slice_2d, contour_slice,
        contour_idx=contour_idx, point_idx=pt, jac=None, k=None, **CFG)

    profiles = curv.psi_vs_z(volume['Mx'], volume['My'], volume['Mz'], frobenius_2d, contour_fn,
                        z_ref=z_idx, ref_result=contour_slice[contour_idx], ref_pt=pt,
                        tol=2 * voxel_nm, window=1, **CFG)

    fig, axes = curv.plot_psi_vs_z(volume['Mz'], profiles, z_ref=z_idx,
                            dz_nm=spacing[0], pixel_size=voxel_nm, ylim = (-195, 195), window=1, smooth=0)


In [ ]:
# 452 point_idxs = [120, 160, 278, 750]     #452 
# 665 point_idxs = [30, 100, 150, 227]  
# 105 = point_idxs = [90, 130, 175, 400] 
# 270 point_idxs = [125, 300, 550, 805]  
point_idxs = [125, 300, 550, 805]  

tracked = curv.track_wall(volume["Mz"], contour_fn, z_idx, contour_slice[contour_idx], tol=3 * voxel_nm)

step = 2                                   # 300 % 4 == 0, so point_idx is one of the sampled points
out_arc = curv.psi_along_wall(volume["Mx"], volume["My"], volume["Mz"], frobenius_2d,
                         tracked, z_idx, tol=3 * voxel_nm, dz_nm=spacing[0],
                         step=step, reduce="window", window=2, **CFG)


profiles_by_pt = {
    p: curv.psi_vs_z(volume['Mx'], volume['My'], volume['Mz'], frobenius_2d, contour_fn,
                z_ref=z_idx, ref_result=contour_slice[contour_idx], ref_pt=p,
                tol=1 * voxel_nm, window=2, **CFG)
    for p in point_idxs
}



In [ ]:
# r0 = list(out_arc["zs"]).index(out_arc["z_ref"])
# drift = np.hypot(out_arc["x"] - out_arc["x"][r0], out_arc["y"] - out_arc["y"][r0])   # (n_z, n_arc), same units as xy

# np.set_printoptions(threshold=np.inf, linewidth=200, precision=2, suppress=True)
# # or temporarily:
# with np.printoptions(threshold=np.inf, linewidth=200, precision=2):
#     print(drift)

In [ ]:
out  =  out_arc
r0 = list(out["zs"]).index(out["z_ref"])
dx = out["x"] - out["x"][r0]
dy = out["y"] - out["y"][r0]

# reference tangent at the tracked columns
t = np.gradient(out["ref_xy"], axis=0)
t /= np.linalg.norm(t, axis=1, keepdims=True)
t = t[out["idx0"]]                                   # (n_arc, 2)

along  = dx * t[:, 0] + dy * t[:, 1]                 # slide along the wall
across = -dx * t[:, 1] + dy * t[:, 0]                # wall displacement (normal direction)

with np.printoptions(precision=2, suppress=True, linewidth=200):
    for name, a in (("total", np.hypot(dx, dy)), ("along", np.abs(along)), ("across", np.abs(across))):
        print(f"{name:7s} median per z row:", np.nanmedian(a, axis=1))
        print(f"{name:7s} max    per z row:", np.nanmax(a, axis=1))
print("NaN columns per z row:", np.isnan(dx).sum(axis=1))   # columns lost by tracking
print("neighbouring columns on the same contour point (per z row):",
      [len(c) - len(np.unique(np.round(np.c_[x_, y_], 6), axis=0)) if False else None for c, x_, y_ in []] or "see below")

for n, k in enumerate(out["zs"]):
    xy_k = np.c_[out["x"][n], out["y"][n]]
    xy_k = xy_k[~np.isnan(xy_k).any(axis=1)]
    n_dup = len(xy_k) - len(np.unique(xy_k.round(6), axis=0))
    print(f"z={k}: {n_dup} duplicated column positions")

fig, axs = plt.subplots(1, 2, figsize=(14, 3), layout="constrained")
for a, data, ttl in zip(axs, (np.abs(along), np.abs(across)), ("|along-wall slide|", "|across-wall shift|")):
    m = a.pcolormesh(out["arc"], out["zs"], data, shading="nearest")
    fig.colorbar(m, ax=a, label="nm"); a.set_title(ttl)
    a.set_xlabel("arc (nm)"); a.set_ylabel("z index")

In [ ]:
np.median(np.diff(out["arc"]))

In [ ]:
r0 = list(out["zs"]).index(out["z_ref"])
x, y = out["x"], out["y"]

# 1. slice-to-slice movement (tests the "saturating drift" observation)
for name, rows in (("below", range(r0 - 1, -1, -1)), ("above", range(r0 + 1, len(x)))):
    prev = r0
    for n in rows:
        st = np.hypot(x[n] - x[prev], y[n] - y[prev])
        print(f"{name} z={out['zs'][n]}: median step {np.nanmedian(st):.2f}, max {np.nanmax(st):.2f}")
        prev = n

# 2. how often neighbouring columns jump (thr in nm; pick ~ one voxel)
thr = 3.0
drift = np.hypot(x - x[r0], y - y[r0])
print("fraction of column pairs jumping > thr, per z row:",
      np.round((np.abs(np.diff(drift, axis=1)) > thr).mean(axis=1), 3))

In [ ]:
QUANT = {"c2":    ("cos²ψ  (1 = Néel, 0 = Bloch)", 0, 1, "cividis"),
         "neel":  ("−cosψ  (+1 = Néel +, −1 = Néel −)", -1, 1, "RdBu_r"),
         "bloch": ("sinψ  (signed Bloch component)", -1, 1, "RdBu_r"),
         "fwhm":  ("FWHM (nm)", None, None, "viridis")}

def _dpsi_ds(s, psi, good):
    """dψ/ds in deg per unit s. ψ (radians) is unwrapped and differentiated separately
    on each contiguous run of good points, so ±180° wraps don't create spikes and
    gaps in `good` don't get bridged. Points outside the runs are NaN (breaks the line)."""
    d = np.full(len(s), np.nan)
    idx = np.flatnonzero(good)
    if idx.size < 2:
        return d
    for run in np.split(idx, np.flatnonzero(np.diff(idx) > 1) + 1):
        if len(run) >= 2:
            d[run] = np.gradient(np.degrees(np.unwrap(psi[run])), s[run])
    return d


def plot_chi_heatmap_w_width_and_chigrad(out, profiles_by_pt, quantity="c2", z_center_idx=None, unit="nm",
                     dz_nm=None, ylim=(-195, 195), cmap_z="viridis",
                     kcolor="red", line_color="white", center_on="central",
                     figsize=(30, 15), fs=15, num_fs=22,
                     overlay="kappa", wcolor="green", width_mode="mean", width_ref="layer",
                     stagger=90, cbar_pad=None):
    """
    Top: heatmap of `quantity` with optional kappa / width overlays.
    Below: for each selected point, ψ vs position along the normal (row 1) and
    dψ/ds (row 2), one line per z layer.

    center_on : where x = 0 sits in the bottom panels
        "central" -> at the Gaussian-fit center (mu) of the central slice; every z layer
                     is shifted by that same amount, so layers stay comparable
        "contour" -> at the contour itself (s = 0); the central-slice mu is drawn as the
                     red line instead
        "layer"   -> each layer is shifted by its own mu (the old center=True)
    The red dashed line (0) and black dashed lines (±FWHM/2) are always taken from
    the central slice's fit. Central slice = z_center_idx if given, else out["z_ref"]
    (the nearest available slice is used if that one has no profile at a point).

    overlay, wcolor, width_mode, width_ref, stagger, cbar_pad : as before.

    Returns fig, (a1, {"kappa": ax or None, "width": ax or None},
                  {"psi": [axes], "dpsi": [axes]})
    """
    if overlay not in ("kappa", "width", "both", "none"):
        raise ValueError("overlay must be 'kappa', 'width', 'both' or 'none'")
    if width_mode not in ("mean", "per_z", "both"):
        raise ValueError("width_mode must be 'mean', 'per_z' or 'both'")
    if center_on not in ("central", "contour", "layer"):
        raise ValueError("center_on must be 'central', 'contour' or 'layer'")
    show_k = overlay in ("kappa", "both")
    show_w = overlay in ("width", "both")
    if cbar_pad is None:
        cbar_pad = 0.09 if overlay == "both" else 0.02

    lab, vmin, vmax, cmap = QUANT[quantity]
    z = curv._z(out, z_center_idx); arc = out["arc"]
    cm = plt.get_cmap(cmap).reversed(); cm.set_bad("0.85")

    pts = list(profiles_by_pt)
    n = len(pts)
    arc_full = curv.arc_length(out["ref_xy"])
    pt_arc = {p: arc_full[p] for p in pts}
    z_ref = out["z_ref"]
    z_central = z_center_idx if z_center_idx is not None else z_ref

    rc = {"font.family": "serif",
          "font.serif": ["Times New Roman", "Times", "STIXGeneral", "DejaVu Serif"],
          "mathtext.fontset": "stix",
          "font.size": fs, "axes.labelsize": fs,
          "xtick.labelsize": fs, "ytick.labelsize": fs}

    with plt.rc_context(rc):
        fig = plt.figure(figsize=figsize, layout="constrained")
        gs = GridSpec(3, n, figure=fig, height_ratios=[1.2, 1, 0.8])
        a1 = fig.add_subplot(gs[0, :])

        m = a1.pcolormesh(arc, z, np.ma.masked_invalid(out[quantity]), cmap=cm,
                          vmin=vmin, vmax=vmax, shading="nearest")
        cb = fig.colorbar(m, ax=a1, pad=cbar_pad)
        cb.set_label(lab, fontsize=fs)
        cb.ax.tick_params(labelsize=fs)
        a1.set_xlabel(f"arc length along reference wall ({unit})", fontsize=fs)
        a1.set_ylabel(f"z ({unit})" + ("" if z_center_idx is None else ", 0 = film center"),
                      fontsize=fs)

        # white lines where each normal was taken, numbered to match the bottom panels
        ztop = np.nanmax(z)
        for i, p in enumerate(pts):
            a1.axvline(pt_arc[p], color=line_color, lw=2, ls="--")
            a1.text(pt_arc[p], ztop, f" {i+1}", color=line_color, va="top", ha="left",
                    fontsize=num_fs, fontweight="bold")

        ak = aw = None

        # --- kappa overlay (reference slice), kappa = 0 at center ---
        if show_k:
            ak = a1.twinx()
            kappa = out["kappa"][list(out["zs"]).index(out["z_ref"])]
            ak.plot(arc, kappa, color=kcolor, lw=3)
            ak.axhline(0, color="gray", lw=0.8, ls="--")
            kmax = np.nanmax(np.abs(kappa))
            ak.set_ylim(-1.05 * kmax, 1.05 * kmax)
            ak.set_ylabel(f"κ (1/{unit})", color=kcolor, fontsize=fs)
            ak.tick_params(axis="y", labelsize=fs)
            curv._style_twin(ak, kcolor)
            ak.set_xlim(arc.min(), arc.max())

        # --- DW width overlay: change from the mean width (centered on 0) ---
        if show_w:
            W = curv._width_array(out)
            dW, _ = curv._width_deviation(W, width_ref)
            aw = a1.twinx()
            if width_mode in ("per_z", "both"):
                for row in dW:
                    aw.plot(arc, row, color=wcolor, lw=0.8,
                            alpha=0.3 if width_mode == "both" else 0.5)
            if width_mode in ("mean", "both"):
                # with warnings.catch_warnings():
                #     warnings.simplefilter("ignore", RuntimeWarning)
                mean = np.nanmean(dW, axis=0)
                lo, hi = np.nanpercentile(dW, [16, 84], axis=0)
                aw.fill_between(arc, lo, hi, color=wcolor, alpha=0.2, lw=0)
                aw.plot(arc, mean, color=wcolor, lw=3)
            aw.axhline(0, color=wcolor, lw=1, ls="--")
            wmax = np.nanpercentile(np.abs(dW), 99)
            wmax = wmax if np.isfinite(wmax) and wmax > 0 else 1.0
            aw.set_ylim(-1.05 * wmax, 1.05 * wmax)
            aw.set_ylabel(f"Δ DW width from mean ({unit})", color=wcolor, fontsize=fs)
            aw.tick_params(axis="y", labelsize=fs)
            curv._style_twin(aw, wcolor, side_offset=stagger if show_k else None)
            aw.set_xlim(arc.min(), arc.max())

        # --- bottom: ψ (row 1) and dψ/ds (row 2) vs position along the normal ---
        zs_all = sorted(set().union(*[set(pr) for pr in profiles_by_pt.values()]))
        dz = dz_nm if dz_nm is not None else 1.0
        zn_all = np.array(zs_all) * dz
        norm = Normalize(zn_all.min(),
                         zn_all.max() if zn_all.max() > zn_all.min() else zn_all.min() + 1)
        cz = plt.get_cmap(cmap_z)

        psi_axes, dpsi_axes, all_d = [], [], []
        for i, p in enumerate(pts):
            ax = fig.add_subplot(gs[1, i], sharey=psi_axes[0] if psi_axes else None)
            ad = fig.add_subplot(gs[2, i], sharex=ax,
                                 sharey=dpsi_axes[0] if dpsi_axes else None)
            psi_axes.append(ax); dpsi_axes.append(ad)
            prof = profiles_by_pt[p]
            zs = np.array(sorted(prof))

            # central-slice Gaussian fit -> the "0" line and the FWHM edges
            kc = zs[np.argmin(np.abs(zs - z_central))]
            mu_c, fwhm_c = prof[kc]["mu"], prof[kc]["fwhm"]
            have_mu = np.isfinite(mu_c)
            have_fw = np.isfinite(fwhm_c)
            c0 = mu_c if (center_on == "contour" and have_mu) else 0.0

            for kk in range(-4, 5):
                y = 90 * kk
                if -195 <= y <= 195:
                    col = "magenta" if kk % 2 else ("tab:blue" if kk % 4 == 0 else "cyan")
                    ax.axhline(y, color=col, lw=1, alpha=0.6)
            ad.axhline(0, color="gray", lw=1, alpha=0.6)

            for k in zs:
                pk = prof[k]
                if not pk["good"].any():
                    continue
                if center_on == "central":
                    shift = mu_c if have_mu else 0.0
                elif center_on == "layer":
                    shift = pk["mu"] if np.isfinite(pk["mu"]) else 0.0
                else:
                    shift = 0.0
                c = cz(norm(k * dz))
                good = pk["good"]

                x = pk["s"][good] - shift
                y = np.degrees(pk["psi"][good])
                xb, yb = curv._break_jumps(x, y)
                ax.plot(xb, yb, "-", color=c, lw=1.6)
                ax.plot(x, y, ".", color=c, ms=4)

                d = _dpsi_ds(pk["s"], pk["psi"], good)
                xs = pk["s"] - shift
                ad.plot(xs, d, "-", color=c, lw=1.4)
                ad.plot(xs[good], d[good], ".", color=c, ms=4)
                all_d.append(d[np.isfinite(d)])

            for a in (ax, ad):
                if have_mu:
                    a.axvline(c0, color="r", ls="--", lw=1)
                if have_fw:
                    for e in (c0 - fwhm_c / 2, c0 + fwhm_c / 2):
                        a.axvline(e, color="k", lw=1, ls="--")

            ax.set_ylim(ylim)
            ax.set_xlim(-50,50)
            ax.text(0.03, 0.96, f"{i+1}", transform=ax.transAxes, va="top", ha="left",
                    fontsize=num_fs, fontweight="bold",
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.8))
            ax.tick_params(labelsize=fs, labelbottom=False)
            ad.tick_params(labelsize=fs)
            ad.set_xlabel(f"position along normal ({unit})", fontsize=fs)
            if i == 0:
                ax.set_ylabel("Azimuthal angle ψ (deg)", fontsize=fs)
                ad.set_ylabel(f"dψ/ds (deg/{unit})", fontsize=fs)
            else:
                plt.setp(ax.get_yticklabels(), visible=False)
                plt.setp(ad.get_yticklabels(), visible=False)

        # shared dψ/ds limits, robust to a few noisy points
        dcat = np.concatenate(all_d) if all_d else np.array([])
        if dcat.size:
            dmax = np.nanpercentile(np.abs(dcat), 99)
            if np.isfinite(dmax) and dmax > 0:
                # dpsi_axes[0].set_ylim(-1.5 * dmax, 1.5 * dmax)
                dpsi_axes[0].set_xlim(-50, 50)

        sm = plt.cm.ScalarMappable(norm, cz)
        cb2 = fig.colorbar(sm, ax=psi_axes + dpsi_axes, pad=0.02)
        cb2.set_label(f"z ({unit})", fontsize=fs)
        cb2.ax.tick_params(labelsize=fs)

    return fig, (a1, {"kappa": ak, "width": aw}, {"psi": psi_axes, "dpsi": dpsi_axes})

In [ ]:
fig, axes = plot_chi_heatmap_w_width_and_chigrad(out_arc, profiles_by_pt, quantity="c2", dz_nm=spacing[0], overlay="both")

In [ ]:
fig, axes = plot_chi_heatmap_w_width_and_chigrad(out_arc, profiles_by_pt, quantity="bloch", dz_nm=spacing[0], overlay="both")

In [ ]:
fig, axes = plot_chi_heatmap_w_width_and_chigrad(out_arc, profiles_by_pt, quantity="neel", dz_nm=spacing[0], overlay="both")

In [ ]:
# # point_idxs = [32, 74, 100, 130]     
# point_idxs = [120, 160, 278, 530]     #452 




# fig, axes = curv.plot_chi_heatmap_w_width(out_arc, profiles_by_pt, quantity="c2",
#                                   dz_nm=spacing[0], overlay="both")                  # κ + mean width


fig2, ax2 = curv.plot_points_on_spline(Mz_slice, contour_slice[contour_idx], point_idxs, pixel_size=voxel_nm, out=out_arc)


In [ ]:
stop

# Old

### domain wall width from gradients

In [ ]:
## ID contour spline 

# def contour_curvature(field, level=0.0, smoothing=2.0, n_eval=400, min_points=8,
#                        pixel_size=1.0):
#     """
#     Extract the domain boundary/boundaries of `field` at `level` and
#     compute signed curvature along each.

#     Parameters
#     ----------
#     field : (H, W) array
#         Scalar field, e.g. your Mz array. Sign of (field - level) defines
#         the two domains.
#     level : float, default 0.0
#         Threshold defining the boundary. 0.0 is the natural choice for a
#         field like Mz that is symmetric about zero.
#     smoothing : float, default 2.0
#         Spline smoothing strength, passed to scipy as `s = smoothing *
#         n_points_in_contour`. Larger values fit a smoother curve (less
#         sensitive to pixel staircasing, but can round off genuinely sharp
#         features); smaller values track the raw pixel boundary more
#         closely (noisier curvature). Start around 1-3 and adjust by eye
#         against the plot. Applied in pixel-index space, so this doesn't
#         need to change when you change `pixel_size`.
#     n_eval : int, default 400
#         Number of points each contour is resampled to for evaluating
#         curvature (independent of the original pixel contour length).
#     min_points : int, default 8
#         Contours with fewer raw points than this are skipped (too short
#         to reliably spline-fit; usually stray single-pixel noise).
#     pixel_size : float, default 1.0
#         Physical length per pixel (e.g. 2.0 for 2 nm/pixel). Curvature
#         scales as 1/pixel_size and coordinates scale as pixel_size, so
#         with the default of 1.0 everything stays in pixel/voxel units
#         exactly as before. Assumes isotropic (square) pixels.

#     Returns
#     -------
#     list of dicts, one per contour, each with:
#         'xy'    : (n_eval, 2) array of (x, y) coordinates in physical
#                   units (x = column * pixel_size, y = row * pixel_size),
#                   suitable for plotting over `field` with imshow(...,
#                   origin='lower', extent=[0, W*pixel_size, 0, H*pixel_size]).
#         'kappa' : (n_eval,) array of signed curvature, in 1/[pixel_size units].
#     """
#     ## find contour between up and down domains 
#     contours = find_contours(field, level=level)
#     results = []
#     half_width = 10      # in pixels, how far to sample on each side of the wall
#     n_samples  = 41      # points along each normal
#     s = np.linspace(-half_width, half_width, n_samples) 


    
#     for c in contours:
#         if len(c) < min_points:
#             continue
#         rows, cols = c[:, 0], c[:, 1]
#         closed = np.hypot(rows[0] - rows[-1], cols[0] - cols[-1]) < 1.0
#         per = 1 if closed else 0
#         try:
#             ## define a spline of the curve (for each contour, input x y values and fit with a cubic (3) spline)
#             ##  a spline is a function defined in pieces by polynomials that stitch together neighboring points smoothly
#             tck, _ = splprep([cols, rows], s=smoothing * len(c), per=per, k=3)
#         except Exception:
#             continue

#         uu = np.linspace(0, 1, n_eval) ## segments points on the contour from 0 being the beginning of the line and 1 being the end of the line 
#         x, y = splev(uu, tck, der=0) ## evaluated spline and get x y positions
#         dx, dy = splev(uu, tck, der=1) ## first derivative of the spline -----> This is the unit tangent. Rotate this 90 deg. to get the normal 
#         ddx, ddy = splev(uu, tck, der=2) ## second derivative of the spline


#         # unit tangent and unit normal (x = column, y = row)
#         speed = np.hypot(dx, dy)
#         speed = np.where(speed == 0, 1e-12, speed)
#         tx, ty = dx / speed, dy / speed
#         nx, ny = -ty, tx                     # tangent rotated +90 degrees == the surface normal 

#         ## plane curve definition of curvature (https://en.wikipedia.org/wiki/Curvature)
#         denom = (dx**2 + dy**2) ** 1.5
#         kappa = (dx * ddy - dy * ddx) / np.where(denom == 0, 1e-12, denom)
#         # convert pixel-index-space quantities to physical units
#         kappa = kappa / pixel_size
#         xy = np.stack([x, y], axis=1) * pixel_size
#         results.append({"xy": xy, "kappa": kappa, "normal": np.stack([nx, ny], axis=1)})
#     return results


# def compute_profiles(gradient, xy, normal, pixel_size, half_width, n_samples=81, order=3):
#     """
#     Sample `img` along the normal at each point of a contour.

#     img        : 2D array (e.g. gradient magnitude), indexed [row, col]
#     xy         : (N, 2) physical positions (x=col, y=row) * pixel_size
#     normal     : (N, 2) unit normals (nx, ny)
#     half_width : physical distance to sample on each side of the contour
#     Returns
#     s          : (n_samples,) signed physical distance along the normal
#     profiles   : (N, n_samples) sampled values
#     XY         : (N, n_samples, 2) physical sample positions

#     """
#     s = np.linspace(-half_width, half_width, n_samples)

#     Xp = xy[:, 0][:, None] + s[None, :] * normal[:, 0][:, None]   # physical
#     Yp = xy[:, 1][:, None] + s[None, :] * normal[:, 1][:, None]

#     # map_coordinates wants pixel-index (row, col) = (Y, X)
#     profiles = map_coordinates(gradient, [Yp / pixel_size, Xp / pixel_size],
#                                order=order, mode="nearest")
#     return s, profiles, np.stack([Xp, Yp], axis=-1)



# def plot_domain_curvature(
#     field,
#     gradient, 
#     level=0.0,
#     smoothing=2.0,
#     n_eval=400,
#     pixel_size=1.0,
#     unit_label="voxel",
#     cmap="coolwarm",
#     vmax_percentile=98,
#     linewidth=2.5,
#     background_alpha=0.35,
#     figsize=(7, 6),
# ):
#     """
#     Compute and plot domain-wall curvature for `field`, colored with a
#     diverging colormap centered at zero.

#     Parameters
#     ----------
#     pixel_size : float, default 1.0
#         Physical length per pixel (e.g. 2.0 for 2 nm/pixel). See
#         `contour_curvature` -- curvature is reported in 1/[pixel_size units].
#     unit_label : str, default 'voxel'
#         Unit name used in axis labels and the colorbar (e.g. 'nm'). Purely
#         cosmetic -- set this to match whatever `pixel_size` is expressed in.
#     (all other parameters as in `contour_curvature`, plus plot styling)

#     Returns
#     -------
#     (fig, ax, results) where `results` is the same list `contour_curvature`
#     returns, in case you want the raw per-contour curvature values (e.g.
#     for a histogram, or to compare yellow-vs-blue statistics).
#     """
#     results = contour_curvature(field, level=level, smoothing=smoothing,
#                                  n_eval=n_eval, pixel_size=pixel_size)
#     if not results:
#         raise ValueError("No boundary contours found at this level -- check `level` "
#                           "against your field's actual value range.")

#     all_kappa = np.concatenate([r["kappa"] for r in results])
#     vmax = np.percentile(np.abs(all_kappa), vmax_percentile)
#     norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

#     extent = [ 0, field.shape[1] * pixel_size, field.shape[0] * pixel_size, 0]
#     fig, ax = plt.subplots(figsize=figsize)
#     ax.imshow(field > level, origin="upper", cmap="viridis", alpha=background_alpha,
#               extent=extent)

#     for r in results:
#         xy = r["xy"]
#         segs = np.stack([xy[:-1], xy[1:]], axis=1)
#         lc = LineCollection(segs, cmap=cmap, norm=norm)
#         lc.set_array(r["kappa"][:-1])
#         lc.set_linewidth(linewidth)
#         ax.add_collection(lc)

#     ax.set_xlim(extent[0], extent[1])
#     ax.set_ylim(extent[2], extent[3])
#     ax.set_aspect("equal")
#     ax.set_xlabel(f"x ({unit_label})")
#     ax.set_ylabel(f"y ({unit_label})")
#     ax.set_title("Center line curvature ($\\kappa = dT/ds$)")

#     sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
#     fig.colorbar(sm, ax=ax, label=f"κ (1/{unit_label})")
#     fig.tight_layout()

#     n_pos = int((all_kappa > 0).sum())
#     n_neg = int((all_kappa < 0).sum())
#     print(f"{len(results)} boundary contour(s), {len(all_kappa)} points total.")
#     print(f"Convex-into-'field>level' (kappa>0): {n_pos} points, "
#           f"mean {all_kappa[all_kappa>0].mean():.5f}")
#     print(f"Concave / opposite domain intruding (kappa<0): {n_neg} points, "
#           f"mean {all_kappa[all_kappa<0].mean():.5f}")

#     return fig, ax, results

    

In [ ]:
## Gaussian fitting

FWHM_FACTOR = 2 * np.sqrt(2 * np.log(2))   # ~2.355


def gaussian(s, amp, mu, sigma, offset):
    return amp * np.exp(-0.5 * ((s - mu) / sigma) ** 2) + offset


def fit_gaussian(s, prof):
    """
    Fit amp*exp(-(s-mu)^2 / (2 sigma^2)) + offset to a 1D profile.
    Returns a dict with the same keys whether or not the fit succeeds.
    """
    nan = np.nan
    out = dict(amp=nan, mu=nan, sigma=nan, fwhm=nan, offset=nan,
               amp_err=nan, mu_err=nan, sigma_err=nan, fwhm_err=nan,
               r2=nan, success=False)

    ds = s[1] - s[0]
    ok = np.isfinite(prof)
    if ok.sum() < 6:
        return out
    s_, p_ = s[ok], prof[ok]

    # initial guesses
    offset0 = np.min(p_)
    j = np.argmax(p_)
    amp0 = p_[j] - offset0
    mu0 = s_[j]
    above = p_ > offset0 + amp0 / 2
    width = s_[above].max() - s_[above].min() if above.sum() > 1 else 5 * ds
    sigma0 = max(width / FWHM_FACTOR, ds)

    p0 = [amp0, mu0, sigma0, offset0]
    lower = [0, s_[0], ds / 2, -np.inf]
    upper = [np.inf, s_[-1], s_[-1] - s_[0], np.inf]

    try:
        popt, pcov = curve_fit(gaussian, s_, p_, p0=p0, bounds=(lower, upper))
    except (RuntimeError, ValueError):
        return out

    perr = np.sqrt(np.diag(pcov))
    amp, mu, sigma, offset = popt

    resid = p_ - gaussian(s_, *popt)
    ss_tot = np.sum((p_ - p_.mean()) ** 2)
    r2 = 1 - np.sum(resid ** 2) / ss_tot if ss_tot > 0 else nan

    out.update(amp=amp, mu=mu, sigma=sigma, offset=offset,
               fwhm=FWHM_FACTOR * sigma,
               amp_err=perr[0], mu_err=perr[1], sigma_err=perr[2],
               fwhm_err=FWHM_FACTOR * perr[2],
               r2=r2, success=True)
    return out


def plot_normal_cross_section(field, gradient, results, contour_idx, point_idx,
                              pixel_size, half_width, n_samples=81, order=3,
                              cmap="magma", fit = True):
    """
    Panel 1: field map, full spline, and the selected normal (extent = what's profiled).
    Panel 2: profile of `gradient` along that normal.
    """
    r = results[contour_idx]
    xy, normal = r["xy"], r["normal"]
    i = point_idx % len(xy)                      # allows negative indices

    # profile at just the selected point
    s, prof, XY = compute_profiles(gradient, xy[i:i+1], normal[i:i+1],
                                   pixel_size, half_width, n_samples, order)
    s, prof, XY = s, prof[0], XY[0]

    f = fit_gaussian(s, prof) if fit else None


    nrows, ncols = field.shape


    extent = (-0.5 * pixel_size, (ncols - 0.5) * pixel_size,
              (nrows - 0.5) * pixel_size, -0.5 * pixel_size)   # origin="upper"

    fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11, 4.5),
                                   gridspec_kw={"width_ratios": [1.1, 1]})

    # panel 1: field + spline + selected normal
    im = ax0.imshow(gradient, extent=extent, origin="upper", cmap=cmap)
    ax0.plot(xy[:, 0], xy[:, 1], "w-", lw=1.5, label="spline")
    ax0.plot(XY[:, 0], XY[:, 1], "r-", lw=2, label="profile")
    ax0.plot(xy[i, 0], xy[i, 1], "ro", ms=6, mec="w")
    # arrow showing +s direction
    ax0.annotate("", xy=(XY[-1, 0], XY[-1, 1]), xytext=(xy[i, 0], xy[i, 1]),
                 arrowprops=dict(arrowstyle="-", color="r", lw=2))
    ax0.set_xlabel("x"); ax0.set_ylabel("y")
    ax0.set_title(f"contour {contour_idx}, point {i}")
    ax0.legend(loc="upper right", fontsize=8)
    fig.colorbar(im, ax=ax0, fraction=0.046, pad=0.04)

    # panel 2: profile
    ax1.plot(s, prof, "k.-")
    ax1.axvline(0, color="r", ls="--", lw=1, label="DW center")

    if f is not None and f["success"]:
            sf = np.linspace(s[0], s[-1], 500)
            ax1.plot(sf, gaussian(sf, f["amp"], f["mu"], f["sigma"], f["offset"]),
                    "b-", lw=2,
                    # label=(f"Gaussian fit\n"
                    #         f"mean = {f['mu']:.3g} ± {f['mu_err']:.2g}\n"
                    #         f"FWHM = {f['fwhm']:.3g} ± {f['fwhm_err']:.2g}\n"
                    #         f"amp = {f['amp']:.3g}, offset = {f['offset']:.3g}\n"
                    #         f"R² = {f['r2']:.4f}"))
                    label=(f"Gaussian fit\n"
                            f"FWHM = {f['fwhm']:.3g} (nm)"))
            
            # FWHM marker at half maximum
            half = f["offset"] + f["amp"] / 2
            ax1.hlines(half, f["mu"] - f["fwhm"] / 2, f["mu"] + f["fwhm"] / 2,
                    color="c", lw=2)



            # panel 3: residual at the sampled points
            resid = prof - gaussian(s, f["amp"], f["mu"], f["sigma"], f["offset"])
            ax1.axhline(0, color="gray", lw=1)
            ax1.plot(s, resid, "k.-", lw=0.8, alpha = 0.5, label = 'residual')
            
    elif fit:
        ax1.text(0.05, 0.9, "fit failed", transform=ax1.transAxes, color="r")



    ax1.set_xlabel("distance along normal (nm)")
    ax1.set_ylabel("gradient")
    ax1.set_title("cross-section along normal")
    ax1.legend(loc = 'upper right', fontsize = 10)

    fig.tight_layout()
    return fig, (ax0, ax1), s, prof, f, resid




In [ ]:
## sech2 fitting

FWHM_SECH2 = 2 * np.arccosh(np.sqrt(2))     # = 1.7627

def sech2(s, A, mu, w, off):
    return A / np.cosh((s - mu) / w) ** 2 + off

def fit_sech2(s, prof, fix_offset=None):
    out = dict(success=False, A=np.nan, mu=np.nan, w=np.nan, offset=np.nan,
               fwhm=np.nan, mu_err=np.nan, w_err=np.nan, fwhm_err=np.nan, rms=np.nan)
    ok = np.isfinite(prof)
    if ok.sum() < 6:
        return out
    s_, p_ = s[ok], prof[ok]
    ds = s_[1] - s_[0]
    j = np.argmax(p_)
    off0 = p_.min() if fix_offset is None else fix_offset

    f = sech2 if fix_offset is None else (lambda s, A, mu, w: sech2(s, A, mu, w, fix_offset))
    p0 = [p_[j] - off0, s_[j], 4 * ds] + ([off0] if fix_offset is None else [])
    lo = [0, s_[0], ds / 2] + ([-np.inf] if fix_offset is None else [])
    hi = [np.inf, s_[-1], s_[-1] - s_[0]] + ([np.inf] if fix_offset is None else [])
    try:
        popt, pcov = curve_fit(f, s_, p_, p0=p0, bounds=(lo, hi), maxfev=20000)
    except (RuntimeError, ValueError):
        return out
    perr = np.sqrt(np.diag(pcov))
    out.update(success=True, A=popt[0], mu=popt[1], w=popt[2],
               offset=popt[3] if fix_offset is None else fix_offset,
               fwhm=FWHM_SECH2 * popt[2], mu_err=perr[1], w_err=perr[2],
               fwhm_err=FWHM_SECH2 * perr[2],
               rms=np.sqrt(np.mean((p_ - f(s_, *popt)) ** 2)), f=f, popt=popt)
    return out


def plot_sech2_normal_cross_section(field, gradient, results, contour_idx, point_idx,
                              pixel_size, half_width, n_samples=81, order=3,
                              cmap="magma", fit = True):
    """
    Panel 1: field map, full spline, and the selected normal (extent = what's profiled).
    Panel 2: profile of `gradient` along that normal.
    """
    r = results[contour_idx]
    xy, normal = r["xy"], r["normal"]
    i = point_idx % len(xy)                      # allows negative indices

    # profile at just the selected point
    s, prof, XY = compute_profiles(gradient, xy[i:i+1], normal[i:i+1],
                                   pixel_size, half_width, n_samples, order)
    s, prof, XY = s, prof[0], XY[0]

    f = fit_sech2(s, prof)

    nrows, ncols = field.shape
    extent = (-0.5 * pixel_size, (ncols - 0.5) * pixel_size,
              (nrows - 0.5) * pixel_size, -0.5 * pixel_size)   # origin="upper"

    fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11, 4.5),
                                   gridspec_kw={"width_ratios": [1.1, 1]})

    # panel 1: field + spline + selected normal
    im = ax0.imshow(gradient, extent=extent, origin="upper", cmap=cmap)
    ax0.plot(xy[:, 0], xy[:, 1], "w-", lw=1.5, label="spline")
    ax0.plot(XY[:, 0], XY[:, 1], "r-", lw=2, label="profile")
    ax0.plot(xy[i, 0], xy[i, 1], "ro", ms=6, mec="w")
    # arrow showing +s direction
    ax0.annotate("", xy=(XY[-1, 0], XY[-1, 1]), xytext=(xy[i, 0], xy[i, 1]),
                 arrowprops=dict(arrowstyle="-", color="r", lw=2))
    ax0.set_xlabel("x"); ax0.set_ylabel("y")
    ax0.set_title(f"contour {contour_idx}, point {i}")
    ax0.legend(loc="upper right", fontsize=8)
    fig.colorbar(im, ax=ax0, fraction=0.046, pad=0.04)

    # panel 2: profile
    ax1.plot(s, prof, "k.-")
    ax1.axvline(0, color="r", ls="--", lw=1, label="DW center")

    if f["success"]:
        sf = np.linspace(s[0], s[-1], 500)
        ax1.plot(sf, sech2(sf, f["A"], f["mu"], f["w"], f["offset"]), "b-", lw=2,
                label=f"sech² fit\nmean = {f['mu']:.3g} ± {f['mu_err']:.2g}\n"
                    f"w = {f['w']:.3g} ± {f['w_err']:.2g}\nFWHM = {f['fwhm']:.3g}")
        # FWHM marker at half maximum
        half = f["offset"] + f["A"] / 2
        ax1.hlines(half, f["mu"] - f["fwhm"] / 2, f["mu"] + f["fwhm"] / 2,
                color="c", lw=2)
        ax1.axvline(f["mu"], color="c", ls=":", lw=1)

        # panel 3: residual at the sampled points
        resid = prof - f["f"](s, *f["popt"])
        ax1.axhline(0, color="gray", lw=1)
        ax1.plot(s, resid, "k.-", lw=0.8, alpha = 0.5, label = 'residual')
        
        
    elif fit:
        ax1.text(0.05, 0.9, "fit failed", transform=ax1.transAxes, color="r")

    ax1.set_xlabel("distance along normal (nm)")
    ax1.set_ylabel("gradient")
    ax1.set_title("cross-section along normal")
    ax1.legend(loc = 'upper right', fontsize = 10)

    fig.tight_layout()
    return fig, (ax0, ax1), s, prof, f, resid



def fit_contour_sech2(gradient, result, pixel_size, half_width, n_samples=161, **kw):
    s, profiles, _ = compute_profiles(gradient, result["xy"], result["normal"],
                                      pixel_size, half_width, n_samples)
    fits = [fit_sech2(s, p, **kw) for p in profiles]
    keys = ["success", "A", "mu", "w", "offset", "fwhm", "mu_err", "w_err", "fwhm_err", "rms"]
    result["fit"] = {k: np.array([f[k] for f in fits]) for k in keys}
    result["s"], result["profiles"] = s, profiles
    return result["fit"]


In [ ]:
## sech fitting

FWHM_SECH = 2 * np.arccosh((2))     

def sech(s, A, mu, w, off):
    return A / np.cosh((s - mu) / w) + off

def fit_sech(s, prof, fix_offset=None):
    out = dict(success=False, A=np.nan, mu=np.nan, w=np.nan, offset=np.nan,
               fwhm=np.nan, mu_err=np.nan, w_err=np.nan, fwhm_err=np.nan, rms=np.nan)
    ok = np.isfinite(prof)
    if ok.sum() < 6:
        return out
    s_, p_ = s[ok], prof[ok]
    ds = s_[1] - s_[0]
    j = np.argmax(p_)
    off0 = p_.min() if fix_offset is None else fix_offset

    f = sech if fix_offset is None else (lambda s, A, mu, w: sech(s, A, mu, w, fix_offset))
    p0 = [p_[j] - off0, s_[j], 4 * ds] + ([off0] if fix_offset is None else [])
    lo = [0, s_[0], ds / 2] + ([-np.inf] if fix_offset is None else [])
    hi = [np.inf, s_[-1], s_[-1] - s_[0]] + ([np.inf] if fix_offset is None else [])
    try:
        popt, pcov = curve_fit(f, s_, p_, p0=p0, bounds=(lo, hi), maxfev=20000)
    except (RuntimeError, ValueError):
        return out
    perr = np.sqrt(np.diag(pcov))
    out.update(success=True, A=popt[0], mu=popt[1], w=popt[2],
               offset=popt[3] if fix_offset is None else fix_offset,
               fwhm=FWHM_SECH * popt[2], mu_err=perr[1], w_err=perr[2],
               fwhm_err=FWHM_SECH * perr[2],
               rms=np.sqrt(np.mean((p_ - f(s_, *popt)) ** 2)), f=f, popt=popt)
    return out


def fit_contour_sech(gradient, result, pixel_size, half_width, n_samples=161, **kw):
    s, profiles, _ = compute_profiles(gradient, result["xy"], result["normal"],
                                      pixel_size, half_width, n_samples)
    fits = [fit_sech(s, p, **kw) for p in profiles]
    keys = ["success", "A", "mu", "w", "offset", "fwhm", "mu_err", "w_err", "fwhm_err", "rms"]
    result["fit"] = {k: np.array([f[k] for f in fits]) for k in keys}
    result["s"], result["profiles"] = s, profiles
    return result["fit"]



def plot_sech_normal_cross_section(field, gradient, results, contour_idx, point_idx,
                              pixel_size, half_width, n_samples=81, order=3,
                              cmap="magma", fit = True):
    """
    Panel 1: field map, full spline, and the selected normal (extent = what's profiled).
    Panel 2: profile of `gradient` along that normal.
    """
    r = results[contour_idx]
    xy, normal = r["xy"], r["normal"]
    i = point_idx % len(xy)                      # allows negative indices

    # profile at just the selected point
    s, prof, XY = compute_profiles(gradient, xy[i:i+1], normal[i:i+1],
                                   pixel_size, half_width, n_samples, order)
    s, prof, XY = s, prof[0], XY[0]

    f = fit_sech(s, prof)

    nrows, ncols = field.shape
    extent = (-0.5 * pixel_size, (ncols - 0.5) * pixel_size,
              (nrows - 0.5) * pixel_size, -0.5 * pixel_size)   # origin="upper"

    fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11, 4.5),
                                   gridspec_kw={"width_ratios": [1.1, 1]})

    # panel 1: field + spline + selected normal
    im = ax0.imshow(gradient, extent=extent, origin="upper", cmap=cmap)
    ax0.plot(xy[:, 0], xy[:, 1], "w-", lw=1.5, label="spline")
    ax0.plot(XY[:, 0], XY[:, 1], "r-", lw=2, label="profile")
    ax0.plot(xy[i, 0], xy[i, 1], "ro", ms=6, mec="w")
    # arrow showing +s direction
    ax0.annotate("", xy=(XY[-1, 0], XY[-1, 1]), xytext=(xy[i, 0], xy[i, 1]),
                 arrowprops=dict(arrowstyle="-", color="r", lw=2))
    ax0.set_xlabel("x"); ax0.set_ylabel("y")
    ax0.set_title(f"contour {contour_idx}, point {i}")
    ax0.legend(loc="upper right", fontsize=8)
    fig.colorbar(im, ax=ax0, fraction=0.046, pad=0.04)

    # panel 2: profile
    ax1.plot(s, prof, "k.-")
    ax1.axvline(0, color="r", ls="--", lw=1, label="DW center")

    if f["success"]:
        sf = np.linspace(s[0], s[-1], 500)
        ax1.plot(sf, sech(sf, f["A"], f["mu"], f["w"], f["offset"]), "b-", lw=2,
                label=f"sech fit\nmean = {f['mu']:.3g} ± {f['mu_err']:.2g}\n"
                    f"w = {f['w']:.3g} ± {f['w_err']:.2g}\nFWHM = {f['fwhm']:.3g}")
        # FWHM marker at half maximum
        half = f["offset"] + f["A"] / 2
        ax1.hlines(half, f["mu"] - f["fwhm"] / 2, f["mu"] + f["fwhm"] / 2,
                color="c", lw=2)
        ax1.axvline(f["mu"], color="c", ls=":", lw=1)

        # panel 3: residual at the sampled points
        resid = prof - f["f"](s, *f["popt"])
        ax1.axhline(0, color="gray", lw=1)
        ax1.plot(s, resid, "k.-", lw=0.8, alpha = 0.5, label = 'residual')
        
        
    elif fit:
        ax1.text(0.05, 0.9, "fit failed", transform=ax1.transAxes, color="r")

    ax1.set_xlabel("distance along normal (nm)")
    ax1.set_ylabel("gradient")
    ax1.set_title("cross-section along normal")
    ax1.legend(loc = 'upper right', fontsize = 10)

    fig.tight_layout()
    return fig, (ax0, ax1), s, prof, f, resid

In [ ]:
# Plot test

z_idx = 2


Mz_slice = volume['Mz'][z_idx]
Mx_slice = volume['Mx'][z_idx]
My_slice = volume['My'][z_idx]
Grad_slice_2d = frobenius_2d[z_idx]
Grad_slice_3d = frobenius_norm[z_idx]

sigma_save = sigma_save
results = contour_curvature(Mz_slice, level=0.0, smoothing=sigma_save, n_eval=400, pixel_size=voxel_nm)
fig, ax, ___ = plot_domain_curvature(Mz_slice, Grad_slice_2d, level=0.0, smoothing=sigma_save, pixel_size=voxel_nm, vmax_percentile = 68, unit_label='nm')

curvature_dict = curvature_field_dict(
    results, Mz_slice.shape, pixel_size=voxel_nm, level=0.0,
    slice_axis='z', slice_index=z_idx,
)


# # or all profiles for a contour at once, for averaging, binning by kappa, etc.
# s, profiles, XY = compute_profiles(gradient, results[0]["xy"], results[0]["normal"],
#                                    pixel_size, half_width=10 * pixel_size)

In [ ]:
contour_idx=3
point_idx=125
half = 5

fig, axes, s_gauss, prof_gauss, f_gauss, resid_gauss = plot_normal_cross_section(
    Mz_slice, Grad_slice_2d, results,
    contour_idx=contour_idx, point_idx=point_idx,
    pixel_size=voxel_nm, half_width= half * voxel_nm,
)


fig, axes, s_sech2, prof_sech2, f_sech2, resid_sech2 = plot_sech2_normal_cross_section(
    Mz_slice, Grad_slice_2d, results,
    contour_idx=contour_idx, point_idx=point_idx,
    pixel_size=voxel_nm, half_width=half * voxel_nm,
)

fig, axes, s_sech, prof_sech, f_sech, resid_sech = plot_sech_normal_cross_section(
    Mz_slice, Grad_slice_3d, results,
    contour_idx=contour_idx, point_idx=point_idx,
    pixel_size=voxel_nm, half_width=half* voxel_nm,
)

In [ ]:
plt.plot(s_gauss, resid_gauss, label = f'Gaussian fit \n FWHM = {f_gauss['fwhm']:.1f} nm' )
plt.plot(s_sech2, resid_sech2,  label = f'sech2 fit\n FWHM = {f_sech2['fwhm']:.1f} nm' )
plt.plot(s_sech, resid_sech,  label = f'sech fit\n FWHM = {f_sech['fwhm']:.1f} nm' )
plt.axvline(0, color = 'red', linestyle = '--', label = 'DW center')
plt.axhline(0, color = 'gray', linestyle = '--', label = '0')
plt.xlabel('distance along normal (nm)')
plt.ylabel('residual')
plt.ylim(-0.02, 0.02)
plt.legend(loc = 'upper left')
plt.show()

In [ ]:
## look at half_width dependence on fit thickness

half = 5

fig, axes, s_5, prof_5, f_5, resid_5 = plot_normal_cross_section(
    Mz_slice, Grad_slice_2d, results,
    contour_idx=contour_idx, point_idx=point_idx,
    pixel_size=voxel_nm, half_width= half * voxel_nm,
)

half = 8

fig, axes, s_8, prof_5, f_8, resid_8 = plot_normal_cross_section(
    Mz_slice, Grad_slice_2d, results,
    contour_idx=contour_idx, point_idx=point_idx,
    pixel_size=voxel_nm, half_width= half * voxel_nm,
)

half = 7

fig, axes, s_10, prof_5, f_10, resid_10 = plot_normal_cross_section(
    Mz_slice, Grad_slice_2d, results,
    contour_idx=contour_idx, point_idx=point_idx,
    pixel_size=voxel_nm, half_width= half * voxel_nm,
)




In [ ]:
plt.plot(s_5, resid_5, label = f'half = 5 \n FWHM = {f_5['fwhm']:.1f} nm' )
plt.plot(s_8, resid_8,  label = f'half = 8 \n FWHM = {f_8['fwhm']:.1f} nm' )
plt.plot(s_10, resid_10,  label = f'half = 7 \n FWHM = {f_10['fwhm']:.1f} nm' )
plt.axvline(0, color = 'red', linestyle = '--', label = 'DW center')
plt.axhline(0, color = 'gray', linestyle = '--', label = '0')
plt.xlabel('distance along normal (nm)')
plt.ylabel('residual')
plt.ylim(-0.02, 0.02)
plt.legend(loc = 'upper left')
plt.show()


### thickness dependence

In [ ]:
# Using sech, plot thickness along contour. 

# def _sech(x):
#     return 1 / np.cosh(x)

# MODELS = {
#     # name: (func, extra_p0, extra_lower, extra_upper)
#     "gauss":   (lambda s, A, mu, w, off: A*np.exp(-0.5*((s-mu)/w)**2) + off, [], [], []),
#     "sech2":   (lambda s, A, mu, w, off: A*_sech((s-mu)/w)**2 + off,         [], [], []),
#     "sech":    (lambda s, A, mu, w, off: A*_sech((s-mu)/w) + off,            [], [], []),

# }


# # FWHM = factor * w   (for the Gaussian, w is sigma)
# FWHM_FACTOR = {"gauss": 2*np.sqrt(2*np.log(2)),      # 2.355
#                "sech":  2*np.arccosh(2),             # 2.634
#                "sech2": 2*np.arccosh(np.sqrt(2))}    # 1.763
# LABEL = {"gauss": "Gaussian", "sech": "sech", "sech2": "sech²"}


# def fit_peak(s, prof, model="gauss"):
#     """Fit A*shape((s-mu)/w) + offset. Returns the same keys for every model."""
#     func, k = MODELS[model][0], FWHM_FACTOR[model]
#     nan = np.nan
#     out = dict(success=False, A=nan, mu=nan, w=nan, offset=nan, fwhm=nan, rms=nan,
#                mu_err=nan, w_err=nan, fwhm_err=nan)
#     ok = np.isfinite(prof)
#     if ok.sum() < 6:
#         return out
#     s_, p_ = s[ok], prof[ok]
#     ds = s_[1] - s_[0]
#     j = np.argmax(p_)
#     off0 = p_.min()
#     A0 = p_[j] - off0
#     above = p_ > off0 + A0 / 2
#     width0 = s_[above].max() - s_[above].min() if above.sum() > 1 else 4 * ds
#     w0 = max(width0 / k, ds)                      # start from the half-max width

#     try:
#         popt, pcov = curve_fit(func, s_, p_, p0=[A0, s_[j], w0, off0],
#                                bounds=([0, s_[0], ds/2, -np.inf],
#                                        [np.inf, s_[-1], s_[-1] - s_[0], np.inf]),
#                                maxfev=20000)
#     except (RuntimeError, ValueError):
#         return out
#     perr = np.sqrt(np.diag(pcov))
#     out.update(success=True, A=popt[0], mu=popt[1], w=popt[2], offset=popt[3],
#                fwhm=k * popt[2], mu_err=perr[1], w_err=perr[2], fwhm_err=k * perr[2],
#                rms=np.sqrt(np.mean((p_ - func(s_, *popt)) ** 2)), f=func, popt=popt)
#     return out

# def plot_normal_cross_section(field, gradient, results, contour_idx, point_idx,
#                               pixel_size, half_width, n_samples=81, order=3,
#                               cmap="magma", fit=True, model="gauss", unit="nm",
#                               detailed_label=False):
#     """
#     Panel 1: gradient map, full spline, and the selected normal (extent = what's profiled).
#     Panel 2: profile of `gradient` along that normal, with a `model` fit and its residual.
#     model: any key of MODELS ("gauss", "sech", "sech2").
#     """
#     r = results[contour_idx]
#     xy, normal = r["xy"], r["normal"]
#     i = point_idx % len(xy)                      # allows negative indices

#     # profile at just the selected point
#     s, prof, XY = compute_profiles(gradient, xy[i:i+1], normal[i:i+1],
#                                    pixel_size, half_width, n_samples, order)
#     prof, XY = prof[0], XY[0]

#     f = fit_peak(s, prof, model) if fit else None
#     resid = np.full_like(prof, np.nan)           # defined even if the fit fails or is off

#     nrows, ncols = field.shape
#     extent = (-0.5 * pixel_size, (ncols - 0.5) * pixel_size,
#               (nrows - 0.5) * pixel_size, -0.5 * pixel_size)   # origin="upper"

#     fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11, 4.5),
#                                    gridspec_kw={"width_ratios": [1.1, 1]})

#     # panel 1: gradient map + spline + selected normal
#     im = ax0.imshow(gradient, extent=extent, origin="upper", cmap=cmap)
#     ax0.plot(xy[:, 0], xy[:, 1], "w-", lw=1.5, label="spline")
#     ax0.plot(XY[:, 0], XY[:, 1], "r-", lw=2, label="profile")
#     ax0.plot(xy[i, 0], xy[i, 1], "ro", ms=6, mec="w")
#     ax0.set_xlabel(f"x ({unit})"); ax0.set_ylabel(f"y ({unit})")
#     ax0.set_title(f"contour {contour_idx}, point {i}")
#     ax0.legend(loc="upper right", fontsize=8)
#     fig.colorbar(im, ax=ax0, fraction=0.046, pad=0.04)

#     # panel 2: profile + fit
#     ax1.plot(s, prof, "k.-", label="data")
#     ax1.axvline(0, color="r", ls="--", lw=1, label="DW center")

#     if f is not None and f["success"]:
#         sf = np.linspace(s[0], s[-1], 500)
#         label = f"{LABEL[model]} fit\nFWHM = {f['fwhm']:.3g} {unit}"
#         if detailed_label:
#             wname = "σ" if model == "gauss" else "w"
#             label += (f"\nmean = {f['mu']:.3g} ± {f['mu_err']:.2g}"
#                       f"\n{wname} = {f['w']:.3g} ± {f['w_err']:.2g}"
#                       f"\nA = {f['A']:.3g}, offset = {f['offset']:.3g}")
#         ax1.plot(sf, f["f"](sf, *f["popt"]), "b-", lw=2, label=label)

#         # FWHM marker at half maximum of the fitted curve (valid for all three shapes)
#         half = f["offset"] + f["A"] / 2
#         ax1.hlines(half, f["mu"] - f["fwhm"] / 2, f["mu"] + f["fwhm"] / 2,
#                    color="c", lw=2)

#         # residual at the sampled points
#         resid = prof - f["f"](s, *f["popt"])
#         ax1.axhline(0, color="gray", lw=1)
#         ax1.plot(s, resid, "k.-", lw=0.8, alpha=0.5, label="residual")
#     elif fit:
#         ax1.text(0.05, 0.9, "fit failed", transform=ax1.transAxes, color="r")

#     ax1.set_xlabel(f"distance along normal ({unit})")
#     ax1.set_ylabel("gradient")
#     ax1.set_title("cross-section along normal")
#     ax1.legend(loc="upper right", fontsize=10)

#     fig.tight_layout()
#     return fig, (ax0, ax1), s, prof, f, resid


# def fwhm_from_curve(s, y, baseline):
#     """Half-max width by linear interpolation of the crossings around the peak."""
#     j = np.nanargmax(y)
#     half = baseline + (y[j] - baseline) / 2
#     l = j
#     while l > 0 and y[l] > half:
#         l -= 1
#     r = j
#     while r < len(y) - 1 and y[r] > half:
#         r += 1
#     if y[l] > half or y[r] > half:        # never dropped below half max
#         return np.nan, np.nan, np.nan
#     sl = s[l] + (half - y[l]) * (s[l+1] - s[l]) / (y[l+1] - y[l])
#     sr = s[r-1] + (half - y[r-1]) * (s[r] - s[r-1]) / (y[r] - y[r-1])
#     return sr - sl, sl, sr


# def fwhm_along_contour(gradient, result, pixel_size, half_width,
#                        n_samples=81, order=3, model="gauss"):
#     """Fit `model` to the normal profile at every point of one contour."""
#     s, profiles, XY = compute_profiles(gradient, result["xy"], result["normal"],
#                                        pixel_size, half_width, n_samples, order)
#     N = len(profiles)
#     ny, nx = gradient.shape
#     inside = ((XY[..., 0] >= 0) & (XY[..., 0] <= (nx - 1) * pixel_size) &
#               (XY[..., 1] >= 0) & (XY[..., 1] <= (ny - 1) * pixel_size)).all(axis=1)

#     keys = ["fwhm", "w", "mu", "A", "offset", "rms", "fwhm_err", "mu_err", "w_err"]
#     out = {k: np.full(N, np.nan) for k in keys}
#     out["fwhm_direct"] = np.full(N, np.nan)
#     out["success"] = np.zeros(N, bool)

#     for j, p in enumerate(profiles):
#         if not (inside[j] and np.all(np.isfinite(p))):
#             continue
#         out["fwhm_direct"][j] = fwhm_from_curve(s, p, p.min())[0]   # model-free
#         f = fit_peak(s, p, model)
#         if f["success"]:
#             out["success"][j] = True
#             for k in keys:
#                 out[k][j] = f[k]

#     out.update(inside=inside, pt=np.arange(N), half_width=half_width, model=model)
#     return out

# def plot_fwhm_along_contour(field, result, fw, contour_idx, pixel_size, half_width, model = 'gauss', 
#                             mark_pt=None, unit="nm", ):
#     # quality mask: fit worked, profile inside image, FWHM not truncated by the window
#     good = fw["success"] & fw["inside"] & (fw["fwhm"] < fw["half_width"])
#     xy = result["xy"]

#     fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(13, 4.5),
#                                    gridspec_kw={"width_ratios": [1, 1.3]})

#     # left: contour colored by fitted FWHM
#     ny, nx = field.shape
#     extent = (-0.5 * pixel_size, (nx - 0.5) * pixel_size,
#               (ny - 0.5) * pixel_size, -0.5 * pixel_size)
#     ax0.imshow(field, extent=extent, origin="upper", cmap="viridis")
#     sc = ax0.scatter(xy[good, 0], xy[good, 1], c=fw["fwhm"][good], s=8, cmap='bwr')
#     ax0.plot(xy[~good, 0], xy[~good, 1], ".", c="k", ms=3, label="rejected")
#     if mark_pt is not None:
#         ax0.plot(*xy[mark_pt], "co", ms=8, mfc="none")
#     fig.colorbar(sc, ax=ax0, fraction=0.046, pad=0.04, label=f"FWHM ({unit})")
#     ax0.set_xlabel(f"x ({unit})"); ax0.set_ylabel(f"y ({unit})")
#     ax0.set_title(f"contour {contour_idx}"); ax0.legend(fontsize=8)

#     # right: FWHM vs point index
#     #ax1.errorbar(fw["pt"][good], fw["fwhm"][good], yerr= fw["fwhm_err"][good], color = "b", linestyle = ".", label=f"{model} fit")
#     ax1.errorbar(fw["pt"][good], fw["fwhm"][good], yerr=fw["fwhm_err"][good], color="b", marker="o", fmt = ' ', label=f"{model} fit")
#     ax1.axhline(np.mean(fw["fwhm"][good]), color = "b", label = f"avg dw width = {np.mean(fw["fwhm"][good]):.0f} nm")
#     # ax1.plot(fw["pt"][good], fw["fwhm_direct"][good], ".", c="orange", ms=4,
#     #          label="direct (model-free)")
#     # ax1.errorbar(fw["pt"][~good], fw["fwhm"][~good], yerr= fw["fwhm_err"][~good], color ="r", marker = "o", fmt = ' ', ms=4, label="double peak")
#     ax1.plot(fw["pt"][~good], fw["fwhm"][~good], "rx", ms=4, label="double peak")
#     if mark_pt is not None:
#         ax1.axvline(mark_pt, c="c", ls=":")
#     ax1.set_xlabel("point on spline"); ax1.set_ylabel(f"FWHM ({unit})")
#     ax1.set_title(f"FWHM along contour for profile length = {2* half_width} nm"); 
#     ax1.legend(fontsize=10, loc = 'upper right')
#     fig.tight_layout()
#     return fig, (ax0, ax1)




In [ ]:
half = 5

contour_idx=3
point_idx=125

fig, axes, s_, prof_, f_, resid_ = plot_normal_cross_section(
    Mz_slice, Grad_slice_2d, results,
    contour_idx=contour_idx, point_idx=point_idx,
    pixel_size=voxel_nm, half_width= half * voxel_nm,
    n_samples=81, model="gauss")      

fw = fwhm_along_contour(Grad_slice_2d, results[contour_idx], voxel_nm, half_width=half * voxel_nm)
fig, axes = plot_fwhm_along_contour(Mz_slice, results[contour_idx], fw, contour_idx=contour_idx,
                                    pixel_size=voxel_nm, half_width = half * voxel_nm, mark_pt=point_idx)
print(f"{fw['success'].sum()} / {len(fw['pt'])} fits succeeded")

In [ ]:
# from scipy.spatial import cKDTree

# def overlap_score(xyA, xyB, tol):
#     """Fraction of points of each curve lying within tol of the other (1 = same curve)."""
#     dAB = cKDTree(xyB).query(xyA)[0]
#     dBA = cKDTree(xyA).query(xyB)[0]
#     return 0.5 * ((dAB < tol).mean() + (dBA < tol).mean())


# def find_match(ref_xy, contours, tol, min_overlap=0.3):
#     if not contours:
#         return None, 0.0
#     scores = [overlap_score(ref_xy, c["xy"], tol) for c in contours]
#     j = int(np.argmax(scores))
#     return (j, scores[j]) if scores[j] >= min_overlap else (None, scores[j])


# def fwhm_vs_z(Mz3, grad3, contour_fn, z_ref, ref_result, pixel_size, half_width,
#               tol, min_overlap=0.3, common_tol=None, model = 'gauss'):
#     """
#     Follow one wall through z and summarize its FWHM in each slice.
#     contour_fn  : function(Mz_slice) -> list of contour dicts
#     ref_result  : the contour dict at z_ref (e.g. results[6])
#     tol         : max wall displacement between adjacent slices (same units as xy)
#     common_tol  : if set, only use points within this distance of the z_ref contour,
#                   so every slice averages over roughly the same stretch of wall
#     """
#     nz = len(Mz3)
#     rows, tracked = {}, {}

#     def summarize(k, r, score):
#         fw = fwhm_along_contour(grad3[k], r, pixel_size, half_width, model = 'gauss')
#         good = fw["success"] & fw["inside"] & (fw["fwhm"] < fw["half_width"])
#         if common_tol is not None:
#             good &= cKDTree(ref_result["xy"]).query(r["xy"])[0] < common_tol
#         v = fw["fwhm"][good]
#         rows[k] = dict(
#             n_good=int(good.sum()), n_pts=len(good), overlap=score,
#             mean=np.mean(v) if len(v) else np.nan,
#             median=np.median(v) if len(v) else np.nan,
#             std=np.std(v, ddof=1) if len(v) > 1 else np.nan,
#             stderr = np.std(v, ddof=1)/np.sqrt(len(v)) if len(v) > 1 else np.nan,
#             vals=v.copy(),
#             p16=np.percentile(v, 16) if len(v) else np.nan,
#             p84=np.percentile(v, 84) if len(v) else np.nan,
#             direct=np.nanmedian(fw["fwhm_direct"][good]) if len(v) else np.nan)
#         tracked[k] = r

#     summarize(z_ref, ref_result, 1.0)

#     for direction in (+1, -1):                    # walk up, then down, from z_ref
#         ref_xy = ref_result["xy"]
#         k = z_ref + direction
#         while 0 <= k < nz:
#             contours = contour_fn(Mz3[k])
#             j, score = find_match(ref_xy, contours, tol, min_overlap)
#             if j is None:
#                 print(f"lost the wall at z={k} (best overlap {score:.2f})")
#                 break
#             summarize(k, contours[j], score)
#             ref_xy = contours[j]["xy"]            # chain: compare with the previous slice
#             k += direction
#     return rows, tracked

# def plot_fwhm_vs_z_nobin(Mz3, rows, tracked, z_ref, dz_nm=None, pixel_size=1.0, unit="nm", model = 'gauss'):
#     zs = np.array(sorted(rows))
#     med = np.array([rows[k]["median"] for k in zs])
#     lo = np.array([rows[k]["p16"] for k in zs])
#     hi = np.array([rows[k]["p84"] for k in zs])
#     mean = np.array([rows[k]["mean"] for k in zs])
#     n = np.array([rows[k]["n_good"] for k in zs])

#     fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw={"width_ratios": [1, 1.2]})

#     # left: check that the tracking followed the same wall
#     ny, nx = Mz3[z_ref].shape
#     extent = (-0.5 * pixel_size, (nx - 0.5) * pixel_size, (ny - 0.5) * pixel_size, -0.5 * pixel_size)
#     ax0.imshow(Mz3[z_ref], extent=extent, origin="upper", cmap="gray")
#     cmap = plt.get_cmap("viridis")
#     for k in zs:
#         xy = tracked[k]["xy"]
#         ax0.plot(xy[:, 0], xy[:, 1], "-", lw=1.2, color=cmap((k - zs[0]) / max(1, zs[-1] - zs[0])))
#     sm = plt.cm.ScalarMappable(plt.Normalize(zs[0], zs[-1]), "viridis")
#     fig.colorbar(sm, ax=ax0, fraction=0.046, pad=0.04, label="z index")
#     ax0.set_title(f"tracked wall (background: Mz at z={z_ref})")
#     ax0.set_xlabel(f"x ({unit})"); ax0.set_ylabel(f"y ({unit})")

#     # right: FWHM vs z
#     ax1.errorbar(zs, med, yerr=[med - lo, hi - med], fmt="bo-", capsize=3, label="median (16-84%)")
#     ax1.plot(zs, mean, "s", c="c", ms=4, label="mean")
#     ax1.axvline(z_ref, c="gray", ls=":", label="reference slice")
#     for k, nn in zip(zs, n):
#         ax1.annotate(str(nn), (k, ax1.get_ylim()[0]), ha="center", va="bottom", fontsize=7, color="gray")
#     ax1.set_xlabel("z index"); ax1.set_ylabel(f"FWHM of {model} fit ({unit})")
#     ax1.set_title("FWHM vs. z (numbers = points used)")
#     if dz_nm:
#         sec = ax1.secondary_xaxis("top", functions=(lambda z: z * dz_nm, lambda z: z / dz_nm))
#         sec.set_xlabel(f"z ({unit})")
#     ax1.legend(fontsize=8)
#     fig.tight_layout()
#     return fig, (ax0, ax1)


# def bin_fwhm_by_z(rows, dz_nm, bin_nm=8.0):
#     """
#     Pool the per-point FWHM values of all slices falling in each z bin.
#     Bins start at 0 nm: [0, 8), [8, 16), ...  Slice k is placed at z = k*dz_nm.
#     """
#     zs = np.array(sorted(rows))
#     bins = np.floor(zs * dz_nm / bin_nm + 1e-9).astype(int)
#     out = []
#     for b in np.unique(bins):
#         ks = zs[bins == b]
#         v = np.concatenate([rows[k]["vals"] for k in ks])
#         v = v[np.isfinite(v)]
#         if len(v) == 0:
#             continue
#         out.append(dict(bin=b, center=(b + 0.5) * bin_nm, slices=ks, n=len(v),
#                         mean=v.mean(), median=np.median(v),
#                         std=v.std(ddof=1) if len(v) > 1 else np.nan,  stderr=v.std(ddof=1)/np.sqrt(len(v)) if len(v) > 1 else np.nan  ))
#     return out


# def plot_fwhm_vs_z_bin_onecontour(Mz3, rows, tracked, z_ref, dz_nm, pixel_size=1.0, unit="nm", bin_nm=8.0, model = 'gauss', plot_spline = False):
#     zs = np.array(sorted(rows))
#     binned = bin_fwhm_by_z(rows, dz_nm, bin_nm)
#     xc   = np.array([d["center"] for d in binned])
#     mean = np.array([d["mean"] for d in binned])
#     med  = np.array([d["median"] for d in binned])
#     std  = np.array([d["std"] for d in binned])
#     stderr  = np.array([d["stderr"] for d in binned])
#     n    = np.array([d["n"] for d in binned])


#     if plot_spline == True: 
#         fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(13, 4.8),
#                                     gridspec_kw={"width_ratios": [1, 1.2]})

#         # left: check that the tracking followed the same wall
#         ny, nx = Mz3[z_ref].shape
#         extent = (-0.5 * pixel_size, (nx - 0.5) * pixel_size,
#                 (ny - 0.5) * pixel_size, -0.5 * pixel_size)
#         ax0.imshow(Mz3[z_ref], extent=extent, origin="upper", cmap="gray")
#         cmap = plt.get_cmap("viridis")
#         for k in zs:
#             xy = tracked[k]["xy"]
#             ax0.plot(xy[:, 0], xy[:, 1], "-", lw=1.2,
#                     color=cmap((k - zs[0]) / max(1, zs[-1] - zs[0])))
#         sm = plt.cm.ScalarMappable(plt.Normalize(zs[0] * dz_nm, zs[-1] * dz_nm), "viridis")
#         fig.colorbar(sm, ax=ax0, fraction=0.046, pad=0.04, label=f"z ({unit})")
#         ax0.set_title(f"tracked wall (background: Mz at z={z_ref})")
#         ax0.set_xlabel(f"x ({unit})"); ax0.set_ylabel(f"y ({unit})")

#         # right: binned FWHM vs z (mean and median offset slightly so error bars don't overlap)
#         off = 0.06 * bin_nm
#         ax1.errorbar(xc, mean, yerr=stderr, fmt="bs", capsize=4, ms=6, label="mean ± stderr")
#         # ax1.errorbar(xc + off, med, yerr=stderr, fmt="o", c="orange", mfc="none", capsize=4, ms=6,
#         #              label="median ± stderr")
#         # ax1.axvline(z_ref * dz_nm, c="gray", ls=":", label="reference slice")
#         # for x, nn in zip(xc, n):
#         #     ax1.annotate(f"n={nn}", (x, 0.02), xycoords=("data", "axes fraction"),
#         #                  ha="center", va="bottom", fontsize=8, color="gray")
#         ax1.set_xticks(xc)
#         ax1.set_xticklabels([f"{x - bin_nm/2:.0f}-{x + bin_nm/2:.0f}" for x in xc])
#         ax1.set_xlabel(f"z bin ({unit})")
#         ax1.set_ylabel(f"FWHM of {model} fit ({unit})")
#         ax1.set_title(f"FWHM vs. z, {bin_nm:g} nm bins")
#         ax1.legend(fontsize=8)
#         fig.tight_layout()


#     else: 

#         fig, (ax1) = plt.subplots(1, 1, figsize=(7, 4.8))

#         # left: check that the tracking followed the same wall
#         ny, nx = Mz3[z_ref].shape
        
#         # right: binned FWHM vs z (mean and median offset slightly so error bars don't overlap)
#         off = 0.06 * bin_nm
#         ax1.errorbar(xc, mean, yerr=stderr, fmt="bs", capsize=4, ms=6, label="mean ± stderr")
#         # ax1.errorbar(xc + off, med, yerr=stderr, fmt="o", c="orange", mfc="none", capsize=4, ms=6,
#         #              label="median ± stderr")
#         # ax1.axvline(z_ref * dz_nm, c="gray", ls=":", label="reference slice")
#         # for x, nn in zip(xc, n):
#         #     ax1.annotate(f"n={nn}", (x, 0.02), xycoords=("data", "axes fraction"),
#         #                  ha="center", va="bottom", fontsize=8, color="gray")
#         ax1.set_xticks(xc)
#         ax1.set_xticklabels([f"{x - bin_nm/2:.0f}-{x + bin_nm/2:.0f}" for x in xc])
#         ax1.set_xlabel(f"z bin ({unit})")
#         ax1.set_ylabel(f"FWHM ({unit})")
#         ax1.set_title(f"FWHM of {model} fit vs. z, {bin_nm:g} nm bins \n one contour")
#         ax1.legend(fontsize=8)
#         fig.tight_layout()

#     return fig, binned

In [ ]:
contour_fn = lambda Mz: contour_curvature(Mz, level=0.0, smoothing=sigma_save,
                                          n_eval=100, pixel_size=voxel_nm)

rows, tracked = fwhm_vs_z(volume['Mz'], frobenius_2d, contour_fn,
                          z_ref=4, ref_result=results[6],
                          pixel_size=voxel_nm, half_width=half * voxel_nm,
                          tol=3* voxel_nm, model = 'gauss')

fig, axes = plot_fwhm_vs_z_nobin(volume['Mz'], rows, tracked, z_ref=4,
                           dz_nm=spacing[0], pixel_size=voxel_nm, model = 'gauss')

In [ ]:
fig, binned = plot_fwhm_vs_z_bin_onecontour(volume['Mz'], rows, tracked, z_ref=4,
                                   dz_nm=spacing[0], pixel_size=voxel_nm, bin_nm=voxel_nm, model = 'gauss', plot_spline = False)
for d in binned:
    print(f"{d['center']-4:.0f}-{d['center']+4:.0f} nm: slices {d['slices']}, "
          f"mean {d['mean']:.1f}, median {d['median']:.1f}, std {d['std']:.1f}, n={d['n']}")

In [ ]:
# Now look at the whole sample

# def fwhm_slice(grad2d, contours, pixel_size, half_width, min_good=20, model = 'gauss'):
#     """Fit every contour in one slice and pool the good FWHM values."""
#     vals, cmeans, used = [], [], []
#     n_pts = 0
#     for ci, r in enumerate(contours):
#         fw = fwhm_along_contour(grad2d, r, pixel_size, half_width, model = 'gauss')
#         good = fw["success"] & fw["inside"] & (fw["fwhm"] < fw["half_width"])
#         n_pts += len(good)
#         if good.sum() < min_good:          # skip tiny / mostly-rejected contours
#             continue
#         v = fw["fwhm"][good]
#         vals.append(v)
#         cmeans.append(v.mean())
#         used.append(dict(idx=ci, xy=r["xy"], mean=v.mean(), n_good=int(good.sum())))

#     v = np.concatenate(vals) if vals else np.array([])
#     cm = np.array(cmeans)
#     sd = lambda a: np.std(a, ddof=1) if len(a) > 1 else np.nan
#     return dict(
#         n_good=len(v), n_pts=n_pts, n_contours=len(used),
#         mean=v.mean() if len(v) else np.nan,
#         median=np.median(v) if len(v) else np.nan,
#         std=sd(v),
#         stderr=sd(v) / np.sqrt(len(v)) if len(v) > 1 else np.nan,
#         p16=np.percentile(v, 16) if len(v) else np.nan,
#         p84=np.percentile(v, 84) if len(v) else np.nan,
#         vals=v, contour_means=cm, contours=used)


# def fwhm_vs_z_slices(Mz3, grad3, contour_fn, pixel_size, half_width,
#                      z_indices=None, min_good=20, model = 'gauss'):
#     """Whole-slice FWHM for each z index. No tracking: every contour in the slice is used."""
#     z_indices = range(len(Mz3)) if z_indices is None else z_indices
#     rows = {}
#     for k in z_indices:
#         contours = contour_fn(Mz3[k])
#         rows[k] = fwhm_slice(grad3[k], contours, pixel_size, half_width, min_good, model = 'gauss')
#         r = rows[k]
#         print(f"z={k}: {r['n_contours']} contours, {r['n_good']}/{r['n_pts']} points used, "
#               f"mean {r['mean']:.1f}, median {r['median']:.1f}")
#     return rows


# def bin_slice_fwhm_by_z(rows, dz_nm, bin_nm=8.0):
#     zs = np.array(sorted(rows))
#     bins = zs if bin_nm is None else np.floor(zs * dz_nm / bin_nm + 1e-9).astype(int)
#     sd = lambda a: np.std(a, ddof=1) if len(a) > 1 else np.nan
#     out = []
#     for b in np.unique(bins):
#         ks = zs[bins == b]
#         v = np.concatenate([rows[k]["vals"] for k in ks])
#         v = v[np.isfinite(v)]
#         cm = np.concatenate([rows[k]["contour_means"] for k in ks])
#         if len(v) == 0:
#             continue
#         out.append(dict(
#             center=ks[0] * dz_nm if bin_nm is None else (b + 0.5) * bin_nm,
#             slices=ks, n=len(v), n_contours=len(cm),
#             mean=v.mean(), median=np.median(v), std=sd(v),
#             stderr=sd(v) / np.sqrt(len(v)) if len(v) > 1 else np.nan,
#             stderr_contour=sd(cm) / np.sqrt(len(cm)) if len(cm) > 1 else np.nan))
#     return out


# def plot_fwhm_vs_z_slices(Mz3, rows, z_ref, dz_nm, pixel_size=1.0, unit="nm",
#                           bin_nm=8.0, err="stderr", show_median=False, cmap="viridis", model = 'gauss'):
#     """err: 'std' | 'stderr' (std/sqrt(points)) | 'stderr_contour' (std of contour means/sqrt(contours))"""
#     binned = bin_slice_fwhm_by_z(rows, dz_nm, bin_nm)
#     xc = np.array([d["center"] for d in binned])
#     mean = np.array([d["mean"] for d in binned])
#     med = np.array([d["median"] for d in binned])
#     e = np.array([d[err] for d in binned])

#     fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(13, 4.8),
#                                    gridspec_kw={"width_ratios": [1, 1.2]})

#     # left: every contour in the reference slice, colored by its mean FWHM
#     ny, nx = Mz3[z_ref].shape
#     extent = (-0.5 * pixel_size, (nx - 0.5) * pixel_size,
#               (ny - 0.5) * pixel_size, -0.5 * pixel_size)
#     ax0.imshow(Mz3[z_ref], extent=extent, origin="upper", cmap="gray")
#     cs = rows[z_ref]["contours"]
#     allm = [c["mean"] for c in cs]
#     if allm:
#         vmin, vmax = min(allm), max(allm)
#         for c in cs:
#             sc = ax0.scatter(c["xy"][:, 0], c["xy"][:, 1], c=np.full(len(c["xy"]), c["mean"]),
#                              vmin=vmin, vmax=vmax, s=3, cmap=cmap)
#         fig.colorbar(sc, ax=ax0, fraction=0.046, pad=0.04, label=f"contour mean FWHM ({unit})")
#     ax0.set_title(f"contours used at z={z_ref} ({rows[z_ref]['n_contours']})")
#     ax0.set_xlabel(f"x ({unit})"); ax0.set_ylabel(f"y ({unit})")

#     # right: whole-slice FWHM vs z
#     ax1.errorbar(xc, mean, yerr=e, fmt="bs-" if bin_nm is None else "bs",
#                  capsize=4, ms=6, label=f"mean ± {err}")
#     if show_median:
#         ax1.errorbar(xc, med, yerr=e, fmt="o", c="orange", mfc="none", capsize=4, ms=6,
#                      label=f"median ± {err}")
#     if bin_nm is not None:
#         ax1.set_xticks(xc)
#         ax1.set_xticklabels([f"{x - bin_nm/2:.0f}-{x + bin_nm/2:.0f}" for x in xc])
#         ax1.set_xlabel(f"z bin ({unit})")
#         ax1.set_title(f"FWHM of {model}  vs. z, all contours, {bin_nm:g} nm bins")
#     else:
#         ax1.set_xlabel(f"z ({unit})")
#         ax1.set_title(f"FWHM of {model} fit vs. z, all contours")
#     ax1.set_ylabel(f"FWHM ({unit})")
#     ax1.legend(fontsize=8)
#     fig.tight_layout()
#     return fig, (ax0, ax1), binned

In [ ]:
contour_fn = lambda Mz: contour_curvature(Mz, level=0.0, smoothing=sigma_save,
                                          n_eval=400, pixel_size=voxel_nm)

rows = fwhm_vs_z_slices(volume['Mz'], frobenius_2d, contour_fn,
                        voxel_nm, half * voxel_nm, z_indices=range(0, len(volume['Mz'])), model = 'gauss')





In [ ]:
fig, axes, binned = plot_fwhm_vs_z_slices(volume['Mz'], rows, z_ref=4,
                                          dz_nm=spacing[0], pixel_size=voxel_nm,
                                          bin_nm=voxel_nm, err="stderr", model = 'gauss')

In [ ]:
stop

### Calculate the polar and aziumuthal angle directly 

In [ ]:
# seperate polar and azimuthal angle along the wall

# def sample_angles(Mx, My, Mz, xy, normal, pixel_size, half_width, n_samples=161, order=3):
#     """theta, phi, |m| along every normal; arrays have shape (N, n_samples)."""
#     s, mx, _ = compute_profiles(Mx, xy, normal, pixel_size, half_width, n_samples, order)
#     _, my, _ = compute_profiles(My, xy, normal, pixel_size, half_width, n_samples, order)
#     _, mz, _ = compute_profiles(Mz, xy, normal, pixel_size, half_width, n_samples, order)
#     norm = np.sqrt(mx**2 + my**2 + mz**2)
#     theta = np.arccos(np.clip(mz / norm, -1, 1))
#     phi = np.arctan2(my/ norm, mx/ norm)
#     return s, theta, phi, norm


# # ---------------- theta: 2*arctan(exp(+-(s-mu)/D)) ----------------
# def theta_model(s, mu, D, sgn):
#     return 2 * np.arctan(np.exp(np.clip(sgn * (s - mu) / D, -100, 100)))


# def fit_theta(s, theta):
#     out = dict(success=False, mu=np.nan, D=np.nan, mu_err=np.nan, D_err=np.nan,
#                sgn=np.nan, rms=np.nan)
#     ds = s[1] - s[0]
#     sgn = 1.0 if theta[-1] > theta[0] else -1.0      # theta goes 0->pi or pi->0 along +s
#     mu0 = s[np.argmin(np.abs(theta - np.pi / 2))]
#     f = lambda s, mu, D: theta_model(s, mu, D, sgn)
#     try:
#         popt, pcov = curve_fit(f, s, theta, p0=[mu0, 4 * ds],
#                                bounds=([s[0], ds / 2], [s[-1], s[-1] - s[0]]))
#     except (RuntimeError, ValueError):
#         return out
#     perr = np.sqrt(np.diag(pcov))
#     out.update(success=True, mu=popt[0], D=popt[1], mu_err=perr[0], D_err=perr[1],
#                sgn=sgn, rms=np.sqrt(np.mean((theta - f(s, *popt)) ** 2)), f=f)
#     return out


# ---------------- phi: tanh step vs constant ----------------
def phi_step(s, phic, dphi, sphi, wphi):
    return phic + 0.5 * dphi * np.tanh((s - sphi) / wphi)


In [ ]:
def plot_normal_cross_section(field, gradient, results, contour_idx, point_idx,
                              pixel_size, half_width, n_samples=81, order=3,
                              cmap="magma", fit=True, model="gauss", unit="nm",
                              detailed_label=False):
    """
    Panel 1: gradient map, full spline, and the selected normal (extent = what's profiled).
    Panel 2: profile of `gradient` along that normal, with a `model` fit and its residual.
    model: any key of MODELS ("gauss", "sech", "sech2").
    """
    r = results[contour_idx]
    xy, normal = r["xy"], r["normal"]
    i = point_idx % len(xy)                      # allows negative indices

    # profile at just the selected point
    s, prof, XY = compute_profiles(gradient, xy[i:i+1], normal[i:i+1],
                                   pixel_size, half_width, n_samples, order)
    prof, XY = prof[0], XY[0]

    f = fit_peak(s, prof, model) if fit else None
    resid = np.full_like(prof, np.nan)           # defined even if the fit fails or is off

    nrows, ncols = field.shape
    extent = (-0.5 * pixel_size, (ncols - 0.5) * pixel_size,
              (nrows - 0.5) * pixel_size, -0.5 * pixel_size)   # origin="upper"

    fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11, 4.5),
                                   gridspec_kw={"width_ratios": [1.1, 1]})

    # panel 1: gradient map + spline + selected normal
    im = ax0.imshow(gradient, extent=extent, origin="upper", cmap=cmap)
    ax0.plot(xy[:, 0], xy[:, 1], "w-", lw=1.5, label="spline")
    ax0.plot(XY[:, 0], XY[:, 1], "r-", lw=2, label="profile")
    ax0.plot(xy[i, 0], xy[i, 1], "ro", ms=6, mec="w")
    ax0.set_xlabel(f"x ({unit})"); ax0.set_ylabel(f"y ({unit})")
    ax0.set_title(f"contour {contour_idx}, point {i}")
    ax0.legend(loc="upper right", fontsize=8)
    fig.colorbar(im, ax=ax0, fraction=0.046, pad=0.04)

    # panel 2: profile + fit
    ax1.plot(s, prof, "k.-", label="data")
    ax1.axvline(0, color="r", ls="--", lw=1, label="DW center")

    if f is not None and f["success"]:
        sf = np.linspace(s[0], s[-1], 500)
        label = f"{LABEL[model]} fit\nFWHM = {f['fwhm']:.3g} {unit}"
        if detailed_label:
            wname = "σ" if model == "gauss" else "w"
            label += (f"\nmean = {f['mu']:.3g} ± {f['mu_err']:.2g}"
                      f"\n{wname} = {f['w']:.3g} ± {f['w_err']:.2g}"
                      f"\nA = {f['A']:.3g}, offset = {f['offset']:.3g}")
        ax1.plot(sf, f["f"](sf, *f["popt"]), "b-", lw=2, label=label)

        # FWHM marker at half maximum of the fitted curve (valid for all three shapes)
        half = f["offset"] + f["A"] / 2
        ax1.hlines(half, f["mu"] - f["fwhm"] / 2, f["mu"] + f["fwhm"] / 2,
                   color="c", lw=2)

        # residual at the sampled points
        resid = prof - f["f"](s, *f["popt"])
        ax1.axhline(0, color="gray", lw=1)
        ax1.plot(s, resid, "k.-", lw=0.8, alpha=0.5, label="residual")
    elif fit:
        ax1.text(0.05, 0.9, "fit failed", transform=ax1.transAxes, color="r")

    ax1.set_xlabel(f"distance along normal ({unit})")
    ax1.set_ylabel("gradient")
    ax1.set_title("cross-section along normal")
    ax1.legend(loc="upper right", fontsize=10)

    fig.tight_layout()
    return fig, (ax0, ax1), s, prof, f, resid


In [ ]:

def psi_window_profile(Mx, My, Mz, gradient, r, i, pixel_size, half_width,
                       n_samples=161, order=3, sin_min=0.2, model="gauss",
                       window=1, window_from="fit"):
    """psi along the normal at point i of contour r, plus the FWHM window from the gradient."""
    xy, normal = r["xy"], r["normal"]
    sl = slice(i, i + 1)
    s, th, ph, _ = sample_angles(Mx, My, Mz, xy[sl], normal[sl], pixel_size,
                                 half_width, n_samples, order)
    th, ph = th[0], ph[0]
    g = compute_profiles(gradient, xy[sl], normal[sl], pixel_size,
                         half_width, n_samples, order)[1][0]

    alpha = np.arctan2(normal[i, 1], normal[i, 0])
    psi = wrap(ph - alpha)

    fit = fit_peak(s, g, model)
    if window_from == "fit" and fit["success"]:
        mu, fwhm = fit["mu"], fit["fwhm"]
    else:
        fwhm, cl, cr = fwhm_from_curve(s, g, np.nanmin(g))
        mu = 0.5 * (cl + cr)
    ok = np.isfinite(mu) and np.isfinite(fwhm)
    lo, hi = (mu - window * fwhm, mu + window * fwhm) if ok else (np.nan, np.nan)
    inwin = (s >= lo) & (s <= hi) if ok else np.zeros_like(s, bool)
    good = inwin & (np.sin(th) > sin_min)
    return dict(s=s, psi=psi, theta=th, good=good, inwin=inwin, mu=mu, fwhm=fwhm,
                lo=lo, hi=hi, alpha=alpha, ok=ok)



# def _break_jumps(x, y_deg, jump=180):
#     """Insert NaN where the wrapped angle jumps, so lines don't cross the plot at +-180."""
#     x, y = np.asarray(x, float), np.asarray(y_deg, float)
#     j = np.where(np.abs(np.diff(y)) > jump)[0] + 1
#     return np.insert(x, j, np.nan), np.insert(y, j, np.nan)



In [ ]:
# def orient_normals(Mz, result, pixel_size, probe=4):
#     """Flip the normal so +s points toward Mz>0. Safe to call twice."""
#     xy, n = result["xy"], result["normal"]
#     d = probe * pixel_size
#     f = lambda p: map_coordinates(Mz, [p[:, 1] / pixel_size, p[:, 0] / pixel_size],
#                                   order=1, mode="nearest")
#     flip = 1.0 if np.nanmedian(f(xy + d * n) - f(xy - d * n)) >= 0 else -1.0
#     result["normal"] = flip * n
#     result["flip"] = result.get("flip", 1.0) * flip
#     result["kappa_n"] = result["flip"] * result["kappa"]   # >0: wall bends toward +s


# SQ2 = 1 / np.sqrt(2)
# wrap = lambda a: np.angle(np.exp(1j * a))


# def psi_gradient(p, smooth=0, min_run=3):
#     """
#     dpsi/ds (rad per unit length) on the 'good' points of one profile.
#     psi is unwrapped inside each contiguous run of good points before differentiating,
#     so a jump across +-180 deg is not read as a huge rotation. NaN elsewhere.
#     smooth : odd Savitzky-Golay window (in samples) applied to the unwrapped psi; 0 = none
#     """
#     s, psi, good = p["s"], p["psi"], p["good"]
#     d = np.full_like(s, np.nan)
#     idx = np.where(good)[0]
#     if len(idx) == 0:
#         return d
#     for run in np.split(idx, np.where(np.diff(idx) > 1)[0] + 1):
#         if len(run) < min_run:
#             continue
#         u = np.unwrap(psi[run])
#         if smooth and smooth % 2 == 1 and len(run) > smooth:
#             u = savgol_filter(u, smooth, 2)
#         d[run] = np.gradient(u, s[run])
#     return d


# def wall_profile(Mx, My, Mz, gradient, r, i, pixel_size, half_width,
#                  n_samples=161, order=3, sin_min=0.2, model="gauss",
#                  window=0.5, window_from="fit"):
#     """
#     Single source of truth for one normal profile (used by both plots).
#     Enforces: +s points toward +Mz, so psi = +-180 is positive Neel (+Mz -> -Mz).
#     """
#     orient_normals(Mz, r, pixel_size)              # contour-wide orientation
#     xy = r["xy"]
#     sl = slice(i, i + 1)
#     n_i = r["normal"][i:i + 1].copy()

#     def sample(n_loc):
#         s, th, ph, _ = sample_angles(Mx, My, Mz, xy[sl], n_loc, pixel_size,
#                                      half_width, n_samples, order)
#         _, g, XY = compute_profiles(gradient, xy[sl], n_loc, pixel_size,
#                                     half_width, n_samples, order)
#         return s, th[0], ph[0], g[0], XY[0]

#     s, th, ph, g, XY = sample(n_i)

#     # local enforcement: Mz must be larger on the +s side than on the -s side
#     far = np.abs(s) > 0.5 * np.abs(s).max()
#     mz = np.cos(th)
#     flipped = bool(np.nanmean(mz[far & (s > 0)]) < np.nanmean(mz[far & (s < 0)]))
#     if flipped:
#         n_i = -n_i
#         s, th, ph, g, XY = sample(n_i)

#     alpha = np.arctan2(n_i[0, 1], n_i[0, 0])
#     psi = wrap(ph - alpha)

#     fit = fit_peak(s, g, model)
#     if window_from == "fit" and fit["success"]:
#         mu, fwhm = fit["mu"], fit["fwhm"]
#     else:
#         fwhm, cl, cr = fwhm_from_curve(s, g, np.nanmin(g))
#         mu = 0.5 * (cl + cr)
#     ok = np.isfinite(mu) and np.isfinite(fwhm)
#     lo, hi = (mu - window * fwhm, mu + window * fwhm) if ok else (np.nan, np.nan)
#     inwin = (s >= lo) & (s <= hi) if ok else np.zeros_like(s, bool)
#     good = inwin & (np.sin(th) > sin_min)

#     return dict(s=s, psi=psi, phi=ph, theta=th, grad=g, XY=XY, fit=fit, n_i=n_i,
#                 good=good, inwin=inwin, mu=mu, fwhm=fwhm, lo=lo, hi=hi,
#                 alpha=alpha, ok=ok, flipped=flipped)


# psi_window_profile = wall_profile      # old name still works

In [ ]:
# def psi_vs_z(Mx3, My3, Mz3, grad3, contour_fn, z_ref, ref_result, ref_pt,
#              pixel_size, half_width, tol, z_indices=None, min_overlap=0.3, **kw):
#     """kw is passed to wall_profile (n_samples, order, sin_min, model, window, window_from)."""
#     nz = len(Mz3)
#     allowed = set(range(nz)) if z_indices is None else set(z_indices)
#     out = {}

#     def run(k, r, i):
#         p = wall_profile(Mx3[k], My3[k], Mz3[k], grad3[k], r, i,
#                          pixel_size, half_width, **kw)
#         p.update(xy=r["xy"].copy(), pt=i, pos=r["xy"][i].copy())
#         out[k] = p

#     i0 = ref_pt % len(ref_result["xy"])
#     run(z_ref, ref_result, i0)

#     for direction in (+1, -1):
#         ref_xy, ref_pos = out[z_ref]["xy"], out[z_ref]["pos"]
#         k = z_ref + direction
#         while 0 <= k < nz and k in allowed:
#             contours = contour_fn(Mz3[k])
#             j, score = find_match(ref_xy, contours, tol, min_overlap)
#             if j is None:
#                 print(f"lost the wall at z={k} (best overlap {score:.2f})")
#                 break
#             r = contours[j]
#             d, i = cKDTree(r["xy"]).query(ref_pos)
#             if d > tol:
#                 print(f"z={k}: nearest point is {d:.1f} away (> tol), stopping")
#                 break
#             run(k, r, int(i))
#             ref_xy, ref_pos = out[k]["xy"], out[k]["pos"]
#             k += direction

#     nf = [k for k in sorted(out) if out[k]["flipped"]]
#     if nf:
#         print(f"note: normal flipped locally (contour-wide orientation disagreed) at z = {nf}")
#     return out

In [ ]:
# def plot_angle_decomposition(Mx, My, Mz, gradient, results, contour_idx, point_idx,
#                              pixel_size, half_width, n_samples=161, order=3,
#                              cmap="magma", sin_min=0.2, unit="nm",
#                              jac=None, k=None, model='gauss'):
#     r = results[contour_idx]
#     xy = r["xy"]
#     i = point_idx % len(xy)
#     sl = slice(i, i + 1)

#     P = wall_profile(Mx, My, Mz, gradient, r, i, pixel_size, half_width,
#                      n_samples=n_samples, order=order, sin_min=sin_min, model=model)
#     s, th, ph, g, XY = P["s"], P["theta"], P["phi"], P["grad"], P["XY"]
#     psi, alpha, f, n_i = P["psi"], P["alpha"], P["fit"], P["n_i"]
#     ok = np.sin(th) > sin_min

#     ft = fit_theta(s, th)

#     nrows, ncols = Mz.shape
#     extent = (-0.5 * pixel_size, (ncols - 0.5) * pixel_size,
#               (nrows - 0.5) * pixel_size, -0.5 * pixel_size)

#     fig = plt.figure(figsize=(18, 8))
#     gs = GridSpec(3, 2, figure=fig, width_ratios=[1.1, 1],
#                   height_ratios=[1.3, 1.3, 1.6], hspace=0.1)
#     ax0 = fig.add_subplot(gs[:, 0])
#     ax1 = fig.add_subplot(gs[0, 1])
#     ax2 = fig.add_subplot(gs[1, 1], sharex=ax1)
#     ax3 = fig.add_subplot(gs[2, 1], sharex=ax1)

#     # map
#     im = ax0.imshow(gradient, extent=extent, origin="upper", cmap=cmap)
#     ax0.plot(xy[:, 0], xy[:, 1], "w-", lw=1.5, label="spline")
#     ax0.plot(XY[:, 0], XY[:, 1], "r-", lw=2, label="profile")
#     ax0.plot(xy[i, 0], xy[i, 1], "ro", ms=6, mec="w")
#     ax0.set_xlabel(f"x ({unit})"); ax0.set_ylabel(f"y ({unit})")
#     ax0.set_title(f"contour {contour_idx}, point {i}")
#     ax0.legend(loc="upper right", fontsize=8)
#     fig.colorbar(im, ax=ax0, fraction=0.046, pad=0.04)

#     sf = np.linspace(s[0], s[-1], 800)

#     # theta
#     ax1.plot(s, np.degrees(th), "k.", label="data")
#     if ft["success"]:
#         ax1.plot(sf, np.degrees(ft["f"](sf, ft["mu"], ft["D"])), "m-", lw=2,
#                  label=f"Δ = {ft['D']:.3g} ± {ft['D_err']:.2g} {unit}")
#     ax1.set_ylabel("Polar Angle θ (deg)"); ax1.legend(fontsize=8)
#     ax1.set_title("angle decomposition along normal")

#     ax1.axvline(f["mu"] - f["fwhm"] / 2, color="tab:green", lw=2, linestyle='--')
#     ax1.axvline(f["mu"] + f["fwhm"] / 2, color="tab:green", lw=2, linestyle='--')
#     ax1.axvline(0 - ft['D'] / 2, color="magenta", lw=2, linestyle='--')
#     ax1.axvline(0 + ft['D'] / 2, color="magenta", lw=2, linestyle='--')

#     # phi (lab frame) and psi (wall frame)
#     ax2.plot(s, np.degrees(ph), ".", color="0.65", ms=4, label="φ (lab frame)")
#     ax2.plot(s[ok], np.degrees(psi[ok]), ".", color="k", ms=5,
#              label=f"ψ = φ − α  (α = {np.degrees(alpha):.0f}°)")
#     for y in (-90, 90):
#         ax2.axhline(y, color="magenta", lw=1, alpha=0.7,
#                     label="Bloch (ψ = ±90°)" if y == -90 else None)
#     for y in (-180, 0, 180):
#         ax2.axhline(y, color="cyan", lw=1, alpha=0.7,
#                     label="Néel (ψ = 0, ±180°)" if y == -180 else None)

#     ax2.axvline(f["mu"] - f["fwhm"] / 2, color="tab:green", lw=2, linestyle='--')
#     ax2.axvline(f["mu"] + f["fwhm"] / 2, color="tab:green", lw=2, linestyle='--')
#     ax2.axvline(0 - ft['D'] / 2, color="magenta", lw=2, linestyle='--')
#     ax2.axvline(0 + ft['D'] / 2, color="magenta", lw=2, linestyle='--')

#     ax2.set_ylim(-195, 195)
#     ax2.set_yticks([-180, -90, 0, 90, 180])
#     ax2.set_ylabel("Azimuthal Angle ψ (deg)")
#     ax2.legend(fontsize=7, ncol=2, loc="upper right")

#     # gradient
#     ax3.plot(s, g, "k.", label="Gradient profile")

#     if jac is not None and k is not None:
#         gxy = np.sqrt(sum(jac[f"dM{c}_d{d}"][k] ** 2 for c in "xyz" for d in "xy"))
#         gz = np.sqrt(sum(jac[f"dM{c}_dz"][k] ** 2 for c in "xyz"))
#         pp = {}
#         for name, arr, st, lab in (("xy", gxy, "r-", "in-plane (x,y) part"),
#                                    ("z", gz, "g--", "through-thickness (z) part")):
#             pp[name] = compute_profiles(arr, xy[sl], n_i, pixel_size,
#                                         half_width, n_samples, order)[1][0]
#             ax3.plot(s, pp[name], st, lw=1.5, label=lab)
#         ax3.plot(s, np.hypot(pp["xy"], pp["z"]), "c:", lw=2,
#                  label="√(xy² + z²)  (should match Grad_slice)")

#     if f is not None and f["success"]:
#         sf = np.linspace(s[0], s[-1], 500)
#         label = f"{LABEL[model]} fit\nFWHM = {f['fwhm']:.3g} {unit}"
#         wname = "σ" if model == "gauss" else "w"
#         label += (f"\nmean = {f['mu']:.3g} ± {f['mu_err']:.2g}")
#                   #f"\n{wname} = {f['w']:.0g} ± {f['w_err']:.1g}")
#         ax3.plot(sf, f["f"](sf, *f["popt"]), "b-", lw=2, label=label)

#         half = f["offset"] + f["A"] / 2
#         ax3.hlines(half, f["mu"] - f["fwhm"] / 2, f["mu"] + f["fwhm"] / 2,
#                    color="tab:green", lw=2)
#         ax3.axvline(f["mu"] - f["fwhm"] / 2, color="tab:green", lw=2, linestyle='--')
#         ax3.axvline(f["mu"] + f["fwhm"] / 2, color="tab:green", lw=2, linestyle='--')

#         resid = g - f["f"](s, *f["popt"])
#         ax3.axhline(0, color="gray", lw=1)
#         ax3.plot(s, resid, "k.-", lw=0.8, alpha=0.5, label="residual")

#     ax3.axvline(0 - ft['D'] / 2, color="magenta", lw=2, linestyle='--')
#     ax3.axvline(0 + ft['D'] / 2, color="magenta", lw=2, linestyle='--')

#     ax3.set_ylabel("|∇m|"); ax3.legend(fontsize=8)
#     ax3.set_xlabel(f"distance along normal ({unit})")

#     for a in (ax1, ax2, ax3):
#         a.axvline(0, color="r", ls="--", lw=1)
#     plt.setp(ax1.get_xticklabels(), visible=False)
#     plt.setp(ax2.get_xticklabels(), visible=False)

#     return fig, (ax0, ax1, ax2, ax3), dict(s=s, theta=th, phi=ph, psi=psi, alpha=alpha,
#                                            grad=g, ft=ft, profile=P)

In [ ]:
# def _wrap_to(y_deg, lo):
#     return (np.asarray(y_deg, float) - lo) % 360 + lo


# def plot_psi_vs_z(Mz3, profiles, z_ref, dz_nm, pixel_size, unit="nm", cmap="viridis",
#                   center=False, ylim = (-30, 195), sin_min=0.2, window=0.5, show_points=True, smooth=0,
#                   wrap_lo=None):
#     zs = np.array(sorted(profiles))
#     zn = zs * dz_nm
#     norm = Normalize(zn.min(), zn.max() if zn.max() > zn.min() else zn.min() + 1)
#     cm = plt.get_cmap(cmap)
#     P0 = profiles[z_ref]                           # reference slice for the FWHM lines

#     fig = plt.figure(figsize=(14, 7))
#     gs = GridSpec(2, 2, figure=fig, width_ratios=[1.1, 1], hspace=0.08)
#     ax0 = fig.add_subplot(gs[:, 0])
#     ax1 = fig.add_subplot(gs[0, 1])
#     ax2 = fig.add_subplot(gs[1, 1], sharex=ax1)

#     ny, nx = Mz3[z_ref].shape
#     extent = (-0.5 * pixel_size, (nx - 0.5) * pixel_size,
#               (ny - 0.5) * pixel_size, -0.5 * pixel_size)
#     ax0.imshow(Mz3[z_ref], extent=extent, origin="upper", cmap="gray")
#     for k, z in zip(zs, zn):
#         p = profiles[k]
#         ax0.plot(p["xy"][:, 0], p["xy"][:, 1], "-", lw=1, color=cm(norm(z)))
#         if show_points:
#             ax0.plot(*p["pos"], "o", ms=5, color=cm(norm(z)), mec="w", mew=0.5)
#     ax0.set_xlabel(f"x ({unit})"); ax0.set_ylabel(f"y ({unit})")

#     seen = set()
#     for kk in range(-4, 5):
#         y = 90 * kk
#         if not -195 <= y <= 195:
#             continue
#         if kk % 2:
#             col, lab = "magenta", "Bloch (ψ = ±90°)"
#         elif kk % 4 == 0:
#             col, lab = "tab:blue", "Néel − (−Mz → +Mz), ψ = 0"
#         else:
#             col, lab = "cyan", "Néel + (+Mz → −Mz), ψ = ±180°"
#         ax1.axhline(y, color=col, lw=1, alpha=0.6, label=lab if lab not in seen else None)
#         seen.add(lab)

#     for k, z in zip(zs, zn):
#         p = profiles[k]
#         if not p["good"].any():
#             continue
#         x = p["s"][p["good"]] - (p["mu"] if center else 0.0)
#         y = np.degrees(p["psi"][p["good"]])
#         if wrap_lo is not None:
#             y = _wrap_to(y, wrap_lo)
#         xb, yb = _break_jumps(x, y)
#         ax1.plot(xb, yb, "-", color=cm(norm(z)), lw=1.6)
#         ax1.plot(x, y, ".", color=cm(norm(z)), ms=4)

#     if not center:
#         ax1.axvline(0, color="r", ls="--", lw=1)


#     ax1.set_ylim(ylim)
#     ax1.set_ylabel("Azimuthal Angle ψ (deg)")
#     #ax1.legend(fontsize=7, ncol=2, loc="lower left") ############################################# Change here

#     c0 = P0["mu"] if center else 0.0               # FWHM edges of the reference slice
#     edges = (P0["mu"] - P0["fwhm"] / 2 - c0, P0["mu"] + P0["fwhm"] / 2 - c0)
#     for x in edges:
#         ax1.axvline(x, color="tab:green", lw=2, linestyle='--')

#     ax2.axhline(0, color="gray", lw=1)
#     for k, z in zip(zs, zn):
#         p = profiles[k]
#         d = np.degrees(psi_gradient(p, smooth=smooth))
#         m = p["inwin"] & np.isfinite(p["mu"])
#         if not np.isfinite(d[m]).any():
#             continue
#         off = p["mu"] if center else 0.0
#         ax2.plot(p["s"][m] - off, d[m], ".-", color=cm(norm(z)), lw=1.2, ms=3)

#     ax2.axvline(0, color="r", ls="--", lw=1,
#                 label="gradient peak (s − μ = 0)" if center else "contour (s = 0)")
#     ax2.set_xlabel(f"position along normal ({unit}) \n  +s → +Mz domain")
#     ax2.set_ylabel(f"dψ/ds (deg/{unit})")
#     # ax2.legend(fontsize=7, loc="upper right")
#     for x in edges:
#         ax2.axvline(x, color="tab:green", lw=2, linestyle='--')

#     sm = plt.cm.ScalarMappable(norm, cm)
#     fig.colorbar(sm, ax=[ax1, ax2], fraction=0.046, pad=0.02, label=f"z ({unit})")
#     return fig, (ax0, ax1, ax2)

In [ ]:
# z_ref, contour_idx, point_idx = 2, 2, 100



# CFG = dict(pixel_size=voxel_nm, half_width=half * voxel_nm,
#            n_samples=161, order=3, sin_min=0.2, model="gauss")

# contour_fn = lambda Mz: contour_curvature(Mz, level=0.0, smoothing=sigma_save,
#                                           n_eval=400, pixel_size=voxel_nm)

# # per-slice arrays, all from z_ref
# Mx_slice, My_slice, Mz_slice = (volume[c][z_ref] for c in ("Mx", "My", "Mz"))
# Grad_slice = frobenius_2d[z_ref]
# results_3 = contour_fn(Mz_slice)


# fig, axes, out = plot_angle_decomposition(
#     Mx_slice, My_slice, Mz_slice, Grad_slice, results_3,
#     contour_idx=contour_idx, point_idx=point_idx, jac=None, k=None, **CFG)

# # 2) through z, same contour, same gradient, same window
# profiles = psi_vs_z(volume['Mx'], volume['My'], volume['Mz'], frobenius_2d, contour_fn,
#                     z_ref=z_ref, ref_result=results_3[contour_idx], ref_pt=point_idx,
#                     tol=3 * voxel_nm, window=1, **CFG)

# fig, axes = plot_psi_vs_z(volume['Mz'], profiles, z_ref=z_ref,
#                           dz_nm=spacing[0], pixel_size=voxel_nm, ylim = (-195, 195), window=2, smooth=0)

In [ ]:
def plot_cos2psi_vs_z(Mz3, profiles, z_ref, dz_nm, pixel_size, unit="nm", cmap="viridis",
                      center=False, ylim=(-0.05, 1.05), sin_min=0.2, window=0.5,
                      show_points=True, smooth=0, wrap_lo=None):
    """
    Top   : cos^2(psi) inside each slice's window, colored by z.
            1 = Neel (either sign), 0 = Bloch.
    Bottom: dpsi/ds, same x axis (unchanged).
    """
    zs = np.array(sorted(profiles))
    zn = zs * dz_nm
    norm = Normalize(zn.min(), zn.max() if zn.max() > zn.min() else zn.min() + 1)
    cm = plt.get_cmap(cmap)
    P0 = profiles[z_ref]                           # reference slice for the FWHM lines

    fig = plt.figure(figsize=(14, 7))
    gs = GridSpec(2, 2, figure=fig, width_ratios=[1.1, 1], hspace=0.08)
    ax0 = fig.add_subplot(gs[:, 0])
    ax1 = fig.add_subplot(gs[0, 1])
    ax2 = fig.add_subplot(gs[1, 1], sharex=ax1)

    ny, nx = Mz3[z_ref].shape
    extent = (-0.5 * pixel_size, (nx - 0.5) * pixel_size,
              (ny - 0.5) * pixel_size, -0.5 * pixel_size)
    ax0.imshow(Mz3[z_ref], extent=extent, origin="upper", cmap="gray")
    for k, z in zip(zs, zn):
        p = profiles[k]
        ax0.plot(p["xy"][:, 0], p["xy"][:, 1], "-", lw=1, color=cm(norm(z)))
        if show_points:
            ax0.plot(*p["pos"], "o", ms=5, color=cm(norm(z)), mec="w", mew=0.5)
    ax0.set_xlabel(f"x ({unit})"); ax0.set_ylabel(f"y ({unit})")

    # reference lines: cos^2(psi) = 1 -> Neel, 0 -> Bloch
    ax1.axhline(1, color="cyan", lw=1, alpha=0.8, label="Néel (cos²ψ = 1)")
    ax1.axhline(0, color="magenta", lw=1, alpha=0.8, label="Bloch (cos²ψ = 0)")

    # cos^2(psi) from the radian psi; periodic, so no wrapping or jump-breaking is needed
    for k, z in zip(zs, zn):
        p = profiles[k]
        if not p["good"].any():
            continue
        x = p["s"][p["good"]] - (p["mu"] if center else 0.0)
        c2 = np.cos(p["psi"][p["good"]]) ** 2
        ax1.plot(x, c2, "-", color=cm(norm(z)), lw=1.6)
        ax1.plot(x, c2, ".", color=cm(norm(z)), ms=4)


    if not center:
        ax1.axvline(0, color="r", ls="--", lw=1)

        
    ax1.set_ylim(ylim)
    ax1.set_ylabel("cos²ψ")
    ax1.legend(fontsize=7, ncol=2, loc="lower left")

    c0 = P0["mu"] if center else 0.0               # FWHM edges of the reference slice
    edges = (P0["mu"] - P0["fwhm"] / 2 - c0, P0["mu"] + P0["fwhm"] / 2 - c0)
    for x in edges:
        ax1.axvline(x, color="tab:green", lw=2, linestyle='--')

    ax2.axhline(0, color="gray", lw=1)
    for k, z in zip(zs, zn):
        p = profiles[k]
        d = np.degrees(psi_gradient(p, smooth=smooth))
        m = p["inwin"] & np.isfinite(p["mu"])
        if not np.isfinite(d[m]).any():
            continue
        off = p["mu"] if center else 0.0
        ax2.plot(p["s"][m] - off, d[m], ".-", color=cm(norm(z)), lw=1.2, ms=3)

    ax2.axvline(0, color="r", ls="--", lw=1,
                label="gradient peak (s − μ = 0)" if center else "contour (s = 0)")
    ax2.set_xlabel(f"position along normal ({unit}) \n  +s → +Mz domain")
    ax2.set_ylabel(f"dψ/ds (deg/{unit})")
    for x in edges:
        ax2.axvline(x, color="tab:green", lw=2, linestyle='--')

    sm = plt.cm.ScalarMappable(norm, cm)
    fig.colorbar(sm, ax=[ax1, ax2], fraction=0.046, pad=0.02, label=f"z ({unit})")
    return fig, (ax0, ax1, ax2)

In [ ]:
fig, axes = plot_cos2psi_vs_z(volume['Mz'], profiles, z_ref=z_ref,
                          dz_nm=spacing[0], pixel_size=voxel_nm, ylim = (-0.1, 1.1), window=1, smooth=0)

In [ ]:
stop

In [ ]:
# from scipy.spatial import cKDTree

# def track_wall(Mz3, contour_fn, z_ref, ref_result, tol, min_overlap=0.3):
#     """Matching contour at every z the wall can be followed to (chained from z_ref)."""
#     tracked = {z_ref: ref_result}
#     for direction in (+1, -1):
#         ref_xy = ref_result["xy"]
#         k = z_ref + direction
#         while 0 <= k < len(Mz3):
#             cs = contour_fn(Mz3[k])
#             j, score = find_match(ref_xy, cs, tol, min_overlap)
#             if j is None:
#                 print(f"lost the wall at z={k} (best overlap {score:.2f})")
#                 break
#             tracked[k] = cs[j]
#             ref_xy = cs[j]["xy"]
#             k += direction
#     return tracked


# def _reduce_profile(P, mode="center"):
#     """One profile -> (neel, bloch, cos^2 psi). mode: 'center' (at gradient peak mu) or 'window'."""
#     g = P["good"]
#     if not P["ok"] or g.sum() < 3:
#         return np.nan, np.nan, np.nan
#     s, psi = P["s"][g], P["psi"][g]
#     neel, bloch = -np.cos(psi), np.sin(psi)
#     if mode == "center":
#         if not (s[0] <= P["mu"] <= s[-1]):
#             return np.nan, np.nan, np.nan
#         n, b = np.interp(P["mu"], s, neel), np.interp(P["mu"], s, bloch)
#         den = n**2 + b**2
#         return n, b, (n**2 / den if den > 0 else np.nan)
#     w = np.sin(P["theta"][g]) ** 2                      # weight by in-plane fraction
#     return (np.average(neel, weights=w), np.average(bloch, weights=w),
#             np.average(np.cos(psi) ** 2, weights=w))


# def psi_along_wall(Mx3, My3, Mz3, grad3, tracked, z_ref, pixel_size, half_width, tol,
#                    dz_nm, step=4, reduce="center", **kw):
#     """
#     kw -> wall_profile (model, window, window_from, sin_min, n_samples, order).
#     Returns arrays of shape (n_z, n_arc): neel, bloch, c2, fwhm, kappa, x, y.
#     The arc coordinate is that of the reference slice; other slices are matched by position.
#     """
#     ref = tracked[z_ref]
#     zs = np.array(sorted(tracked))
#     row = {k: n for n, k in enumerate(zs)}
#     idx0 = np.arange(0, len(ref["xy"]), step)
#     M = len(idx0)
#     A = {n: np.full((len(zs), M), np.nan)
#          for n in ("neel", "bloch", "c2", "fwhm", "kappa", "x", "y")}

#     def fill(k, r, idx, which):
#         for m in which:
#             i = int(idx[m])
#             P = wall_profile(Mx3[k], My3[k], Mz3[k], grad3[k], r, i,
#                              pixel_size, half_width, **kw)
#             A["neel"][row[k], m], A["bloch"][row[k], m], A["c2"][row[k], m] = \
#                 _reduce_profile(P, reduce)
#             A["fwhm"][row[k], m] = P["fwhm"]
#             A["kappa"][row[k], m] = r.get("kappa_n", r["kappa"])[i]   # set by orient_normals
#             A["x"][row[k], m], A["y"][row[k], m] = r["xy"][i]

#     fill(z_ref, ref, idx0, range(M))
#     for direction in (+1, -1):
#         pos, alive = ref["xy"][idx0].copy(), np.ones(M, bool)
#         k = z_ref + direction
#         while k in tracked and alive.any():
#             r = tracked[k]
#             d, idx = cKDTree(r["xy"]).query(pos)
#             alive &= d <= tol                           # drop points that lost their match
#             fill(k, r, idx, np.where(alive)[0])
#             pos = np.where(alive[:, None], r["xy"][idx], pos)   # chain from slice to slice
#             k += direction
#     return dict(zs=zs, z_ref=z_ref, dz_nm=dz_nm, pixel_size=pixel_size,
#                 arc=arc_length(ref["xy"])[idx0], ref_xy=ref["xy"], idx0=idx0, **A)


# QUANT = {"c2":    ("cos²ψ  (1 = Néel, 0 = Bloch)", 0, 1, "cividis"),
#          "neel":  ("−cosψ  (+1 = Néel +, −1 = Néel −)", -1, 1, "RdBu_r"),
#          "bloch": ("sinψ  (signed Bloch component)", -1, 1, "RdBu_r"),
#          "fwhm":  ("FWHM (nm)", None, None, "viridis")}

# def _z(out, z_center_idx):
#     return (out["zs"] - (0 if z_center_idx is None else z_center_idx)) * out["dz_nm"]

In [ ]:
# def arc_length(xy):
#     """Cumulative distance along a contour, in the same units as xy (nm)."""
#     return np.r_[0, np.cumsum(np.hypot(*np.diff(xy, axis=0).T))]

In [ ]:
# import numpy as np
# from scipy.interpolate import splprep, splev
# from skimage.measure import find_contours


# def contour_curvature_evenspacing(field, level=0.0, smoothing=0.0, n_eval=400,
#                       point_spacing=None, min_eval=16, uniform_arclength=True,
#                       min_points=8, pixel_size=8.0):
#     """
#     Extract the domain boundary/boundaries of `field` at `level` and
#     compute signed curvature along each.

#     Parameters
#     ----------
#     field : (H, W) array
#         Scalar field, e.g. your Mz array.
#     level : float, default 0.0
#         Threshold defining the boundary.
#     smoothing : float, default 2.0
#         Spline smoothing strength (s = smoothing * n_points_in_contour).
#     n_eval : int, default 400
#         Fixed number of evaluation points per contour. Only used when
#         `point_spacing` is None.
#     point_spacing : float or None, default None
#         Desired spacing between evaluation points, in physical units
#         (same units as `pixel_size`, e.g. nm). If given, the number of
#         points per contour is  round(arc_length / point_spacing) + 1,
#         so it scales with each contour's length. Overrides `n_eval`.
#     min_eval : int, default 16
#         Lower bound on the number of points when `point_spacing` is used,
#         so very short contours still get a usable curvature profile.
#     uniform_arclength : bool, default True
#         If True (and `point_spacing` is given), points are equally spaced
#         in arc length rather than in the spline parameter.
#     min_points : int, default 8
#         Contours with fewer raw points than this are skipped.
#     pixel_size : float, default 1.0
#         Physical length per pixel. Assumes square pixels.

#     Returns
#     -------
#     list of dicts, one per contour, each with:
#         'xy'     : (n, 2) coordinates in physical units
#         'kappa'  : (n,) signed curvature, in 1/[pixel_size units]
#         'normal' : (n, 2) unit normals
#         'length' : total arc length of the fitted spline, physical units
#     where n = n_eval, or the arc-length-derived count if point_spacing is set.
#     """
#     contours = find_contours(field, level=level)
#     results = []

#     for c in contours:
#         if len(c) < min_points:
#             continue
#         rows, cols = c[:, 0], c[:, 1]
#         closed = np.hypot(rows[0] - rows[-1], cols[0] - cols[-1]) < 1.0
#         per = 1 if closed else 0
#         try:
#             tck, _ = splprep([cols, rows], s=smoothing * len(c), per=per, k=3)
#         except Exception:
#             continue

#         # --- arc length of the fitted spline (dense evaluation) ---
#         u_dense = np.linspace(0, 1, max(2000, 10 * len(c)))
#         xd, yd = splev(u_dense, tck, der=0)
#         s_cum = np.r_[0, np.cumsum(np.hypot(np.diff(xd), np.diff(yd)))]  # pixels
#         length_px = s_cum[-1]
#         length_phys = length_px * pixel_size

#         # --- choose evaluation parameters ---
#         if point_spacing is None:
#             n = n_eval
#             uu = np.linspace(0, 1, n)
#         else:
#             n = max(min_eval, int(round(length_phys / point_spacing)) + 1)
#             if uniform_arclength:
#                 # invert s(u): find u values at equally spaced arc lengths
#                 uu = np.interp(np.linspace(0, length_px, n), s_cum, u_dense)
#             else:
#                 uu = np.linspace(0, 1, n)

#         x, y = splev(uu, tck, der=0)
#         dx, dy = splev(uu, tck, der=1)
#         ddx, ddy = splev(uu, tck, der=2)

#         speed = np.hypot(dx, dy)
#         speed = np.where(speed == 0, 1e-12, speed)
#         tx, ty = dx / speed, dy / speed
#         nx, ny = -ty, tx

#         denom = (dx**2 + dy**2) ** 1.5
#         kappa = (dx * ddy - dy * ddx) / np.where(denom == 0, 1e-12, denom)
#         kappa = kappa / pixel_size
#         xy = np.stack([x, y], axis=1) * pixel_size

#         results.append({
#             "xy": xy,
#             "kappa": kappa,
#             "normal": np.stack([nx, ny], axis=1),
#             "length": length_phys,
#         })
#     return results


# import numpy as np
# import matplotlib.pyplot as plt
# from matplotlib.collections import LineCollection
# from matplotlib.colors import Normalize


# def plot_domain_curvature_evenspace(image, field=None, contour_fn=None, level=0.0,
#                           smoothing=2.0, point_spacing=None, pixel_size=1.0,
#                           vmax_percentile=68, unit_label="nm",
#                           cmap="coolwarm", image_cmap="viridis", image_clim=None,
#                           linewidth=2.5, show_points=False, ax=None):
#     """
#     Overlay the evenly spaced curvature spline on an image.

#     Parameters
#     ----------
#     image : (H, W) array
#         Background image to display (e.g. Mx).
#     field : (H, W) array or None
#         Field the domain boundary is extracted from (e.g. Mz). If None,
#         the contour is extracted from `image` itself.
#     contour_fn : callable or None
#         Function taking a 2D field and returning the list of dicts from
#         `contour_curvature` (e.g. your `contour_fn_3`). If given, `level`,
#         `smoothing`, and `point_spacing` are ignored, since the function
#         carries its own settings. If None, `contour_curvature` is called
#         with those arguments.
#     point_spacing : float or None
#         Spacing between points in physical units. Defaults to `pixel_size`.
#     vmax_percentile : float
#         Colour scale is symmetric, +/- this percentile of |kappa| over all
#         contours (clips outliers so typical curvature stays visible).
#     unit_label : str
#         Length unit for axis and colorbar labels.
#     show_points : bool
#         Draw a dot at every evaluation point to visualise the spacing.
#     ax : matplotlib Axes or None
#         Draw into an existing axes if given.

#     Returns
#     -------
#     fig, ax, results
#         `results` is the list of contour dicts (xy, kappa, normal, length).
#     """
#     if field is None:
#         field = image

#     # --- get contours + curvature ---
#     if contour_fn is not None:
#         results = contour_fn(field)
#     else:
#         results = contour_curvature_evenspacing(
#             field, level=level, smoothing=smoothing,
#             point_spacing=point_spacing if point_spacing is not None else pixel_size,
#             pixel_size=pixel_size)

#     if len(results) == 0:
#         raise ValueError("No contours found. Check `level` and `min_points`.")

#     # --- symmetric colour scale from all curvature values ---
#     all_k = np.concatenate([r["kappa"] for r in results])
#     vmax = np.nanpercentile(np.abs(all_k), vmax_percentile)
#     norm = Normalize(vmin=-vmax, vmax=vmax)

#     # --- figure / background image ---
#     if ax is None:
#         fig, ax = plt.subplots(figsize=(8, 8))
#     else:
#         fig = ax.figure

#     H, W = image.shape
#     extent = [0, W * pixel_size, 0, H * pixel_size]
#     imshow_kw = {}
#     if image_clim is not None:
#         imshow_kw["vmin"], imshow_kw["vmax"] = image_clim
#     ax.imshow(image, cmap=image_cmap, alpha = 0.5, origin="lower", extent=extent, **imshow_kw)

#     # --- curvature-coloured spline segments ---
#     lc = None
#     for r in results:
#         xy, k = r["xy"], r["kappa"]
#         segs = np.stack([xy[:-1], xy[1:]], axis=1)   # (n-1, 2, 2)
#         k_seg = 0.5 * (k[:-1] + k[1:])               # colour by segment midpoint
#         lc = LineCollection(segs, cmap=cmap, norm=norm, linewidth=linewidth)
#         lc.set_array(k_seg)
#         ax.add_collection(lc)
#         if show_points:
#             ax.scatter(xy[:, 0], xy[:, 1], c=k, cmap=cmap, norm=norm,
#                        s=8, linewidths=0.3, zorder=3)

#     cbar = fig.colorbar(lc, ax=ax, fraction=0.046, pad=0.04, extend="both")
#     cbar.set_label(f"Signed curvature (1/{unit_label})")

#     ax.set_xlim(extent[0], extent[1])
#     ax.set_ylim(extent[2], extent[3])
#     ax.set_aspect("equal")
#     ax.set_xlabel(f"x ({unit_label})")
#     ax.set_ylabel(f"y ({unit_label})")
#     return fig, ax, results

In [ ]:
def plot_chi_wall_grid(out, Mz3, quantity="c2", ncols=4, pad_nm=40, z_center_idx=None,
                       unit="nm", size=10):
    lab, vmin, vmax, cmap = QUANT[quantity]
    z = _z(out, z_center_idx); zs = out["zs"]; px = out["pixel_size"]
    n = len(zs); nr = int(np.ceil(n / ncols))
    fig, axs = plt.subplots(nr, ncols, figsize=(3.4*ncols, 3.4*nr), squeeze=False,
                            sharex=True, sharey=True)
    xl = (np.nanmin(out["x"]) - pad_nm, np.nanmax(out["x"]) + pad_nm)
    yl = (np.nanmin(out["y"]) - pad_nm, np.nanmax(out["y"]) + pad_nm)
    ny, nx = Mz3[zs[0]].shape
    extent = (-.5*px, (nx-.5)*px, (ny-.5)*px, -.5*px)
    for a in axs.flat[n:]:
        a.set_visible(False)
    for r, (k, zk) in enumerate(zip(zs, z)):
        a = axs.flat[r]
        a.imshow(Mz3[k], extent=extent, origin="upper", cmap="gray")
        sc = a.scatter(out["x"][r], out["y"][r], c=out[quantity][r], cmap=cmap,
                       vmin=vmin, vmax=vmax, s=size)
        a.set_xlim(xl); a.set_ylim(yl[1], yl[0])
        a.set_title(f"z = {zk:g} {unit}", fontsize=9)
    fig.colorbar(sc, ax=axs, fraction=0.025, pad=0.02, label=lab)
    return fig, axs

# fig, axs = plot_chi_wall_grid(out, volume['Mz'], quantity="neel", ncols=3)

In [ ]:
def plot_chi_heatmap(out, Mz_ref, quantity="c2", z_center_idx=None, unit="nm"):
    lab, vmin, vmax, cmap = QUANT[quantity]
    z = _z(out, z_center_idx); arc = out["arc"]
    cm = plt.get_cmap(cmap).copy(); cm.set_bad("0.85")

    fig = plt.figure(figsize=(14, 6))
    gs = GridSpec(2, 2, figure=fig, width_ratios=[1, 1.5], height_ratios=[1, 5],
                  hspace=0.05, wspace=0.25)
    a0 = fig.add_subplot(gs[:, 0])
    ak = fig.add_subplot(gs[0, 1])
    a1 = fig.add_subplot(gs[1, 1], sharex=ak)

    px = out["pixel_size"]; ny, nx = Mz_ref.shape
    a0.imshow(Mz_ref, extent=(-.5*px, (nx-.5)*px, (ny-.5)*px, -.5*px), origin="upper", cmap="gray")
    sc = a0.scatter(*out["ref_xy"][out["idx0"]].T, c=arc, cmap="plasma", s=8)
    fig.colorbar(sc, ax=a0, fraction=0.046, pad=0.04, label=f"arc length ({unit})")
    a0.set_xlabel(f"x ({unit})"); a0.set_ylabel(f"y ({unit})")

    ak.plot(arc, out["kappa"][list(out["zs"]).index(out["z_ref"])], "k-", lw=1)
    ak.axhline(0, color="gray", lw=0.8)
    ak.set_ylabel(f"κ (1/{unit})", fontsize=8); plt.setp(ak.get_xticklabels(), visible=False)

    m = a1.pcolormesh(arc, z, np.ma.masked_invalid(out[quantity]), cmap=cm,
                      vmin=vmin, vmax=vmax, shading="nearest")
    fig.colorbar(m, ax=a1, pad=0.01, label=lab)
    a1.set_xlabel(f"arc length along reference wall ({unit})")
    a1.set_ylabel(f"z ({unit})" + ("" if z_center_idx is None else ", 0 = film center"))
    return fig, (a0, ak, a1)

# fig, axes = plot_chi_heatmap(out, volume['Mz'][z_ref], quantity="c2")

In [ ]:
import warnings
from matplotlib.colors import Normalize

def plot_chi_arc_lines(out, quantity="c2", z_center_idx=None, cmap="viridis", unit="nm"):
    lab, vmin, vmax, _ = QUANT[quantity]
    z = _z(out, z_center_idx); arc = out["arc"]; Q = out[quantity]
    norm = Normalize(z.min(), z.max() if z.max() > z.min() else z.min() + 1)
    cm = plt.get_cmap(cmap)

    fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 6.5), sharex=True,
                                 gridspec_kw={"height_ratios": [3, 1.2], "hspace": 0.08})
    for r, zk in enumerate(z):
        a1.plot(arc, Q[r], "-", lw=1.2, color=cm(norm(zk)))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        mean, sd = np.nanmean(Q, axis=0), np.nanstd(Q, axis=0)
    a1.plot(arc, mean, "k-", lw=2.5, label="mean over z")
    if vmin is not None:
        a1.set_ylim(vmin - .05, vmax + .05)
    if quantity == "c2":
        a1.axhline(1, color="cyan", lw=1, alpha=.8); a1.axhline(0, color="magenta", lw=1, alpha=.8)
    else:
        a1.axhline(0, color="gray", lw=1)
    a1.set_ylabel(lab); a1.legend(fontsize=8, loc="lower right")
    fig.colorbar(plt.cm.ScalarMappable(norm, cm), ax=[a1, a2], pad=0.02, label=f"z ({unit})")

    a2.plot(arc, sd, "k-")
    a2.set_ylabel("std over z"); a2.set_xlabel(f"arc length along reference wall ({unit})")
    return fig, (a1, a2)

# fig, axes = plot_chi_arc_lines(out, quantity="c2")

In [ ]:
# ---- same setup as your script (nothing redefined) ----
# 452 = 3, 665 = 2
z_ref, contour_idx, point_idx = 2, 2, 100

CFG = dict(pixel_size=voxel_nm, half_width=half * voxel_nm,
           n_samples=80, order=3, sin_min=0.2, model="gauss")

contour_fn = lambda Mz: contour_curvature(Mz, level=0.0, smoothing=sigma_save,
                                          n_eval=400, pixel_size=voxel_nm)
Mz_slice = volume["Mz"][z_ref]
results_3 = contour_fn(Mz_slice)

# ---- along-wall analysis on that same contour ----
tracked = track_wall(volume["Mz"], contour_fn, z_ref, results_3[contour_idx],
                     tol=3 * voxel_nm)

step = 1                                   # 300 % 4 == 0, so point_idx is one of the sampled points
out_arc = psi_along_wall(volume["Mx"], volume["My"], volume["Mz"], frobenius_2d,
                         tracked, z_ref, tol=3 * voxel_nm, dz_nm=spacing[0],
                         step=step, reduce="center", window=1, **CFG)

# fig, axes = plot_chi_heatmap(out_arc, Mz_slice, quantity="c2")
# fig, axs = plot_chi_wall_grid(out_arc, volume["Mz"], quantity="neel", ncols=3)
# fig, axes = plot_chi_arc_lines(out_arc, quantity="c2")

In [ ]:
def plot_chi_heatmap(out, Mz_ref=None, quantity="c2", z_center_idx=None, unit="nm"):
    lab, vmin, vmax, cmap = QUANT[quantity]
    z = _z(out, z_center_idx); arc = out["arc"]
    cm = plt.get_cmap(cmap).reversed(); cm.set_bad("0.85") # .copy()

    fig, a1 = plt.subplots(figsize=(30, 6))


    print(arc.shape)
    m = a1.pcolormesh(arc, z, np.ma.masked_invalid(out[quantity]), cmap=cm,
                      vmin=vmin, vmax=vmax, shading="nearest")
    fig.colorbar(m, ax=a1, pad=0.05, label=lab)
    a1.set_xlabel(f"arc length along reference wall ({unit})")
    a1.set_ylabel(f"z ({unit})" + ("" if z_center_idx is None else ", 0 = film center"))

    # kappa overlay on a secondary y-axis
    ak = a1.twinx()
    kcolor = "red"
    kappa = out["kappa"][list(out["zs"]).index(out["z_ref"])]
    ak.plot(arc, kappa, color = kcolor, lw=2)
    ak.axhline(0, color="gray", lw=0.8, ls="--")

    ak.axvline(279*8, color="white", lw=2, ls="--")
    ak.axvline(309*8, color="white", lw=2, ls="--")
    ak.axvline(686*8, color="white", lw=2, ls="--")


    kmax = np.nanmax(np.abs(kappa))
    ak.set_ylim(-1.05 * kmax, 1.05 * kmax)
    ak.set_ylabel(f"κ (1/{unit})", color=kcolor)
    ak.set_xlim(arc.min(), arc.max())

    return fig, (a1, ak)

fig, axes = plot_chi_heatmap(out_arc, quantity="c2")

In [ ]:
z_ref, contour_idx, point_idx = 2, 0, 10


Mz_slice = volume["Mz"][z_ref]
sigma_save = 0.05

CFG = dict(pixel_size=voxel_nm, half_width=half * voxel_nm,
           n_samples=81, order=3, sin_min=0.2, model="gauss")

contour_fn_3 = lambda Mz: contour_curvature_evenspacing(
    Mz, level=0.0, smoothing=sigma_save,
    point_spacing=voxel_nm,        # one point per voxel of arc length; adjust to taste
    pixel_size=voxel_nm)

# fig, ax, ___ = plot_domain_curvature_evenspace(Mz_slice, Grad_slice_2d, level=0.0, smoothing=sigma_save, pixel_size=voxel_nm, vmax_percentile = 68, unit_label='nm')


# fig, ax, results = plot_domain_curvature_evenspace(
#     Mz_slice,                      # background image
#     field=Mz_slice,                # contour source
#     contour_fn=contour_fn_3,       # uses its own smoothing / point_spacing
#     pixel_size=voxel_nm,
#     vmax_percentile=68,
#     unit_label='nm',
#     show_points=True)              # optional: show the even spacing
# plt.show()

results_3 = contour_fn_3(Mz_slice)

fig, axes, out = plot_angle_decomposition(
    Mx_slice, My_slice, Mz_slice, Grad_slice, results_3,
    contour_idx=contour_idx, point_idx=point_idx, jac=None, k=None, **CFG)

In [ ]:



# ---- along-wall analysis on that same contour ----
tracked_3 = track_wall(volume["Mz"], contour_fn, z_ref, results_3[contour_idx],
                     tol=3 * voxel_nm)

step = 1                                 # 300 % 4 == 0, so point_idx is one of the sampled points
out_arc_3 = psi_along_wall(volume["Mx"], volume["My"], volume["Mz"], frobenius_2d,
                         tracked_3, z_ref, tol=3 * voxel_nm, dz_nm=spacing[0],
                         step=step, reduce="center", window=1, **CFG)


# per-slice arrays, all from z_ref
Mx_slice, My_slice, Mz_slice = (volume[c][z_ref] for c in ("Mx", "My", "Mz"))
Grad_slice = frobenius_2d[z_ref]
results_3 = contour_fn_3(Mz_slice)



# 2) through z, same contour, same gradient, same window
profiles_3 = psi_vs_z(volume['Mx'], volume['My'], volume['Mz'], frobenius_2d, contour_fn_3,
                    z_ref=z_ref, ref_result=results_3[contour_idx], ref_pt=point_idx,
                    tol=3 * voxel_nm, window=1, **CFG)

fig, axes = plot_psi_vs_z(volume['Mz'], profiles_3, z_ref=z_ref,
                          dz_nm=spacing[0], pixel_size=voxel_nm, ylim = (-195, 195), window=2, smooth=0)

In [ ]:
# def plot_chi_heatmap(out, profiles_by_pt, quantity="c2", z_center_idx=None, unit="nm",
#                      dz_nm=None, ylim=(-195, 195), cmap_z="viridis",
#                      kcolor="red", line_color="white", center=False, figsize=(30, 12),
#                      fs=15, num_fs=22):
#     """
#     fs     : font size for axis labels, tick labels, and colorbar labels (>= 15)
#     num_fs : font size for the panel numbers
#     """
#     lab, vmin, vmax, cmap = QUANT[quantity]
#     z = _z(out, z_center_idx); arc = out["arc"]
#     cm = plt.get_cmap(cmap).reversed(); cm.set_bad("0.85")

#     pts = list(profiles_by_pt)
#     n = len(pts)
#     arc_full = arc_length(out["ref_xy"])
#     pt_arc = {p: arc_full[p] for p in pts}
#     z_ref = out["z_ref"]

#     rc = {"font.family": "serif",
#           "font.serif": ["Times New Roman", "Times", "STIXGeneral", "DejaVu Serif"],
#           "mathtext.fontset": "stix",           # Times-like math (κ, ψ, etc.)
#           "font.size": fs, "axes.labelsize": fs,
#           "xtick.labelsize": fs, "ytick.labelsize": fs}

#     with plt.rc_context(rc):
#         fig = plt.figure(figsize=figsize, layout="constrained")
#         gs = GridSpec(2, n, figure=fig, height_ratios=[1.2, 1])
#         a1 = fig.add_subplot(gs[0, :])

#         m = a1.pcolormesh(arc, z, np.ma.masked_invalid(out[quantity]), cmap=cm,
#                           vmin=vmin, vmax=vmax, shading="nearest")
#         cb = fig.colorbar(m, ax=a1, pad=0.02)
#         cb.set_label(lab, fontsize=fs)
#         cb.ax.tick_params(labelsize=fs)
#         a1.set_xlabel(f"arc length along reference wall ({unit})", fontsize=fs)
#         a1.set_ylabel(f"z ({unit})" + ("" if z_center_idx is None else ", 0 = film center"),
#                       fontsize=fs)

#         # white lines where each normal was taken, numbered to match the bottom panels
#         ztop = np.nanmax(z)
#         for i, p in enumerate(pts):
#             a1.axvline(pt_arc[p], color=line_color, lw=2, ls="--")
#             a1.text(pt_arc[p], ztop, f" {i+1}", color=line_color, va="top", ha="left",
#                     fontsize=num_fs, fontweight="bold")

#         # kappa overlay on a secondary y-axis, kappa = 0 at center
#         ak = a1.twinx()
#         kappa = out["kappa"][list(out["zs"]).index(out["z_ref"])]
#         ak.plot(arc, kappa, color=kcolor, lw=3)
#         ak.axhline(0, color="gray", lw=0.8, ls="--")
#         kmax = np.nanmax(np.abs(kappa))
#         ak.set_ylim(-1.05 * kmax, 1.05 * kmax)
#         ak.set_ylabel(f"κ (1/{unit})", color=kcolor, fontsize=fs)
#         ak.tick_params(axis="y", colors=kcolor, labelsize=fs)
#         ak.set_xlim(arc.min(), arc.max())

#         # bottom panels: psi vs position along the normal, one per point_idx
#         zs_all = sorted(set().union(*[set(pr) for pr in profiles_by_pt.values()]))
#         dz = dz_nm if dz_nm is not None else 1.0
#         zn_all = np.array(zs_all) * dz
#         norm = Normalize(zn_all.min(),
#                          zn_all.max() if zn_all.max() > zn_all.min() else zn_all.min() + 1)
#         cz = plt.get_cmap(cmap_z)

#         bottom = []
#         for i, p in enumerate(pts):
#             ax = fig.add_subplot(gs[1, i], sharey=bottom[0] if bottom else None)
#             bottom.append(ax)
#             prof = profiles_by_pt[p]
#             zs = np.array(sorted(prof))

#             for kk in range(-4, 5):
#                 y = 90 * kk
#                 if -195 <= y <= 195:
#                     col = "magenta" if kk % 2 else ("tab:blue" if kk % 4 == 0 else "cyan")
#                     ax.axhline(y, color=col, lw=1, alpha=0.6)

#             for k in zs:
#                 pk = prof[k]
#                 if not pk["good"].any():
#                     continue
#                 x = pk["s"][pk["good"]] - (pk["mu"] if center else 0.0)
#                 y = np.degrees(pk["psi"][pk["good"]])
#                 xb, yb = _break_jumps(x, y)
#                 c = cz(norm(k * dz))
#                 ax.plot(xb, yb, "-", color=c, lw=1.6)
#                 ax.plot(x, y, ".", color=c, ms=4)

#             P0 = prof[z_ref] if z_ref in prof else prof[zs[0]]
#             c0 = P0["mu"] if center else 0.0
#             for e in (P0["mu"] - P0["fwhm"] / 2 - c0, P0["mu"] + P0["fwhm"] / 2 - c0):
#                 ax.axvline(e, color="k", lw=1, ls="--")
#             if not center:
#                 ax.axvline(0, color="r", ls="--", lw=1)

#             ax.set_ylim(ylim)
#             # panel number in the top-left corner (replaces the title)
#             ax.text(0.03, 0.96, f"{i+1}", transform=ax.transAxes, va="top", ha="left",
#                     fontsize=num_fs, fontweight="bold",
#                     bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.8))
#             ax.set_xlabel(f"position along normal ({unit})", fontsize=fs)
#             ax.tick_params(labelsize=fs)
#             if i == 0:
#                 ax.set_ylabel("Azimuthal angle ψ (deg)", fontsize=fs)
#             else:
#                 plt.setp(ax.get_yticklabels(), visible=False)

#         sm = plt.cm.ScalarMappable(norm, cz)
#         cb2 = fig.colorbar(sm, ax=bottom, pad=0.02)
#         cb2.set_label(f"z ({unit})", fontsize=fs)
#         cb2.ax.tick_params(labelsize=fs)

#     return fig, (a1, ak, bottom)


In [ ]:
point_idxs = [32, 60, 100, 130]     

profiles_by_pt = {
    p: psi_vs_z(volume['Mx'], volume['My'], volume['Mz'], frobenius_2d, contour_fn_3,
                z_ref=z_ref, ref_result=results_3[contour_idx], ref_pt=p,
                tol=3 * voxel_nm, window=1, **CFG)
    for p in point_idxs
}

fig, axes = plot_chi_heatmap(out_arc_3, profiles_by_pt, quantity="c2", dz_nm=spacing[0])

In [ ]:
# def plot_points_on_spline(Mz_ref, result, point_idxs, pixel_size, out=None, unit="nm",
#                           figsize=(8, 8), mark_color="white"):
#     """
#     Mz_ref     : reference Mz slice (e.g. Mz_slice)
#     result     : results_3[contour_idx], the same contour passed to psi_vs_z / plot_angle_decomposition
#     point_idxs : the 4 indices into the full contour (e.g. [270, 300, 340, 380])
#     out        : optional out_arc, used to check consistency with the heatmap
#     """
#     xy = np.asarray(result["xy"])
#     arc_full = arc_length(xy)
#     ny, nx = Mz_ref.shape
#     extent = (-0.5 * pixel_size, (nx - 0.5) * pixel_size,
#               (ny - 0.5) * pixel_size, -0.5 * pixel_size)

#     fig, ax = plt.subplots(figsize=figsize, layout="constrained")
#     ax.imshow(Mz_ref, extent=extent, origin="upper", cmap="gray")
#     sc = ax.scatter(xy[:, 0], xy[:, 1], c=arc_full, cmap="plasma", s=6)
#     fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.02, label=f"arc length ({unit})")

#     for i, p in enumerate(point_idxs):
#         x, y = xy[p]
#         ax.plot(x, y, "o", ms=10, mfc="none", mec=mark_color, mew=2)
#         ax.annotate(f"{i+1}  (idx {p}, {arc_full[p]:.0f} {unit})", (x, y),
#                     xytext=(8, 8), textcoords="offset points",
#                     color=mark_color, fontsize=11, fontweight="bold",
#                     bbox=dict(boxstyle="round,pad=0.2", fc="black", alpha=0.5, lw=0))

#     # start/end markers so the contour direction is clear
#     ax.plot(*xy[0], "s", color="tab:green", ms=7, label="start (idx 0)")
#     ax.plot(*xy[-1], "^", color="tab:red", ms=7, label=f"end (idx {len(xy)-1})")
#     ax.legend(loc="upper right", fontsize=8)
#     ax.set_xlabel(f"x ({unit})"); ax.set_ylabel(f"y ({unit})")
#     ax.set_title("Selected normals on the reference contour")

#     # consistency checks against the heatmap's data
#     if out is not None:
#         same_xy = (len(out["ref_xy"]) == len(xy)) and np.allclose(out["ref_xy"], xy)
#         print(f"out['ref_xy'] matches this contour: {same_xy}")
#         for i, p in enumerate(point_idxs):
#             a_heat = out["arc"][np.where(out["idx0"] == p)[0]]
#             msg = f"{a_heat[0]:.1f}" if len(a_heat) else "not on the step grid"
#             print(f"  {i+1}: idx {p} -> arc_full {arc_full[p]:.1f} | heatmap arc {msg} "
#                   f"| xy = ({xy[p,0]:.1f}, {xy[p,1]:.1f})")
#     return fig, ax

# fig2, ax2 = plot_points_on_spline(Mz_slice, results_3[contour_idx], point_idxs,
#                                   pixel_size=voxel_nm, out=out_arc)

In [ ]:
stop

# Caluclate the Divergence from the Jacobian

In [ ]:
def compute_divergence(jacobian):
    """
    standard definition:
        div(M)_x = dMx/dx + dMx/dy + dMx/dz
        div(M)_y = dMy/dx + dMy/dy + dMy/dz
        div(M)_z = dMz/dx + dMz/dy + dMz/dz

    Parameters
    ----------
    jacobian : dict
        Output of compute_jacobian, containing dMx_dx, dMy_dy, dMz_dz, plus 'spacing' and 'shape'.

    Returns
    -------
    curl : dict
        'div_x', 'div_y', 'div_z' (each shape (nz, ny, nx), units
        per nm assuming jacobian was computed with physical spacing),
        'div_magnitude' 
    """
    div_x = jacobian['dMx_dx']
    div_y = jacobian['dMy_dy']
    div_z = jacobian['dMz_dz']

    div_M = div_x + div_y + div_z
    rho_m = -div_M

    return {
        'div_M': div_M,
        'div_x': div_x,
        'div_y': div_y,
        'div_z': div_z,
        'rho_m': rho_m,
        'shape': jacobian['shape'],
    }

def classify_sources_sinks(divergence, threshold=None, threshold_percentile=95):
    """
    Classify voxels as source, sink, or negligible, based on the sign
    and magnitude of the scalar divergence field.

    Parameters
    ----------
    divergence : dict
        Output of compute_divergence, must contain 'div_M'.
    threshold : float or None
        Absolute divergence magnitude above which a voxel is considered
        a meaningful source/sink (rather than noise). If None, computed
        automatically from threshold_percentile.
    threshold_percentile : float
        If threshold is None, use this percentile of |div_M| (over the
        whole volume) as the cutoff, e.g. 90 means only the top 10% of
        divergence magnitudes count as real sources/sinks.

    Returns
    -------
    divergence : dict
        Same dict, with 'source_mask' (div_M > threshold),
        'sink_mask' (div_M < -threshold), and 'threshold_used' added.
    """
    div_M = divergence['div_M']

    if threshold is None:
        threshold = np.percentile(np.abs(div_M), threshold_percentile)

    divergence['source_mask'] = div_M > threshold
    divergence['sink_mask'] = div_M < -threshold
    divergence['threshold_used'] = threshold

    n_source = divergence['source_mask'].sum()
    n_sink = divergence['sink_mask'].sum()
    print(f"threshold = {threshold:.4g} (per nm): "
          f"{n_source} source voxels, {n_sink} sink voxels "
          f"out of {div_M.size} total")

    return divergence

In [ ]:
divergence = compute_divergence(jacobian)
divergence = classify_sources_sinks(divergence, threshold_percentile=95)

# Calculate Cross product 

In [ ]:
def compute_lifshitz_invariants(jacobian, volume_smoothed, curl=None):
    """
    Compute DMI-relevant antisymmetric exchange (Lifshitz invariant)
    combinations, for both interfacial (Cnv) and bulk DMI symmetry.

    Parameters
    ----------
    jacobian : dict
        Output of compute_jacobian.
    volume_smoothed : dict
        Must contain 'Mx', 'My', 'Mz'.
    curl : dict or None
        Optionally pass an already-computed compute_curl(...) output to
        avoid recomputing curl_x/y/z for the bulk term; if None, curl is
        computed inline.

    Returns
    -------
    lifshitz : dict
        'interfacial' -- Mz*(dMx/dx + dMy/dy) - (Mx*dMz/dx + My*dMz/dy),
            relevant for interfacial DMI / Neel-type textures (Cnv
            symmetry, e.g. PMA film on heavy metal).
        'bulk' -- M . curl(M), relevant for bulk DMI / Bloch-type
            textures (e.g. B20 chiral magnets).
        'L_xz_x', 'L_yz_y' -- the two individual interfacial terms, in
            case you want to inspect x- vs y-direction contributions
            separately (useful for checking wall orientation dependence).
        'shape', 'spacing' carried through.
    """
    Mx, My, Mz = volume_smoothed['Mx'], volume_smoothed['My'], volume_smoothed['Mz']

    L_xz_x = Mx * jacobian['dMz_dx'] - Mz * jacobian['dMx_dx'] 
    L_yz_y = My * jacobian['dMz_dy'] - Mz * jacobian['dMy_dy'] 
    interfacial = L_xz_x + L_yz_y

    if curl is None:
        curl_x = jacobian['dMz_dy'] - jacobian['dMy_dz']
        curl_y = jacobian['dMx_dz'] - jacobian['dMz_dx']
        curl_z = jacobian['dMy_dx'] - jacobian['dMx_dy']
    else:
        curl_x, curl_y, curl_z = curl['curl_x'], curl['curl_y'], curl['curl_z']

    bulk = Mx * curl_x + My * curl_y + Mz * curl_z

    return {
        'interfacial': interfacial,
        'L_xz_x': L_xz_x,
        'L_yz_y': L_yz_y,
        'bulk': bulk,
        'shape': jacobian['shape'],
        'spacing': jacobian['spacing'],
    }


lifshitz = compute_lifshitz_invariants(jacobian, volume, curl=curl)  # reuse curl from before

In [ ]:
def plot_scalar_maps(data_dict, keys, axis='z', index=None,
                      cmap='RdBu_r', seq_cmap='viridis',
                      diverging_keys=None, vmin=None, vmax=None,
                      shared_scale=True, title = ''):
    """
    Generic three-panel slice plot for any named scalar volumes stored
    in data_dict (e.g. jacobian, curl, topo, or lifshitz dict outputs).

    Parameters
    ----------
    data_dict : dict
        Any of the dicts produced above (must contain 'shape' and the
        requested keys as (nz, ny, nx) arrays).
    keys : tuple of 3 str
        Which entries of data_dict to plot side by side.
    axis, index : slice selection, as in the other plotting functions.
    diverging_keys : set of str or None
        Keys that should use the diverging (signed) colormap; anything
        not listed here is treated as non-negative and plotted with
        seq_cmap on a 0-to-max scale. If None, everything is treated
        as diverging/signed (reasonable default for curl, jacobian,
        lifshitz, and topo charge, which are all signed quantities).
    """
    nz, ny, nx = data_dict['shape']
    default_index = {'x': nx // 2, 'y': ny // 2, 'z': nz // 2}
    if index is None:
        index = default_index[axis]

    def get_slice(data, axis, index):
        if axis == 'x':
            return data[:, :, index], 'y (voxel)', 'z (voxel)', [0, ny, 0, nz]
        elif axis == 'y':
            return data[:, index, :], 'x (voxel)', 'z (voxel)', [0, nx, 0, nz]
        elif axis == 'z':
            return data[index, :, :], 'x (voxel)', 'y (voxel)', [0, nx, 0, ny]
        else:
            raise ValueError("axis must be 'x', 'y', or 'z'")

    if diverging_keys is None:
        diverging_keys = set(keys)

    if shared_scale and (vmin is None or vmax is None):
        div_keys_present = [k for k in keys if k in diverging_keys]
        if div_keys_present:
            all_vals = np.concatenate([data_dict[k].ravel() for k in div_keys_present])
            abs_max = np.nanmax(np.abs(all_vals))
            vmin = -abs_max if vmin is None else vmin
            vmax = abs_max if vmax is None else vmax

    fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
    for ax, key in zip(axes, keys):
        data = data_dict[key]
        img, xlabel, ylabel, extent = get_slice(data, axis, index)

        if key in diverging_keys:
            panel_cmap = cmap
            if shared_scale:
                panel_vmin, panel_vmax = vmin, vmax
            else:
                abs_max = np.nanmax(np.abs(img))
                panel_vmin, panel_vmax = -abs_max, abs_max
        else:
            panel_cmap = seq_cmap
            panel_vmin, panel_vmax = 0, np.nanmax(img)

        im = ax.imshow(img, origin='lower', aspect='auto', cmap=panel_cmap,
                        vmin=panel_vmin, vmax=panel_vmax, extent=extent)
        ax.set_title(key)
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        fig.colorbar(im, ax=ax, label=key)

    fig.suptitle(f"{axis}-slice at index {index}")
    plt.tight_layout()
    plt.show()
    return fig, axes

# Compute topological charge

In [ ]:
def compute_topological_charge_density(jacobian, volume_smoothed):
    """
    Compute the local topological (skyrmion) charge density,
        q = [M . (dM/dx x dM/dy)] / |M|^3
    which is equivalent to using the unit vector m = M/|M| directly,
    without needing a separate normalization step.

    Parameters
    ----------
    jacobian : dict
        Output of compute_jacobian.
    volume_smoothed : dict
        Must contain 'Mx', 'My', 'Mz', 'magnitude'.

    Returns
    -------
    topo : dict
        'q_density' -- per-voxel topological charge density (per nm^2,
            given physical spacing in the jacobian), such that summing
            q_density * dx * dy over an x-y slice and dividing by 4*pi
            gives that layer's skyrmion number.
        'shape', 'spacing' carried through for convenience.
    """
    Mx, My, Mz = volume_smoothed['Mx'], volume_smoothed['My'], volume_smoothed['Mz']
    mag = volume_smoothed.get('magnitude', np.sqrt(Mx**2 + My**2 + Mz**2))

    M = np.stack([Mx, My, Mz], axis=-1)                     # (nz, ny, nx, 3)
    dM_dx = np.stack([jacobian['dMx_dx'], jacobian['dMy_dx'], jacobian['dMz_dx']], axis=-1)
    dM_dy = np.stack([jacobian['dMx_dy'], jacobian['dMy_dy'], jacobian['dMz_dy']], axis=-1)

    cross = np.cross(dM_dx, dM_dy)                          # (nz, ny, nx, 3)
    triple_product = np.einsum('...i,...i->...', M, cross)  # M . (dM/dx x dM/dy)

    q_density = triple_product / (mag**3 * 4 * np.pi)

    return {'q_density': q_density, 'shape': jacobian['shape'], 'spacing': jacobian['spacing']}

def compute_skyrmion_number_per_layer(topo):
    """
    Integrate q_density over each x-y (z-fixed) layer to get the
    skyrmion number for that layer, using the physical in-plane voxel
    area from the jacobian's spacing.

    Returns
    -------
    Q_per_layer : (nz,) array
    """
    dz, dy, dx = topo['spacing']
    voxel_area = dy * dx
    Q_per_layer = np.sum(topo['q_density'], axis=(1, 2)) * voxel_area
    return Q_per_layer

In [ ]:
topo = compute_topological_charge_density(jacobian, volume_smoothed)
Q_per_layer = compute_skyrmion_number_per_layer(topo)
print("Skyrmion number per layer:", Q_per_layer)

# Calculate energy densities

In [ ]:
def calc_energy_density(volume, div, jacobian, energy_terms, spacing=(8.0, 8.0, 8.0)): 
    ''' 
    PMA:      -K * mz^2 
    Exchange: -A * (div_m)^2
    DMI = 𝑫[(𝒎×𝝏_𝒚 𝒎)_𝒙−(𝒎×𝝏_𝒙 𝒎)_𝒚]
    '''
    Mx, My, Mz = volume['Mx'], volume['My'], volume['Mz']


    pma = energy_terms[0] * (Mz ** 2)
    Exchange = energy_terms[1]*(div['div_M'])**2
    L_xz_x = Mz * jacobian['dMx_dx'] - Mx * jacobian['dMz_dx'] 
    L_yz_y = My * jacobian['dMz_dy'] - Mz * jacobian['dMy_dy'] 
    idmi = energy_terms[2]*(L_xz_x - L_yz_y)


    return {
        # 'PMA': pma,
        # 'Divergence': Exchange,
        # 'L_xz_x': L_xz_x,
        # 'L_yz_y': L_yz_y,
        'iDMI': idmi, 
        'shape': jacobian['shape'],
        'spacing': jacobian['spacing'],
    }


energy_terms = [1,1,1]
energy_densities = calc_energy_density(volume, divergence, jacobian, energy_terms, spacing=spacing)

In [ ]:
plot_scalar_maps(divergence, keys=('div_M', 'rho_m', 'div_x'), axis='z', index=None)

# Visualize source/sink regions directly (cast bool -> float for plotting)
divergence['source_mask_f'] = divergence['source_mask'].astype(float)
divergence['sink_mask_f'] = divergence['sink_mask'].astype(float)


In [ ]:
plot_energy_maps_fancy(divergence, 'div_M', axis='z', index=None, cmap='coolwarm', vmin=None, vmax=None, voxel_nm = voxel_nm, title = '')


In [ ]:
plot_energy_maps_fancy(energy_densities, 'iDMI', axis='z', index=None, cmap='turbo', vmin=None, vmax=None, voxel_nm = voxel_nm, title = '')
plot_energy_maps_fancy(energy_densities, 'iDMI', axis='z', index=0, cmap='turbo', vmin=None, vmax=None,voxel_nm = voxel_nm,  title = '')

In [ ]:
#plot_energy_maps_fancy(energy_densities, 'PMA', axis='z', index=None, cmap='Oranges', vmin=None, vmax=None,voxel_nm = voxel_nm, title = '')

# Compare DMI assumptions

In [ ]:
# from idmi_plot_compare import build_idmi_maps, plot_idmi_comparison



# # Mx, My, Mz shaped (nz, ny, nx); idmi is your Jacobian-based result
# maps = build_idmi_maps(volume['Mx'], volume['My'], volume['Mz'], D=1, normalize=True,   # or True for unit vectors
#                        periodic=(False, False))        # omit to use the built-in central-difference version

# figs = plot_idmi_comparison(maps, plot_energy_maps_fancy, axis='z', index=0, cmap='turbo')

# Save

In [ ]:
stop

In [ ]:
def make_json_safe(obj):
    """
    Recursively convert numpy types within a nested dict/list/tuple
    structure into native Python types, so it can be safely passed to
    json.dump. Handles numpy ints, floats, bools, and arrays.
    """
    if isinstance(obj, dict):
        return {k: make_json_safe(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [make_json_safe(v) for v in obj]
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, (np.integer,)):
        return int(obj)
    elif isinstance(obj, (np.floating,)):
        return float(obj)
    elif isinstance(obj, (np.bool_,)):
        return bool(obj)
    else:
        return obj

In [ ]:
def save_volume_analysis(save_dir, sample_name, use_smooth = False, date = 'None', volume_smoothed=None,
                          **named_field_dicts):
    """
    Save the cropped (and optionally smoothed) volume plus any number of
    derived-field dictionaries (jacobian, curl, divergence, lifshitz/dmi,
    topo, etc.) to disk, in a format easy to reload into another notebook.

    Parameters
    ----------
    save_dir : str
        Directory to save into (created if it doesn't exist).
    sample_name : str
        Identifier used in filenames.
    cropped : dict
        Original cropped sample dict, must contain 'Mx', 'My', 'Mz',
        'magnitude', 'shape'.
    volume_smoothed : dict or None
        Output of make_smoothed_volume, if used; saved separately so the
        new notebook can choose to work from raw or smoothed fields.
    **named_field_dicts : dict of dict
        Any number of derived-field dictionaries, passed as keyword
        arguments, e.g.:
            save_volume_analysis(..., jacobian=jacobian, curl=curl,
                                  divergence=divergence, lifshitz=lifshitz,
                                  topo=topo)
        Each dict's array-valued entries are saved; non-array entries
        (e.g. 'shape', 'spacing', 'sigma') are stored as metadata.
    """
    os.makedirs(save_dir, exist_ok=True)


    metadata = {'sample_name': sample_name, 'saved_fields': list(named_field_dicts.keys())}

    # --- Smoothed volume, if provided ---
    if use_smooth:
        smoothed_path = os.path.join(save_dir, f'{sample_name}_volume_smoothed_{date}.npz')
        np.savez_compressed(smoothed_path,
                             Mx=volume_smoothed['Mx'], My=volume_smoothed['My'],
                             Mz=volume_smoothed['Mz'], magnitude=volume_smoothed['magnitude'],
                             shape=np.array(volume_smoothed['shape']))
        metadata['sigma'] = volume_smoothed.get('sigma')

    # --- Each derived-field dict (jacobian, curl, divergence, lifshitz, topo, ...) ---
    for field_name, field_dict in named_field_dicts.items():
        print(field_name)
        array_entries = {}
        field_meta = {}
        for key, val in field_dict.items():
            if isinstance(val, np.ndarray):
                array_entries[key] = val
            else:
                # non-array metadata like 'shape', 'spacing', 'sigma'
                field_meta[key] = val if not isinstance(val, tuple) else list(val)

        field_path = os.path.join(save_dir, f'{sample_name}_{field_name}_{date}.npz')
        np.savez_compressed(field_path, **array_entries)
        metadata[f'{field_name}_meta'] = field_meta
        metadata[f'{field_name}_keys'] = list(array_entries.keys())

    meta_path = os.path.join(save_dir, f'{sample_name}_volume_analysis_metadata_{date}.json')
    with open(meta_path, 'w') as f:
        json.dump(make_json_safe(metadata), f, indent=2)

    print(f"Saved to {save_dir}:")
    if volume_smoothed is not None:
        print(f"  {sample_name}_smoothed_{date}.npz")
    for field_name in named_field_dicts:
        print(f"  {sample_name}_{field_name}_{date}.npz")
    print(f"  {sample_name}_volume_analysis_metadata_{date}.json")

In [ ]:
date

In [ ]:

# save_volume_analysis(
#     save_dir=data_dir,
#     sample_name=sample_name,
#     volume_smoothed=volume,
#     use_smooth = use_smoothed, 
#     date = date,
#     jacobian=jacobian,
#     curl=curl,

# )

In [ ]:
print(gradient.keys()) 
print(curl.keys())
print(energy_densities.keys())

In [ ]:

save_volume_analysis(
    save_dir=data_dir,
    sample_name=sample_name,
    volume_smoothed=volume,
    use_smooth = use_smoothed, 
    date = date,
    jacobian=jacobian,
    curl=curl,
    grad = gradient, 
    # divergence=divergence,     # substitute your actual variable name here
    energies = energy_densities,
    # Nsk = topo,
    # curvature=curvature_dict,
)


In [ ]:
import json
import numpy as np
import pyvista as pv

def dicts_to_vti(path, dicts, spacing, axis_order='zyx', scalars_extra=None,
                 vectors_extra=None, vector_groups=None, as_point_data=True,
                 dtype=np.float32, metadata=None):
    """
    path          : output file, e.g. 'results/sampleA.vti'
    dicts         : {'curl': curl, 'grad': grad, 'energy': energies}; every 3D array
                    with the volume's shape becomes a field named '<prefix>_<key>'
    spacing       : voxel size in nm, listed in the SAME order as axis_order
                    (axis_order='zyx' -> (dz, dy, dx))
    axis_order    : how your arrays are indexed: 'zyx' or 'zxy'
    scalars_extra : {'wall_mask': wall_mask, ...} additional 3D arrays
    vectors_extra : {'m': (unit_Mx, unit_My, unit_Mz), ...} additional vector fields
    vector_groups : {'curl_vec': (curl, ('curl_x','curl_y','curl_z'))} vector fields
                    built from dict keys, in (x, y, z) component order
    as_point_data : True  -> values stored at voxel centers as point data (supports
                             contours, smooth slices, streamlines in ParaView)
                    False -> true cell data (blocky voxels, no interpolation)
    """
    order = axis_order.lower()
    sp = dict(zip(order, spacing))
    dx, dy, dz = sp['x'], sp['y'], sp['z']

    def to_zyx(a):
        a = np.asarray(a)
        if order == 'zxy':
            a = np.transpose(a, (0, 2, 1))
        return a

    def flat(a):
        # (z, y, x) C-order flattening has x varying fastest, which is what VTK expects
        return np.ascontiguousarray(to_zyx(a)).astype(dtype).ravel(order='C')

    # volume shape from the first 3D array found
    ref = None
    for d in dicts.values():
        for v in d.values():
            if isinstance(v, np.ndarray) and v.ndim == 3:
                ref = to_zyx(v).shape
                break
        if ref:
            break
    nz, ny, nx = ref

    if as_point_data:
        grid = pv.ImageData(dimensions=(nx, ny, nz), spacing=(dx, dy, dz),
                            origin=(dx / 2, dy / 2, dz / 2))   # sample at voxel centers
        store = grid.point_data
    else:
        grid = pv.ImageData(dimensions=(nx + 1, ny + 1, nz + 1), spacing=(dx, dy, dz),
                            origin=(0, 0, 0))
        store = grid.cell_data

    saved, skipped = [], []
    for prefix, d in dicts.items():
        for key, v in d.items():
            if (isinstance(v, np.ndarray) and v.ndim == 3
                    and to_zyx(v).shape == (nz, ny, nx)):
                store[f"{prefix}_{key}"] = flat(v)
                saved.append(f"{prefix}_{key}")
            else:
                skipped.append(f"{prefix}.{key}")

    for name, v in (scalars_extra or {}).items():
        store[name] = flat(v)
        saved.append(name)

    for name, comps in (vectors_extra or {}).items():
        store[name] = np.stack([flat(c) for c in comps], axis=1)
        saved.append(name)

    for name, (d, keys) in (vector_groups or {}).items():
        store[name] = np.stack([flat(d[k]) for k in keys], axis=1)
        saved.append(name)

    grid.save(path)
    print(f"saved {path}: grid (x, y, z) = ({nx}, {ny}, {nz}), spacing = ({dx}, {dy}, {dz}) nm")
    print("fields:", saved)
    if skipped:
        print("skipped (not 3D arrays of the volume shape):", skipped)

    if metadata is not None:                      # sidecar file for bookkeeping
        with open(path.rsplit('.', 1)[0] + "_meta.json", "w") as f:
            json.dump(metadata, f, indent=2, default=str)
    return grid

In [ ]:
dicts_to_vti(
    f"{data_dir}/{sample_name}_vector_analysis_volume_{date}.vti",
    dicts={'curl': curl, 'grad': gradient, 'energy': energy_densities},
    spacing=(voxel_nm, voxel_nm, voxel_nm),                 # match axis_order; see notes below
    axis_order='zyx',                     # use 'zxy' if your arrays are (z, x, y)
    vector_groups={'curl_vec': (curl, ('curl_x', 'curl_y', 'curl_z'))},
    # scalars_extra={'wall_mask': wall_mask.astype(np.uint8)}
)

In [ ]:
# dicts_to_vti(
#     f"{data_dir}/{sample_name}_vector_analysis_volume_{date}.vti",
#     dicts={'curl': curl},
#     spacing=(voxel_nm, voxel_nm, voxel_nm),                 # match axis_order; see notes below
#     axis_order='zyx',                     # use 'zxy' if your arrays are (z, x, y)
#     vector_groups={'curl_vec': (curl, ('curl_x', 'curl_y', 'curl_z'))},
#     # scalars_extra={'wall_mask': wall_mask.astype(np.uint8)}
# )